# DSC 2026 · Task 2 — Vietnamese Legal QA · Pipeline **v13.2: private test**

v13 is v12 with a rebuilt input side for the **private test phase**:

- **Retrieval, generation and packaging:** unchanged from v12.
- **Test questions:** read from the dataset's **current** Kaggle version (`phucdangg/legalqa-task2-clean-data`, 1,918 questions).
- **Caches:** every cache that stores per-question output is now keyed by the question set.

### What v13 changes

| cell | what v12 would have done with a new test file | v13 |
|---|---|---|
| **0** | Skipped Kaggle whenever an older copy was on disk. | Downloads again whenever Kaggle's file listing changed. Each version gets its own folder, and nothing is deleted. |
| **2** | Read `public-official.json` by name. | Finds the file with exactly `DSC_EXPECTED_N_TEST` questions. Public + private files are merged when their ids don't overlap and the counts add up. Otherwise it **stops** and prints what it found. |
| **3** | Collapsed list files without an `id`/`qa_id` field into a single key, `"None"`. | Reads a dict, a list, a wrapped list, JSONL, CSV or parquet. Missing or duplicate ids are errors. |
| **7** | Keyed test contexts by retrieval settings only, so it would have **loaded the 1,000 public contexts**. | Keys contexts by settings **and** question set, then checks them against the test file. |
| **9 · 11** | Reused an answer only for the same id and the same prompt. | Also reuses it for the same prompt under another id, so public questions repeated in the private set cost nothing. |
| **12** | Checked ids against `test_ctx`, so a stale 1,000-answer file would pass. | Checks ids against the test file, including the exact count. |
| **13** | Only opened the archive. | Re-reads the test file and matches every id. |
| **3 · 5** | No check. | Fingerprints the corpus rows. Embeddings are never paired with a reordered corpus. |
| **6e** | Measured the lexref fire rate once, on the public test. | Measures it again on the current test set. |
| **0 · 5** | On Modal, stopped at CELL 0 when only `KAGGLE_KEY` was set, and silently fell back to the base encoder once the work dir moved off Drive. | `KAGGLE_KEY` is accepted as the Kaggle token. `ENCODER_ADOPTED.json` paths are re-rooted to the current work dir. |

### Settings (Modal Secrets / environment, set before CELL 0)

| variable | default | meaning |
|---|---|---|
| `DSC_EXPECTED_N_TEST` | `1918` | number of questions the test set must have (`0` = any) |
| `DSC_TEST_FILE` | — | force the test file(s), comma-separated |
| `DSC_TEST_ID_FIELD` | — | take ids from this record field instead of the dict keys |
| `DSC_DATA_REFRESH` | `auto` | `force`: always re-download · `off`: use the local copy, no Kaggle call |
| `DSC_KAGGLE_ROOT` | `/content/kaggle_data` | where dataset versions are unpacked (use a volume path on Modal) |
| `DSC_WORK_DIR` | `/vol/dsc2026/work` outside Colab | caches: point it at the volume that holds your v12 caches |
| `DSC_ACCEPT_CORPUS` | — | `1` only if a changed corpus is known to keep its row order |
| `KAGGLE_API_TOKEN` or `KAGGLE_KEY` · `HF_TOKEN` | — | Kaggle download · adapter download and push (on Modal: the `kaggle-secret` and `huggingface-secret` secrets) |

### Credentials

Nothing is hardcoded. CELL 0 resolves each secret from **Colab Secrets (🔑) → environment → masked prompt**.

> ⚠️ **Rotate the HF token and the Kaggle key that were pasted into chat.**

## Run order: private test (1,918 questions)

| | cell | what it does | A100 time |
|---|---|---|---|
| ✅ | **0** | credentials, **current Kaggle version** (printed with its file listing) | 2–5 min |
| ✅ | **2 · 3** | config, **test-set discovery** (prints the file and count), ingestion, labels | 2 min |
| ✅ | **4 – 6 · 6b · 6c** | BM25, dense index (adopted encoder, corpus stamp), retrieval, reranker | 15 min |
| ✅ | **6d · 6e · 6f** | the labs re-apply their saved decisions. If HyDE is adopted, 6f drafts the new questions | seconds · 5–8 min |
| ✅ | **7** | contexts for the 1,918 questions, checked against the test file | 10–20 min |
| ✅ | **T4 · 9 · 10** | adapter, helpers, val gate (cached) | 5 min |
| ✅ | **11** | inference. Resumes after a disconnect; answers are reused wherever the exact prompt was already answered | ≈ 1.5–3 h |
| ✅ | **12 · 13** | assemble, guards (ids and count against the test file), package, push, verify | 3 min |

**Before you upload,** check four lines in the output:

- **CELL 0:** the listing shows the new private file.
- **CELL 2:** `test set: … → 1,918 questions ✓`.
- **CELL 12:** `✓ ALL GUARDS PASSED`.
- **CELL 13:** `✓ 1,918 answers = the 1,918 questions of …`.

The archive is `submission_n1918_<set>.json.zip`, so it can't be mistaken for a public-phase file.

**On Modal**, `modal run scripts/run_notebook_modal.py --check-only` checks the volume first, and the same command without `--check-only` runs this notebook unchanged on an A100-80GB.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 0 — Credentials, Kaggle data, HF adapter
# ══════════════════════════════════════════════════════════════════════
# ── build stamp ───────────────────────────────────────────────────────
# Printed first so you always know which revision is running. Colab keeps a
# stale copy whenever a notebook is re-uploaded under the same name, and a
# revision number in the output is the only reliable way to tell.
NB_VERSION = "v13.2-private"
NB_CHANGES = {
 "v7.1": "training cells T1-T4 added to the verified v6 pipeline",
 "v7.2": "cell 13 diagnoses a missing SUB_ZIP instead of raising NameError",
 "v7.3": "prerequisite guards on 17 cells — name the missing dep and its cell",
 "v7.4": "gen_batch 8->32 (free-VRAM aware), train batch 1->4, MAX_TRAIN_EXAMPLES + ETAs",
 "v7.5": "param gate: counts the training peak, verifies tie_word_embeddings",
 "v7.6": "reranker front-load (512-tok cap) + lexref on + guarded fuzzy known-QA",
 "v7.7": "CELL 10c dedup lab — METEOR aligns 1-to-1, so duplicated tokens are pure cost",
 "v7.8": "CELL 10d supervised clause selection + CITE_STRATEGY switch in cell 12",
 "v7.9": "T2: batch derived from free VRAM (logits tensor is the constraint) + hard 25GB gate",
 "v8.0": "ROOT CAUSE: JAX preallocates 75% of the GPU. XLA_PYTHON_CLIENT_PREALLOCATE=false restored to CELL 0",
 "v8.1": "no-SFT path: train ctx opt-in, T4 no-ops when unchanged, cell 12 auto-adopts the measured winner",
 "v8.2-inference": "SFT cells REMOVED — Run All cannot start a training run",
 "v12.0": "retrieval labs 6e/6f, label resolution, encoder round 2, fuzzy calibration",
 "v13.0-private": "PRIVATE TEST: current Kaggle version, test file discovered + verified, test-keyed caches",
 "v13.1-private": "private test holds 1,918 questions (DSC_EXPECTED_N_TEST default)",
 "v13.2-private": "runs on Modal: KAGGLE_KEY accepted, adopted-encoder paths re-rooted to the work dir",
}
print(f"NOTEBOOK {NB_VERSION} — {NB_CHANGES[NB_VERSION]}")
print(f"  history: {' | '.join(sorted(NB_CHANGES))}\n")

# Tokens resolve Colab Secrets → env → masked prompt. Never hardcoded: Colab
# saves cell OUTPUT into the .ipynb and cell 12 can push to a public HF repo,
# so a literal token ships itself.
# ══ GPU PREALLOCATION — MUST BE THE FIRST THING THIS NOTEBOOK DOES ══
# Colab preloads JAX and TensorFlow. JAX grabs 75% of the GPU the moment it
# initialises — 64 GB of an 80 GB card — and that memory is invisible to
# torch.cuda.memory_allocated(), which is why it reads as "held outside the
# allocator". It is not a leaked model. These must be set BEFORE any import
# that touches JAX/TF, so they live in the very first cell.
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"]   = "platform"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"]     = "true"
os.environ["PYTORCH_CUDA_ALLOC_CONF"]       = "expandable_segments:True"
import sys
_late = [m for m in ("jax", "jaxlib", "tensorflow") if m in sys.modules]
if _late:
    print(f"⚠ {_late} were ALREADY imported before these settings took effect.")
    print("  They may already hold GPU memory. Runtime → Restart session and run")
    print("  this cell FIRST, before anything else.\n")

import re, json, stat, time, getpass, shutil, subprocess, unicodedata
from pathlib import Path
from collections import defaultdict

KAGGLE_OWNER = "phucdangg"
DATA_SLUG    = f"{KAGGLE_OWNER}/legalqa-task2-clean-data"
ADAPTER_SLUG = f"{KAGGLE_OWNER}/qwen25-3b-legal-lora"
HF_ADAPTER   = "dangphuc2109/legalqa-qwen2.5-3b-adapter"
HF_SUBMIT    = "dangphuc2109/legalqa-qwen2.5-3b-e"   # submissions land here
BASE_MODEL   = "Qwen/Qwen2.5-3B-Instruct"
LOCAL_ROOT   = Path(os.environ.get("DSC_KAGGLE_ROOT", "/content/kaggle_data"))   # Modal: a volume path
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import userdata; IN_COLAB = True
except ImportError:
    IN_COLAB = False

def resolve(name, *, required=True, hint=""):
    if IN_COLAB:
        try:
            v = userdata.get(name)
            if v and v.strip(): print(f"  {name}: Colab Secrets ✓"); return v.strip()
        except userdata.SecretNotFoundError:
            print(f"  {name}: no such secret → 🔑 Secrets → + Add new secret")
        except userdata.NotebookAccessError:
            print(f"  {name}: locked → 🔑 Secrets → Notebook access ON  ← usual cause")
        except Exception as e:
            print(f"  {name}: {type(e).__name__}")
    v = os.environ.get(name, "").strip()
    if v: print(f"  {name}: environment ✓"); return v
    if IN_COLAB:
        if hint: print(f"      {hint}")
        try: v = getpass.getpass(f"      Paste {name} (hidden, Enter to skip): ").strip()
        except Exception: v = ""
        if v: os.environ[name] = v; print(f"  {name}: entered ✓"); return v
    if required: raise RuntimeError(f"{name} unavailable — add it to 🔑 Secrets.")
    return None

print("resolving credentials…")
# v13.2: Modal's kaggle-secret (made from .env by run_modal.py) carries KAGGLE_KEY, not
# KAGGLE_API_TOKEN. Same token, other name: accept it.
if not os.environ.get("KAGGLE_API_TOKEN", "").strip() and os.environ.get("KAGGLE_KEY", "").strip():
    os.environ["KAGGLE_API_TOKEN"] = os.environ["KAGGLE_KEY"].strip()
KAGGLE_TOKEN = resolve("KAGGLE_API_TOKEN", hint="Kaggle → Settings → API → Create New Token")
HF_TOKEN     = resolve("HF_TOKEN", required=False, hint="huggingface.co/settings/tokens (WRITE — cell 12 pushes)")

# ── Kaggle: write all three credential shapes; Colab pins varying CLI builds ──
kdir = Path.home()/".kaggle"; kdir.mkdir(parents=True, exist_ok=True)
(kdir/"kaggle.json").write_text(json.dumps({"username": KAGGLE_OWNER, "key": KAGGLE_TOKEN}))
(kdir/"access_token").write_text(KAGGLE_TOKEN)
for f in ("kaggle.json", "access_token"):
    os.chmod(kdir/f, stat.S_IRUSR | stat.S_IWUSR)          # 0600
os.environ.update(KAGGLE_API_TOKEN=KAGGLE_TOKEN, KAGGLE_USERNAME=KAGGLE_OWNER,
                  KAGGLE_KEY=KAGGLE_TOKEN)
if shutil.which("kaggle") is None:
    subprocess.run(["pip", "install", "-q", "--upgrade", "kaggle"], check=False)

def kaggle_auth_check():
    """Version-tolerant. The 1.7.x CLI rewrite dropped `--page-size`, so the
    probe that worked last month now exits non-zero with 'unrecognized
    arguments' — a CLI mismatch, NOT an auth failure. Walk a chain of probe
    forms; only a genuine credential error is fatal."""
    probes = [["kaggle","datasets","list","-m","--page-size","1"],
              ["kaggle","datasets","list","-m"],
              ["kaggle","datasets","list","--mine"],
              ["kaggle","config","view"]]
    last = ""
    for cmd in probes:
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode == 0:
            return True, " ".join(cmd[1:]), ""
        out = (r.stderr or r.stdout) or ""
        last = out
        low = out.lower()
        if any(s in low for s in ("401","403","unauthorized","forbidden",
                                  "invalid","credential","authenticat")):
            return False, " ".join(cmd[1:]), out          # real auth failure
        if "unrecognized arguments" in low or "invalid choice" in low:
            continue                                      # CLI version skew
    return None, "", last                                 # inconclusive

_ok, _via, _err = kaggle_auth_check()
if _ok is True:
    print(f"  Kaggle auth OK as {KAGGLE_OWNER}  (via `kaggle {_via}`)")
elif _ok is False:
    raise RuntimeError(
        f"Kaggle auth FAILED (`kaggle {_via}`):\n{_err[:400]}\n"
        f"Check: token not expired · copied whole including the KGAT_ prefix · "
        f"belongs to '{KAGGLE_OWNER}'.")
else:
    # Every probe form was rejected by this CLI build. Do not block on it —
    # the download below is the real test and fails loudly on its own.
    print(f"  Kaggle auth: inconclusive (CLI build rejected every probe form); "
          f"the download will verify it")

def pull(slug, dest, marker):
    """Idempotent: `marker` proves the unzip finished. A non-empty directory
    alone would treat a download interrupted at 4% as cached."""
    dest = Path(dest)
    if next(dest.rglob(marker), None) is not None:
        print(f"  {slug}: cached"); return dest
    dest.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(dest),
                        "--unzip", "--force"], capture_output=True, text=True)
    assert r.returncode == 0, f"{slug}: {(r.stderr or r.stdout)[:400]}"
    print(f"  {slug} → {dest}"); return dest

# ── v13: the data is ALWAYS the dataset's CURRENT Kaggle version ──────
# The private test ships as a NEW VERSION of the same Kaggle dataset. v12 skipped
# Kaggle whenever any older copy sat on disk, so a live runtime or a persistent
# volume would have gone on answering the 1,000 public questions. Now the
# dataset's current file listing (names, sizes, dates) fingerprints the version:
# each version is unpacked into its own folder, a folder is reused only when the
# listing is identical, and older folders are never modified or deleted.
#   DSC_DATA_REFRESH = auto   (default) download again whenever the listing changed
#                    = force  always download a fresh copy
#                    = off    newest local copy, no Kaggle call (offline re-runs)
import hashlib
DATA_REFRESH  = os.environ.get("DSC_DATA_REFRESH", "auto").strip().lower()
DATA_VERSIONS = LOCAL_ROOT / "data"
assert DATA_REFRESH in ("auto", "force", "off"), f"DSC_DATA_REFRESH={DATA_REFRESH!r}: use auto, force or off"

def kaggle_listing(slug):
    """The dataset's CURRENT file listing as normalised rows, or None when no
    CLI form answers (version skew: 1.7.x dropped some flags, older builds
    want -d)."""
    forms = [["kaggle", "datasets", "files", slug, "--csv", "--page-size", "200"],
             ["kaggle", "datasets", "files", "-d", slug, "--csv", "--page-size", "200"],
             ["kaggle", "datasets", "files", slug, "--csv"],
             ["kaggle", "datasets", "files", "-d", slug, "--csv"],
             ["kaggle", "datasets", "files", slug],
             ["kaggle", "datasets", "files", "-d", slug]]
    for cmd in forms:
        try:
            r = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
        except Exception:
            continue
        if r.returncode != 0:
            continue
        rows = [re.sub(r"\s+", " ", l).strip() for l in r.stdout.splitlines()]
        rows = [l for l in rows if l and set(l) - set("- ")
                and not l.lower().startswith(("warning", "next page"))]
        if len(rows) >= 2:                            # a header and at least one file
            return rows
    return None

def _complete(d):
    d = Path(d)
    return (d / ".complete.json").exists() and next(d.rglob("legal_chunks.parquet"), None) is not None

def pull_version(slug, listing):
    """Download the current version into a folder of its own. It is unpacked
    under a temporary name and renamed only once complete, so an interrupted
    download can never pass for a finished one."""
    tag = (hashlib.sha1("\n".join(listing).encode("utf-8")).hexdigest()[:10] if listing
           else time.strftime("t%Y%m%d-%H%M%S"))
    dest = DATA_VERSIONS / f"v_{tag}"
    if listing and _complete(dest):
        print(f"  {slug}: current version already on disk ({dest.name} — listing unchanged)")
        return dest
    tmp = DATA_VERSIONS / f".partial_{tag}_{os.getpid()}"
    tmp.mkdir(parents=True)
    r = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(tmp),
                        "--unzip", "--force"], capture_output=True, text=True)
    assert r.returncode == 0, f"{slug}: {(r.stderr or r.stdout)[:400]}"
    files = sorted(str(p.relative_to(tmp)) for p in tmp.rglob("*") if p.is_file())
    assert any(f.endswith("legal_chunks.parquet") for f in files), (
        f"{slug}: the download holds no legal_chunks.parquet — files: {files[:12]}")
    (tmp / ".complete.json").write_text(json.dumps(
        {"slug": slug, "listing": listing, "files": files,
         "time": time.strftime("%Y-%m-%d %H:%M:%S")}, ensure_ascii=False, indent=1))
    if dest.exists():                                 # an incomplete folder from an older attempt
        dest = DATA_VERSIONS / f"v_{tag}_{time.strftime('%H%M%S')}"
    os.replace(tmp, dest)
    print(f"  {slug} → {dest}  ({len(files)} files)")
    return dest

def newest_local():
    """The newest COMPLETE version folder, else the pre-v13 folder, else None."""
    vs = sorted((d for d in DATA_VERSIONS.glob("v_*") if d.is_dir() and _complete(d)),
                key=lambda d: (d / ".complete.json").stat().st_mtime, reverse=True)
    if vs:
        return vs[0]
    legacy = LOCAL_ROOT / "artifacts-task2"
    return legacy if next(legacy.rglob("legal_chunks.parquet"), None) is not None else None

def _fsig(p, span=1 << 20):
    """Size + first/last MiB — tells two parquet files apart without hashing 1 GB."""
    n = os.path.getsize(p); h = hashlib.sha1()
    with open(p, "rb") as f:
        h.update(f.read(span))
        if n > span:
            f.seek(max(span, n - span)); h.update(f.read(span))
    return f"{n}:{h.hexdigest()[:12]}"

print("\nfetching Kaggle data…")
DATA_VERSIONS.mkdir(parents=True, exist_ok=True)
for _d in DATA_VERSIONS.glob(".partial_*"):
    shutil.rmtree(_d, ignore_errors=True)            # a download this cell started and never finished
_prev_root, DATA_FRESH, _listing = newest_local(), False, None
if DATA_REFRESH == "off":
    assert _prev_root is not None, "DSC_DATA_REFRESH=off, but there is no local copy of the data"
    KAGGLE_DATA_ROOT = _prev_root
    print(f"  ⚠ DSC_DATA_REFRESH=off — using {KAGGLE_DATA_ROOT} WITHOUT checking Kaggle for a newer version")
else:
    if DATA_REFRESH == "auto":
        _listing = kaggle_listing(DATA_SLUG)
        if _listing is None:
            print("  ⚠ the dataset's file listing is unreadable with this CLI build — "
                  "downloading a fresh copy so the version cannot be stale")
    try:
        KAGGLE_DATA_ROOT, DATA_FRESH = pull_version(DATA_SLUG, _listing), True
    except Exception as _ex:
        if _prev_root is None:
            raise
        KAGGLE_DATA_ROOT = _prev_root
        print(f"  ⚠ download FAILED ({type(_ex).__name__}: {str(_ex)[:200]})\n"
              f"    falling back to the local copy {_prev_root}. It may be the OLD version:\n"
              f"    CELL 2 stops the run unless its test file holds the expected question count.")
    if _listing:
        print(f"  Kaggle's current listing ({len(_listing) - 1} files):")
        for _l in _listing[:25]:
            print(f"    {_l[:140]}")
CORPUS_PREV = None                 # set when the corpus FILE differs from the previous copy
if _prev_root is not None and Path(_prev_root) != Path(KAGGLE_DATA_ROOT):
    _a = next(Path(_prev_root).rglob("legal_chunks.parquet"))
    _b = next(Path(KAGGLE_DATA_ROOT).rglob("legal_chunks.parquet"))
    if _fsig(_a) == _fsig(_b):
        print(f"  corpus: legal_chunks.parquet identical to the previous copy ({_prev_root.name}) ✓")
    else:
        CORPUS_PREV = str(_a)
        print(f"  ⚠ corpus: legal_chunks.parquet DIFFERS from the previous copy ({_prev_root.name}).\n"
              f"    Cached embeddings and BM25 are row-aligned to the file they were built from.\n"
              f"    CELL 3 compares the rows; if they moved, CELL 5 stops before pairing them.")
pull(ADAPTER_SLUG, LOCAL_ROOT/"lora", "adapter_config.json")   # the adapter did not change

def find_up(root, marker, levels=1):
    """`levels` = .parent hops FROM THE MARKER FILE. levels=1 is the directory
    containing it; levels=0 would return the file itself (a classic trap)."""
    hits = sorted(Path(root).rglob(marker))
    assert hits, f"{marker} not found under {root}"
    p = hits[0]
    for _ in range(levels + marker.count("/")): p = p.parent
    return p

KAGGLE_DATA_DIR    = find_up(KAGGLE_DATA_ROOT, "legal_chunks.parquet", levels=1)
KAGGLE_ADAPTER_DIR = find_up(LOCAL_ROOT/"lora", "adapter_config.json", levels=1)
print(f"\ndata    → {KAGGLE_DATA_DIR}   (version folder {Path(KAGGLE_DATA_ROOT).name}"
      f"{', fresh from Kaggle' if DATA_FRESH else ''})")
print(f"adapter → {KAGGLE_ADAPTER_DIR}")

# ── Hugging Face ──────────────────────────────────────────────────────
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HOME"] = "/content/hf_cache"     # local NVMe: the HF cache is
                                                # millions of small files and
                                                # Drive's FUSE layer chokes.
try: import hf_transfer                                            # noqa: F401
except ImportError: subprocess.run(["pip","install","-q","hf_transfer"], check=False)
from huggingface_hub import login, snapshot_download, HfApi

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print(f"  HF auth OK as {HfApi(token=HF_TOKEN).whoami().get('name')}")
_api = HfApi(token=HF_TOKEN)

def pick_adapter_subfolder(files):
    """This repo keeps the adapter under checkpoints/generator/hf_adapter/ AND
    carries a mid-training snapshot under runs/checkpoint-3/ with its own
    adapter_config.json. Loading that one gives weights from training step 3 —
    it imports fine and generates fluent nonsense. Shallowest wins; runs/ and
    checkpoint-<n>/ are excluded."""
    cands = [f for f in files if f.endswith("adapter_config.json")]
    good = [f for f in cands if "/runs/" not in f and not re.search(r"/checkpoint-\d+/", f)]
    pool = good or cands
    if not pool: return None, cands
    pool.sort(key=lambda f: (f.count("/"), len(f)))
    return os.path.dirname(pool[0]), cands

def fetch_adapter(repo_id, token=None, local_fallback=None):
    try:
        info = _api.model_info(repo_id, token=token); rev = info.sha
        sub, cands = pick_adapter_subfolder([s.rfilename for s in (info.siblings or [])])
        if sub is None: raise FileNotFoundError(f"no adapter_config.json in {repo_id}")
        print(f"  subfolder: {sub or '<root>'}"
              + (f"  (ignoring {len(cands)-1} mid-training checkpoint(s))" if len(cands)>1 else ""))
        allow  = [f"{sub}/*"] if sub else ["*.json","*.safetensors","*.model","*.txt","*.jinja"]
        ignore = [f"{sub}/runs/*"] if sub else ["runs/*"]
        snap = snapshot_download(repo_id, revision=rev, token=token, max_workers=8,
                                 allow_patterns=allow, ignore_patterns=ignore)
        d = os.path.join(snap, sub) if sub else snap
        assert os.path.exists(os.path.join(d, "adapter_config.json"))
        print(f"  {repo_id} → {d}  (rev {str(rev)[:10]})")
        return d
    except Exception as ex:
        print(f"  ⚠ Hub fetch failed ({type(ex).__name__}: {str(ex)[:140]})")
        fb = str(local_fallback) if local_fallback else None
        if fb and os.path.isfile(fb): fb = os.path.dirname(fb)
        if fb and os.path.exists(os.path.join(fb, "adapter_config.json")):
            print(f"  → Kaggle copy: {fb}"); return fb
        raise

print("\nfetching the PEFT adapter…")
ADAPTER_DIR = fetch_adapter(HF_ADAPTER, HF_TOKEN, KAGGLE_ADAPTER_DIR)
_ac = json.load(open(os.path.join(ADAPTER_DIR, "adapter_config.json"), encoding="utf-8"))
print(f"  r={_ac.get('r')} alpha={_ac.get('lora_alpha')} "
      f"modules_to_save={_ac.get('modules_to_save')}")
print(f"  base: {_ac.get('base_model_name_or_path')}")
print("\n✅ CELL 0 complete")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 1 — Environment + native encoder / reranker
# ══════════════════════════════════════════════════════════════════════
os.environ["PYTORCH_CUDA_ALLOC_CONF"]       = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"]        = "false"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # restated: cell 0 sets
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"]     = "true"    # these before any import
!pip install -q "transformers==4.49.0" "peft==0.14.0" "accelerate==1.4.0"
!pip install -q "faiss-cpu==1.10.0" "bm25s>=0.2.7,<0.3" "pyvi==0.1.1" "pyarrow>=15"

import torch, transformers, peft, faiss, gc, random, pickle, gzip, math
import numpy as np, pandas as pd
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModel, AutoModelForCausalLM,
                          AutoConfig, AutoModelForSequenceClassification)
from peft import PeftModel

assert torch.cuda.is_available(), "No GPU — Runtime → Change runtime type → A100."
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16_OK = torch.cuda.is_bf16_supported()
DTYPE   = torch.bfloat16 if BF16_OK else torch.float16
try: import flash_attn; ATTN_IMPL = "flash_attention_2"
except ImportError: ATTN_IMPL = "sdpa"
print(f"torch {torch.__version__} | numpy {np.__version__} | "
      f"transformers {transformers.__version__} | peft {peft.__version__}")
print(f"{torch.cuda.get_device_name(0)} {VRAM_GB:.0f}GB | {DTYPE} | attn={ATTN_IMPL}")

# Baseline check: with nothing loaded, torch should hold ~0 and the process
# should hold only the CUDA context (~0.5 GB). Anything more is another library
# preallocating — almost always JAX at 75% of the card.
_f0, _t0g = torch.cuda.mem_get_info()
_other = (_t0g - _f0)/1e9 - torch.cuda.memory_reserved()/1e9
print(f"  baseline VRAM: free {_f0/1e9:.1f}/{_t0g/1e9:.0f} GB | "
      f"non-torch {_other:.1f} GB")
if _other > 5.0:
    print(f"  ⚠ {_other:.1f} GB ({_other/(_t0g/1e9):.0%} of the card) is held by something")
    print(f"     other than torch. {'That is JAX preallocating 75%.' if _other/(_t0g/1e9) > 0.6 else ''}")
    print(f"     Runtime → Restart session and run CELL 0 FIRST — it sets")
    print(f"     XLA_PYTHON_CLIENT_PREALLOCATE=false, which only works before")
    print(f"     JAX initialises.")
else:
    print(f"  ✓ GPU is clean — full capacity available")


class NativeEncoder:
    """AutoModel → attention-masked MEAN pooling → L2. sentence-transformers is
    deliberately not used: v3 cannot read this model's modules.json and v5
    cannot run on transformers 4.49, and the generator path is the one thing
    that already works."""
    def __init__(self, model_id, device="cuda", dtype=None, max_len=128):
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModel.from_pretrained(
            model_id, torch_dtype=(dtype or torch.float32)).to(device).eval()
        self.device, self.max_len = device, max_len

    @torch.inference_mode()
    def encode(self, texts, batch_size=256, normalize_embeddings=True,
               convert_to_numpy=True, show_progress_bar=False):
        if isinstance(texts, str): texts = [texts]
        texts, out = list(texts), []
        for i in range(0, len(texts), batch_size):
            enc = self.tok(texts[i:i+batch_size], padding=True, truncation=True,
                           max_length=self.max_len, return_tensors="pt").to(self.device)
            h = self.model(**enc).last_hidden_state
            m = enc["attention_mask"].unsqueeze(-1).to(h.dtype)
            # masked mean: pad positions add nothing to the sum OR the count.
            # Averaging over pads instead shifts every vector — a silent killer.
            v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            if normalize_embeddings:
                v = torch.nn.functional.normalize(v, p=2, dim=1)
            out.append(v.float().cpu().numpy())
        return np.concatenate(out) if convert_to_numpy else out


class NativeCrossEncoder:
    """bge-reranker-v2-m3: XLMRobertaForSequenceClassification, num_labels=1.
    The single logit IS the score; sigmoid is monotonic so ranking is unchanged."""
    def __init__(self, model_id, device="cuda", dtype=None, max_length=512):
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_id, torch_dtype=(dtype or torch.float32)).to(device).eval()
        self.device, self.max_length = device, max_length

    @torch.inference_mode()
    def predict(self, pairs, batch_size=128, **kw):
        pairs, out = list(pairs), []
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            # two-argument call builds the proper <s>q</s></s>passage</s> pair
            # encoding; a pre-joined string would score a different input
            enc = self.tok([str(p[0]) for p in b], [str(p[1]) for p in b],
                           padding=True, truncation=True, max_length=self.max_length,
                           return_tensors="pt").to(self.device)
            out.append(self.model(**enc).logits.view(-1).float().cpu().numpy())
        return np.concatenate(out) if out else np.zeros(0, dtype=np.float32)

print("  ✓ NativeEncoder + NativeCrossEncoder ready")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 2 — Config, paths, parameter-budget gate
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['KAGGLE_DATA_DIR'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 0 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

import dataclasses
try:                                   # Colab: the caches live on Drive
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    _WORK_DIR = "/content/drive/MyDrive/dsc2026/work"
except ImportError:                    # Modal / any Linux container: a persistent volume
    _WORK_DIR = os.environ.get("DSC_WORK_DIR", "/vol/dsc2026/work")
    print(f"not on Colab → work_dir {_WORK_DIR}\n  Set DSC_WORK_DIR to a folder on a PERSISTENT "
          f"volume — the one holding your v11 caches — or every cache is rebuilt from scratch.")

@dataclasses.dataclass
class CFG:
    data_dir: str = str(KAGGLE_DATA_DIR)
    work_dir: str = _WORK_DIR

    emb_model_id: str = "CODE4LIFEOFFICIAL/huydang-dek21-embedding-v2"
    emb_dim: int = 768
    emb_max_len: int = 256           # 258 positions − 2 (RoBERTa offset)
    query_max_len: int = 128         # legal questions exceed the old 64 cap
    reranker_id: str = "BAAI/bge-reranker-v2-m3"
    llm_id: str = "Qwen/Qwen2.5-3B-Instruct"

    top_k_context: int = 8
    max_parts_per_article: int = 2
    cand_pool: int = 100
    rrf_k: int = 60

    max_chunk_chars: int = 1200
    max_seq_len: int = 5632
    max_new_tokens: int = 1400
    repetition_penalty: float = 1.0   # see cell 8 — >1.0 suppresses the statute

    cite_mode: str = "article"        # "article" | "chunk" | "sniper"
    cite_budget: int = 4000           # measured optimum (val 0.5398)
    cite_n_chunks: int = 1            # a 2nd article fragments and hurts

    n_ret_val: int = 400
    n_gen_val: int = 300
    seed: int = 42
    use_known_qa_lookup: bool = True

cfg = CFG()
os.makedirs(cfg.work_dir, exist_ok=True)
W = lambda *p: os.path.join(cfg.work_dir, *p)

def find_file(*names):
    roots = [Path(cfg.data_dir)] + ([Path(KAGGLE_DATA_ROOT)] if globals().get("KAGGLE_DATA_ROOT") else [])
    for root in roots:                   # v13: the chunks' folder first, then the whole version
        for n in names:
            hits = sorted(root.rglob(n))
            if hits: return str(hits[0])
    return None

# ── v13: the TEST SET is discovered and verified, never assumed ───────
# The private phase ships a new test file whose name this notebook does not
# hard-code. Every test-like file in the CURRENT dataset version is loaded,
# normalised to {id: {"question": ...}} and counted, and the file holding
# exactly EXPECTED_N_TEST questions is used. If two files with DISJOINT ids add
# up to it (a public and a private file, say), both are answered. Anything else
# stops here, with the table of what was found — before any GPU time is spent.
#   DSC_EXPECTED_N_TEST   question count to insist on (0 = accept any)
#   DSC_TEST_FILE         file name(s) or path(s), comma-separated, to force the choice
#   DSC_TEST_ID_FIELD     take ids from this record field instead of the dict keys
EXPECTED_N_TEST    = int(os.environ.get("DSC_EXPECTED_N_TEST", "1918") or 0)   # private test: 1,918
TEST_FILE_OVERRIDE = [s.strip() for s in os.environ.get("DSC_TEST_FILE", "").split(",") if s.strip()]
TEST_ID_FIELD      = os.environ.get("DSC_TEST_ID_FIELD", "").strip()   # records' id field, if not the keys
_ID_KEYS = ("id", "qa_id", "qid", "question_id", "questionid", "ques_id", "uid", "_id", "stt", "index")
_Q_KEYS  = ("question", "query", "question_raw", "question_text", "cau_hoi", "câu hỏi",
            "q", "text", "content")
_NOT_TEST = {"legal_chunks.parquet", "qa_unique.parquet", "retrieval_labels.parquet",
             "known_qa.json", "artifacts.json", "splits_v2.json", "dataset-metadata.json",
             ".complete.json"}
_TESTISH  = re.compile(r"test|private|public|official|question|quer", re.I)
_WARNED   = set()

def _field(rec, keys):
    low = {str(k).strip().lower(): k for k in rec}
    return next((low[k] for k in keys if k in low), None)

def _id_str(x):
    if isinstance(x, float) and x.is_integer(): return str(int(x))
    return str(getattr(x, "item", lambda: x)()).strip()           # numpy scalars → python

def load_test_any(path):
    """Any test layout → {id: {"question": text, ...}}, file order kept.
    JSON dict of records or of strings, JSON list of records (also wrapped one
    level deep, e.g. {"data": [...]}), JSONL, CSV/TSV and parquet. Records
    without an id, and duplicate ids, are errors: collapsing them would
    silently drop questions."""
    import pandas as _pd
    p, ext = str(path), os.path.splitext(str(path))[1].lower()
    if ext == ".parquet":
        raw = _pd.read_parquet(p).to_dict("records")
    elif ext in (".csv", ".tsv"):
        raw = _pd.read_csv(p, sep="\t" if ext == ".tsv" else ",", dtype=str,
                           keep_default_na=False).to_dict("records")
    elif ext == ".jsonl":
        with open(p, encoding="utf-8") as f:
            raw = [json.loads(l) for l in f if l.strip()]
    else:
        with open(p, encoding="utf-8") as f:
            raw = json.load(f)
    if isinstance(raw, dict) and raw and not all(isinstance(v, (dict, str)) for v in raw.values()):
        lists = [v for v in raw.values() if isinstance(v, list) and v and isinstance(v[0], dict)]
        if lists:
            raw = max(lists, key=len)                                # {"data": [...]} wrapper
    out, name = {}, os.path.basename(p)
    if isinstance(raw, dict):
        # The outer KEY is the id — exactly how v12 read public-official.json,
        # which Codabench scored. One layout makes that doubtful: keys that are
        # just 0..n-1 (pandas orient="index") while every record carries its own,
        # different id. The keys still win (proven path), loudly; DSC_TEST_ID_FIELD
        # switches to the records' field.
        _recs = [v for v in raw.values() if isinstance(v, dict)]
        if TEST_ID_FIELD and _recs and len(_recs) == len(raw):
            raw = _recs                                             # ids from the named field
        elif _recs and len(_recs) == len(raw):
            _ik = [_field(v, _ID_KEYS) for v in _recs]
            if (all(_ik) and list(map(str, raw)) == [str(i) for i in range(len(raw))]
                    and any(_id_str(v[k]) != str(i) for i, (v, k) in enumerate(zip(_recs, _ik)))):
                if p not in _WARNED:
                    _WARNED.add(p)
                    print(f"  ⚠ {name}: its keys are 0..{len(raw) - 1} (row numbers?) but every record "
                          f"has its own '{_ik[0]}'. Using the KEYS as ids, as v12 did; if Codabench "
                          f"expects the '{_ik[0]}' values, set DSC_TEST_ID_FIELD={_ik[0]}.")
    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, str):
                out[_id_str(k)] = {"question": v}
            elif isinstance(v, dict):
                qk = _field(v, _Q_KEYS)
                out[_id_str(k)] = {**v, "question": v.get(qk, "") if qk else ""}
    elif isinstance(raw, list):
        for i, r in enumerate(raw):
            if not isinstance(r, dict):
                raise ValueError(f"{name}: record {i} is a {type(r).__name__}, not an object")
            ik = _field(r, (TEST_ID_FIELD.lower(),) if TEST_ID_FIELD else _ID_KEYS)
            qk = _field(r, _Q_KEYS)
            if ik is None or r[ik] is None or not str(r[ik]).strip():
                raise ValueError(f"{name}: record {i} has no id (looked for {', '.join(_ID_KEYS)})")
            q = _id_str(r[ik])
            if q in out:
                raise ValueError(f"{name}: duplicate id {q!r} (records {list(out).index(q)} and {i})")
            out[q] = {**r, "question": r.get(qk, "") if qk else ""}
    else:
        raise ValueError(f"{name}: top level is {type(raw).__name__}")
    for v in out.values():
        _q = v["question"]
        v["question"] = unicodedata.normalize("NFC", "" if _q is None else str(_q)).strip()
    return out

def load_test_set(files):
    """One file, or several with disjoint ids, merged in the given order."""
    merged = {}
    for f in files:
        part = load_test_any(f)
        _dup = set(part) & set(merged)
        assert not _dup, (f"{os.path.basename(f)} repeats {len(_dup)} ids of the files before it "
                          f"(e.g. {sorted(_dup)[:3]}) — they cannot be answered as one set")
        merged.update(part)
    return merged

def discover_test_files():
    """→ (chosen files, table of every candidate as (path, n questions, note))."""
    root = Path(globals().get("KAGGLE_DATA_ROOT") or cfg.data_dir)
    if TEST_FILE_OVERRIDE:
        chosen = []
        for s in TEST_FILE_OVERRIDE:
            p = Path(s) if os.path.isabs(s) else next(iter(sorted(root.rglob(s))), None)
            assert p is not None and Path(p).is_file(), f"DSC_TEST_FILE: {s!r} not found under {root}"
            chosen.append(str(p))
        return chosen, [(c, len(load_test_any(c)), "forced by DSC_TEST_FILE") for c in chosen]
    def scan(pred):
        rows = []
        for p in sorted(root.rglob("*")):
            if (not p.is_file() or p.name in _NOT_TEST or "bm25" in str(p).lower()
                    or p.suffix.lower() not in (".json", ".jsonl", ".csv", ".tsv", ".parquet")
                    or not pred(p)):
                continue
            try:
                rows.append((str(p), len(load_test_any(p)), ""))
            except Exception as ex:
                rows.append((str(p), -1, f"unreadable: {type(ex).__name__}: {str(ex)[:90]}"))
        return rows
    cands = scan(lambda p: bool(_TESTISH.search(p.name)))
    if EXPECTED_N_TEST and not any(n == EXPECTED_N_TEST for _, n, _ in cands):
        cands += scan(lambda p: not _TESTISH.search(p.name) and p.stat().st_size < 64 << 20)
    ok = [c for c in cands if c[1] > 0]
    pref = lambda c: (0 if re.search("private", Path(c[0]).name, re.I) else
                      1 if re.search("test", Path(c[0]).name, re.I) else 2,
                      -c[1], -os.path.getmtime(c[0]))
    if EXPECTED_N_TEST:
        exact = sorted((c for c in ok if c[1] == EXPECTED_N_TEST), key=pref)
        if exact:
            if len(exact) > 1:
                _sig = lambda f: frozenset((q, v["question"]) for q, v in load_test_any(f).items())
                if (len({_sig(c[0]) for c in exact}) > 1
                        and pref(exact[0])[0] == pref(exact[1])[0]):
                    raise RuntimeError(
                        f"{len(exact)} DIFFERENT files each hold {EXPECTED_N_TEST:,} questions:\n"
                        + "\n".join(f"    {c[0]}" for c in exact)
                        + "\n  set DSC_TEST_FILE to the one to answer")
            return [exact[0][0]], cands
        from itertools import combinations
        for a, b in combinations(sorted(ok, key=pref), 2):
            if a[1] + b[1] == EXPECTED_N_TEST and not set(load_test_any(a[0])) & set(load_test_any(b[0])):
                return [a[0], b[0]], cands
        raise RuntimeError(
            f"no test file holds {EXPECTED_N_TEST:,} questions (DSC_EXPECTED_N_TEST) in {root}:\n"
            + ("\n".join(f"    {n:>7,}  {c}  {note}" for c, n, note in cands) or "    (no candidate files)")
            + "\n  If the Kaggle version is stale, re-run CELL 0 (DSC_DATA_REFRESH=force). If the"
              "\n  count really changed, set DSC_EXPECTED_N_TEST; to pick a file, set DSC_TEST_FILE.")
    assert ok, f"no readable test file under {root}: {cands}"
    return [sorted(ok, key=pref)[0][0]], cands

TEST_FILES, _TEST_CANDS = discover_test_files()
N_TEST = len(load_test_set(TEST_FILES))
print(f"test set: {' + '.join(os.path.basename(f) for f in TEST_FILES)} → {N_TEST:,} questions"
      + (f"  ✓ = DSC_EXPECTED_N_TEST" if EXPECTED_N_TEST and N_TEST == EXPECTED_N_TEST else ""))
for _c, _n, _note in _TEST_CANDS:
    if _c not in TEST_FILES:
        print(f"  not used: {os.path.relpath(_c, str(globals().get('KAGGLE_DATA_ROOT') or cfg.data_dir))}"
              f" ({_n:,} questions{'; ' + _note if _note else ''})")
if EXPECTED_N_TEST:
    assert N_TEST == EXPECTED_N_TEST, f"{N_TEST:,} test questions, expected {EXPECTED_N_TEST:,}"

PATHS = {"chunks": find_file("legal_chunks.parquet"),
         "qa":     find_file("qa_unique.parquet"),
         "labels": find_file("retrieval_labels.parquet"),
         "known":  find_file("known_qa.json"),
         "test":   TEST_FILES[0],                 # v13: discovered and verified above
         "emb":    W("embeddings_rebuilt.npy"),
         "bm25":   (lambda h: str(h[0]) if h else None)(
                        sorted(Path(cfg.data_dir).rglob("bm25s_index"))
                        or sorted(Path(globals().get("KAGGLE_DATA_ROOT") or cfg.data_dir).rglob("bm25s_index")))}
for k, v in PATHS.items():
    print(f"  {'OK     ' if v and os.path.exists(str(v)) else 'absent '}  {k:<7} {v or ''}")
assert PATHS["chunks"] and PATHS["qa"] and PATHS["test"], "core inputs missing"

# gen_batch derived, not guessed. Qwen2.5-3B has intermediate_size 11008, so a
# prefill activation tensor is batch × seq × 11008 × 2 bytes:
#   batch 32 × 3350 tok -> 2.36 GB per tensor, ~7 GB for the MLP triple
#   batch  8 × 5632 tok -> 0.99 GB per tensor, ~3 GB
# Batch 32 is what OOM'd. generate_answers() halves adaptively from here anyway,
# so a conservative start costs a little throughput and buys reliability.
# Measured: 5.5 s/answer at batch 32. Decode is memory-bandwidth bound, so a
# bigger batch amortises weight reads — batch 8 makes a 92-min inference pass
# take ~3 hours. The earlier OOM happened with ~60 GB already stranded, not
# because batch 32 is inherently too large (6 GB weights + ~12 GB activations
# on a clean 80 GB card). generate_answers() halves adaptively now, so start
# high and let it back off if it must.
_free_gb = torch.cuda.mem_get_info()[0] / 1e9
_gb = 32 if _free_gb > 50 else 16 if _free_gb > 30 else 8 if _free_gb > 18 else 4
PROFILE = {"emb_batch": 512, "rerank_batch": 128, "gen_batch": _gb}
print(f"profile: {PROFILE}   ({_free_gb:.0f} GB free → gen_batch {_gb})")
print(f"  est. full inference: ~{N_TEST*5.5/ (1.0 if _gb>=32 else 0.78 if _gb>=16 else 0.52)/60:.0f} min "
      f"for {N_TEST:,} answers (less for answers whose prompt is unchanged)")

def vram(tag=""):
    """PyTorch-allocated vs whole-process. A large gap means memory is held
    outside the allocator — almost always a model stranded by an earlier run."""
    f, t = torch.cuda.mem_get_info()
    alloc, reserv = torch.cuda.memory_allocated()/1e9, torch.cuda.memory_reserved()/1e9
    used = (t - f)/1e9
    print(f"  VRAM{(' '+tag) if tag else ''}: free {f/1e9:.1f}/{t/1e9:.0f} GB | "
          f"torch alloc {alloc:.1f} reserved {reserv:.1f} | process {used:.1f}")
    if used - reserv > 8.0:
        _frac = (used - reserv) / (t/1e9)
        _why = ("JAX preallocating 75% of the card — CELL 0 must run FIRST, before "
                "any import touches JAX" if _frac > 0.6 else
                "a model stranded by an earlier cell")
        print(f"    ⚠ {used-reserv:.0f} GB ({_frac:.0%}) held OUTSIDE torch's allocator")
        print(f"       likely cause: {_why}")
        print(f"       Runtime → Restart session, then run cells from CELL 0.")
    return f/1e9
vram("at start")

def set_seed(s=cfg.seed):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

def free_vram(*names):
    g = globals()
    for n in names:
        if n in g: del g[n]
    gc.collect(); torch.cuda.empty_cache()

# ── competition gate: total parameters must stay under 4.0B ───────────
def count_params(mid, verbose=False):
    """Meta-device count: exact, no weights downloaded, no VRAM.

    AutoModel.from_config returns the BASE model (Qwen2Model, RobertaModel …),
    which has no lm_head. That is correct ONLY when the checkpoint ties its
    input and output embeddings. Qwen2.5-3B does tie them — but an untied
    checkpoint would add vocab_size × hidden_size (311M here) that this gate
    would never see, and 3.781B + 0.311B = 4.09B is OVER the ceiling. So check
    rather than assume.
    """
    _cfg = AutoConfig.from_pretrained(mid)
    with torch.device("meta"):
        m = AutoModel.from_config(_cfg)
    n = sum(p.numel() for p in m.parameters())
    _tied = getattr(_cfg, "tie_word_embeddings", None)
    if _tied is False:
        _extra = getattr(_cfg, "vocab_size", 0) * getattr(_cfg, "hidden_size", 0)
        n += _extra
        if verbose: print(f"      + lm_head {_extra/1e6:.0f}M (embeddings NOT tied)")
    elif verbose and _tied is True:
        print(f"      tie_word_embeddings=True → lm_head shares the embedding matrix")
    return n

_tot, PARAM_AUDIT = 0, {}
for role, mid in (("generator", cfg.llm_id), ("retriever", cfg.emb_model_id),
                  ("reranker", cfg.reranker_id)):
    n = count_params(mid, verbose=True); _tot += n
    PARAM_AUDIT[role] = {"repo": mid, "params": int(n)}
    print(f"  {role:<10} {mid:<48} {n/1e9:>6.3f}B")

# LoRA is only present between get_peft_model() and merge_and_unload(); after
# merging it folds into the base weights and adds nothing. Budget for the peak.
_lora_est = 32 * 36 * (2*(2048+2048) + 2*(2048+256) + 3*(2048+11008))
print(f"  {'+ LoRA':<10} {'r=32, 7 modules x 36 layers (training only)':<48} "
      f"{_lora_est/1e9:>6.3f}B")
PARAM_AUDIT["lora_peak"] = int(_lora_est)
print(f"  {'-'*76}")
print(f"  {'MERGED':<10} {'deployed inference stack':<48} {_tot/1e9:>6.3f}B / 4.000B")
print(f"  {'PEAK':<10} {'during training (base + LoRA + retrieval)':<48} "
      f"{(_tot+_lora_est)/1e9:>6.3f}B / 4.000B")
print(f"  headroom: {(4e9-_tot)/1e6:.0f}M merged | {(4e9-_tot-_lora_est)/1e6:.0f}M at peak"
      f"  →  {'PASS' if _tot+_lora_est <= 4e9 else 'FAIL'}")
assert _tot <= 4_000_000_000, f"MERGED STACK EXCEEDS THE CEILING: {_tot/1e9:.3f}B"
assert _tot + _lora_est <= 4_000_000_000, (
    f"TRAINING PEAK EXCEEDS THE CEILING: {(_tot+_lora_est)/1e9:.3f}B. Lower LORA_R "
    f"in cell T2 — r=16 halves the adapter to {_lora_est/2/1e6:.0f}M.")
PARAM_AUDIT["total_merged"], PARAM_AUDIT["total_peak"] = int(_tot), int(_tot+_lora_est)


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 3 — Data ingestion (schema-adaptive), labels, frozen splits
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['cfg', 'W', 'PATHS'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 2 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# Never pass hardcoded `columns=` to read_parquet. Read the FOOTER first (free),
# resolve each canonical field to whichever physical alias exists, then project.
import pyarrow.parquet as pq

CHUNK_ALIASES = {
    "chunk_id":        ["chunk_id", "id", "cid"],
    "content":         ["content", "text_raw", "raw_text", "text", "passage", "text_norm"],
    "context_id":      ["context_id", "parent_article_id", "doc_id", "document_id"],
    "document_number": ["document_number", "doc_name", "name", "document_title"],
    "document_title":  ["document_title", "doc_name", "name", "document_number"],
    "name":            ["name", "doc_name", "document_title", "document_number"],
    "article_number":  ["article_number", "article", "dieu_so", "parent_article_id"],
    "article_title":   ["article_title"], "dieu": ["dieu"],
    "khoan":           ["khoan", "clause_number", "clause"], "structure": ["structure"],
}
QA_ALIASES    = {"id": ["id","qa_id"], "question": ["question","question_raw","question_norm"],
                 "answer": ["answer","answer_raw","gold_answer"]}
LABEL_ALIASES = {"qa_id": ["qa_id","id"], "query": ["query","question","question_raw"],
                 "answer": ["answer","answer_raw"], "citations": ["citations"],
                 "positive_chunk_id": ["positive_chunk_id","gold_chunk_id"],
                 "positive_article_id": ["positive_article_id","gold_article_id"],
                 "positive_doc_name": ["positive_doc_name","gold_doc_name"]}

def load_adaptive(path, aliases, required, defaults=None):
    present = list(pq.ParquetFile(path).schema_arrow.names)
    mapping = {}
    for canon, cands in aliases.items():
        for c in cands:
            if c in present: mapping[canon] = c; break
    missing = [r for r in required if r not in mapping]
    if missing:
        raise KeyError(f"{os.path.basename(path)}: cannot resolve {missing}\n  present: {present}")
    df = pd.read_parquet(path, columns=sorted(set(mapping.values())))
    out = pd.DataFrame(index=df.index)
    for canon, phys in mapping.items(): out[canon] = df[phys]
    for canon, val in (defaults or {}).items():
        if canon not in out.columns: out[canon] = val
    print(f"  {os.path.basename(path)}: {len(out):,} rows | "
          f"{len(present)} physical → {len(out.columns)} canonical")
    return out

def norm_key(s):
    """Applied to BOTH sides of every join."""
    if s is None or (isinstance(s, float) and np.isnan(s)): return ""
    s = unicodedata.normalize("NFC", str(s)).lower().strip()
    return re.sub(r"\s+", " ", re.sub(r"^điều\s+", "", s))

def has_val(x):
    if x is None or (isinstance(x, float) and np.isnan(x)): return False
    return str(x).strip() not in ("", "None", "nan", "<NA>")

def load_json_any(path):
    with open(path, encoding="utf-8") as f: raw = json.load(f)
    if isinstance(raw, list):
        return {str(r.get("id") or r.get("qa_id")): r for r in raw}
    return raw

chunks = load_adaptive(PATHS["chunks"], CHUNK_ALIASES, ["chunk_id","content"],
                       defaults={"structure":"dieu","article_title":"","dieu":None,"khoan":None})
chunks["chunk_id"] = chunks["chunk_id"].astype(str).astype(object)
chunks["content"]  = chunks["content"].fillna("").astype(str)
for c in ("document_number","document_title","name","article_number"):
    if c not in chunks.columns: chunks[c] = ""

# 'dieu' is a DISPLAY string ("Điều 17"). This corpus holds non-numeric ids like
# "preamble_0", and "Điều preamble_0" would be printed into every prompt.
if chunks["dieu"].isna().all():
    _an = chunks["article_number"].astype(str).str.strip()
    _num = _an.str.fullmatch(r"\d{1,3}(?:[.\-]\d{1,3})?").fillna(False)
    chunks["dieu"] = ("Điều " + _an).where(_num, other=pd.NA)
    print(f"  dieu synthesised for {int(_num.sum()):,}/{len(chunks):,} numeric ids")

# chunk_id is NOT unique here. Do NOT drop rows: embeddings.npy and the BM25
# index are aligned POSITIONALLY to parquet row order, so removing a row would
# silently misalign every vector after it. Keep all rows; id→pos is one-to-many.
chunk_ids = chunks["chunk_id"].to_numpy()
chunk_pos_all = defaultdict(list)
for i, c in enumerate(chunk_ids): chunk_pos_all[c].append(i)
_nuniq = len(chunk_pos_all)
if _nuniq != len(chunks):
    print(f"  ⚠ duplicate chunk_id: {len(chunks)-_nuniq:,} extra rows across "
          f"{sum(1 for v in chunk_pos_all.values() if len(v)>1):,} ids — rows KEPT")

chunk_keys = [(norm_key(d), norm_key(a))
              for d, a in zip(chunks["document_number"], chunks["article_number"])]
SIB = defaultdict(list)
for i, k in enumerate(chunk_keys): SIB[k].append(i)
SIB = dict(SIB)
_named = sum(1 for k in SIB if k[0] and k[1])
print(f"  article index: {len(SIB):,} (doc, điều) keys | "
      f"{_named:,} ({_named/max(1,len(SIB)):.0%}) fully named → expandable")

qa = load_adaptive(PATHS["qa"], QA_ALIASES, ["id","question","answer"])
qa["id"] = qa["id"].astype(str).astype(object)
qa_by_id = qa.set_index("id")
qa_by_id = qa_by_id[~qa_by_id.index.duplicated(keep="first")]
# ── v13: the test set CELL 2 discovered, normalised to {id: {"question": …}} ──
import hashlib
test_data = load_test_set(TEST_FILES)
_empty_q = [q for q, v in test_data.items() if not v["question"]]
assert len(_empty_q) < max(1, len(test_data) // 2), (
    f"{len(_empty_q):,} of {len(test_data):,} test questions are EMPTY — the loader did not find the "
    f"question field. Fields seen: {sorted(next(iter(test_data.values())))[:12]}")
if _empty_q:
    print(f"  ⚠ {len(_empty_q)} empty test questions (ids {_empty_q[:3]}) — they still get an answer")
if EXPECTED_N_TEST:
    assert len(test_data) == EXPECTED_N_TEST, f"{len(test_data):,} test questions, expected {EXPECTED_N_TEST:,}"
TEST_FP  = hashlib.sha1(json.dumps(sorted((q, v["question"]) for q, v in test_data.items()),
                                   ensure_ascii=False).encode("utf-8")).hexdigest()[:10]
TEST_TAG = f"n{len(test_data)}_{TEST_FP[:8]}"
_dupq = len(test_data) - len({v["question"] for v in test_data.values()})
print(f"  test: {len(test_data):,} questions from {' + '.join(os.path.basename(f) for f in TEST_FILES)}"
      f" | set {TEST_FP}" + (f" | {_dupq} repeated question texts" if _dupq else ""))

# ── v13: the corpus ROWS, as every cached embedding and index sees them ──
def corpus_fp(ids, contents):
    h = hashlib.sha1("\x1f".join(map(str, ids)).encode("utf-8"))
    h.update(np.asarray([len(str(c)) for c in contents], dtype=np.int64).tobytes())
    return h.hexdigest()[:12]
CORPUS_FP, CORPUS_CHANGED = corpus_fp(chunks["chunk_id"], chunks["content"]), False
if globals().get("CORPUS_PREV"):           # CELL 0: the corpus FILE differs from the last copy
    _old = load_adaptive(CORPUS_PREV, {k: CHUNK_ALIASES[k] for k in ("chunk_id", "content")},
                         ["chunk_id", "content"])
    CORPUS_CHANGED = corpus_fp(_old["chunk_id"].astype(str), _old["content"].fillna("").astype(str)) != CORPUS_FP
    print("  corpus rows vs the previous copy: "
          + ("MOVED — CELL 5 will not pair old embeddings with them" if CORPUS_CHANGED
             else "identical ✓ (the file differs only in its bytes)"))
    del _old

def build_known_lookup(obj):
    """Find the largest question→answer mapping one level deep. The file's
    top-level key names vary between releases, so hardcoding one silently
    yields zero overrides and forfeits free exact-match points."""
    cands = []
    def consider(d, path):
        if not isinstance(d, dict) or not d: return
        vals = list(d.values())[:200]
        if all(isinstance(v, dict) for v in vals):
            m = {str(v["question"]): str(v["answer"]) for v in d.values()
                 if isinstance(v, dict) and v.get("question") and v.get("answer")}
            if m: cands.append((len(m), path, m))
        elif all(isinstance(v, str) for v in vals):
            ks = list(d.keys())[:200]
            ql = sum(1 for k in ks if isinstance(k,str) and " " in k and len(k) > 15)
            if ql >= max(1, len(ks)//2):
                cands.append((len(d), path, {str(k): str(v) for k, v in d.items()}))
    consider(obj, "root")
    if isinstance(obj, dict):
        for k, v in obj.items(): consider(v, k)
    if not cands: return {}
    cands.sort(key=lambda x: -x[0])
    print(f"  known_qa: '{cands[0][1]}' with {cands[0][0]:,} entries")
    return cands[0][2]

known_by_q = build_known_lookup(load_json_any(PATHS["known"])) if PATHS["known"] else {}

# ── labels + frozen splits ────────────────────────────────────────────
labels = (load_adaptive(PATHS["labels"], LABEL_ALIASES, ["qa_id"])
          if PATHS["labels"] else pd.DataFrame(columns=["qa_id"]))
labels["qa_id"] = labels["qa_id"].astype(str).astype(object)

# ══ v12: LABEL RESOLUTION — make the gold labels speak the corpus's language ══
# The corpus names a document by its URL slug ('Nghi-dinh-100-2019-ND-CP-xu-phat-
# 16923'); a label may name it by its official number ('100/2019/NĐ-CP'), and an
# article by an id or 'Điều 17' instead of '17'. v11 joined them with norm_key()
# alone — lowercase and strip — so the two spellings never met. A gold label that
# cannot be resolved is not a retrieval failure; it is invisible. The label then
# drops out of RET_VAL and out of CELL 6d's training data, or it resolves to one
# exact chunk_id, so retrieving the right ARTICLE through a sibling chunk still
# scores as a miss.
#
# Everything below is a TRANSLATION into keys the corpus already uses. chunk_keys,
# SIB and hit() are unchanged, and so is every retrieval cache.
import functools
from collections import Counter
LABEL_MAX_DOC_AMBIG = 3        # a document key naming more corpus docs than this is not a match

def _fold(s):
    """Strip diacritics (đ→d) so 'NĐ-CP' and 'nd-cp' compare equal."""
    s = unicodedata.normalize("NFD", str(s).replace("đ", "d").replace("Đ", "D"))
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

_YEAR_TOK = re.compile(r"^(?:19|20)\d{2}$")
_FORMAL   = re.compile(r"(?<![\d/])(\d{1,4})\s*/\s*(?:((?:19|20)\d{2})\s*/\s*)?"
                       r"([A-Za-z][A-Za-z0-9]*(?:\s*-\s*[A-Za-z0-9]+)*)")
# Issuing-body codes, needed only for LOWERCASE slugs (in 'Nghi-dinh-100-2019-ND-CP'
# the codes are recognisable by case). The label side adds its own codes below.
_SYM_VOCAB = {"ND", "CP", "TT", "QD", "TTG", "NQ", "CT", "QH", "L", "CTN", "UBND", "HDND",
              "UBTVQH", "BTC", "BYT", "BCA", "BQP", "BTP", "BNV", "BCT", "BXD", "BGDDT",
              "BLDTBXH", "BTNMT", "BGTVT", "BKHDT", "BNNPTNT", "BTTTT", "BVHTTDL", "BKHCN",
              "NHNN", "TANDTC", "VKSNDTC", "KTNN", "VBHN", "TTLT", "HD", "BTTB", "TLD", "BHXH"}

def _sym_norm(s):
    return "-".join(t for t in re.split(r"[\s\-]+", _fold(s).upper()) if t)

def doc_sig(s):
    """(number, year|None, symbol|None) from either spelling, or None.
         '100/2019/NĐ-CP', 'Nghị định số 100/2019/NĐ-CP'   → ('100', '2019', 'ND-CP')
         '405/QĐ-BNV'                                     → ('405', None,   'QD-BNV')
         'Nghi-dinh-100-2019-ND-CP-xu-phat-16923'         → ('100', '2019', 'ND-CP')
         'Quyet-dinh-405-QD-BNV-2021-phe-duyet-468351'    → ('405', '2021', 'QD-BNV')
         'Bo-luat-Dan-su-1995-44-L-CTN-39391'             → ('44',  '1995', 'L-CTN')"""
    if not has_val(s):
        return None
    raw = unicodedata.normalize("NFC", str(s)).strip()
    if "/" in raw:
        m = _FORMAL.search(_fold(raw))
        if m:
            return (str(int(m.group(1))), m.group(2), _sym_norm(m.group(3)))
    toks = [t for t in re.split(r"[-_\s]+", _fold(raw)) if t]
    num = year = None; sym = []
    for t in toks:
        if num is None:
            if t.isdigit() and len(t) <= 4:
                if _YEAR_TOK.match(t):
                    year = year or t          # 'Bo-luat-Dan-su-1995-44-…': the year comes first
                else:
                    num = str(int(t))
            continue
        if _YEAR_TOK.match(t) and year is None:
            year = t; continue
        if re.fullmatch(r"[A-Za-z]+\d{0,2}", t) and (t.isupper() or t.upper() in _SYM_VOCAB
                                                     or re.fullmatch(r"(?i)qh\d{1,2}", t)):
            sym.append(t.upper()); continue
        break                                 # first title word: the number is complete
    return (num, year, "-".join(sym) or None) if num else None

_ART_RE = re.compile(r"(?:điều|dieu|article|art)[\s_\-.:#]*(\d{1,4}[a-zđ]?)(?![\w])")
_ART_BARE = re.compile(r"(\d{1,4}[a-zđ]?)\.?")

@functools.lru_cache(maxsize=None)
def art_key(s):
    """'17', 'Điều 17', 'điều 17a.', '…_dieu_17' → '17' / '17a'; None if no article."""
    if not has_val(s):
        return None
    t = unicodedata.normalize("NFC", str(s)).lower().strip()
    m = _ART_RE.search(t) or _ART_BARE.fullmatch(t)
    return m.group(1) if m else None

# label-side issuing-body codes teach the slug parser the lowercase spellings
for _v in (labels["positive_doc_name"] if "positive_doc_name" in labels.columns else []):
    _sg = doc_sig(_v)
    if _sg and _sg[2]: _SYM_VOCAB.update(_sg[2].split("-"))
if "citations" in labels.columns:
    for _cits in labels["citations"]:
        for _c in (_cits if _cits is not None else []):
            _sg = doc_sig(dict(_c).get("document_number"))
            if _sg and _sg[2]: _SYM_VOCAB.update(_sg[2].split("-"))

# ── corpus-side indexes (8.5k documents, not 800k rows, for the signatures) ──
_DOC_SLUGS = {k for k, _ in SIB if k}
_BY_NUM = defaultdict(set)                     # number → {(slug, year, body)}
for _dn, _dt in chunks[["document_number", "document_title"]].drop_duplicates().itertuples(index=False):
    _slug = norm_key(_dn)
    if not _slug: continue
    for _src in (_dn, _dt):                    # the title often carries the official number too
        _sg = doc_sig(_src)
        if _sg: _BY_NUM[_sg[0]].add((_slug, _sg[1], _sg[2]))
_ART_IDX, _DOC_POS = defaultdict(list), defaultdict(list)
_an_raw = chunks["article_number"].to_numpy()
for _i, (_d, _a) in enumerate(chunk_keys):
    if not _d: continue
    _DOC_POS[_d].append(_i)
    _ak = art_key(_an_raw[_i])
    if _ak: _ART_IDX[(_d, _ak)].append(_i)
_CTX_IDX = defaultdict(list)                   # article-id join, if both sides carry one
if "context_id" in chunks.columns:
    for _i, _c in enumerate(chunks["context_id"].to_numpy()):
        if has_val(_c): _CTX_IDX[norm_key(_c)].append(_i)

def _body_ok(a, b):
    """'ND-CP' vs 'ND-CP' or a longer parse of the same code ('ND-CP-BHXH')."""
    a, b = a.split("-"), b.split("-")
    return a[:len(b)] == b or b[:len(a)] == a

def resolve_doc(ref):
    """Corpus document keys a reference names, and how they were found. A
    candidate must share the number and CONTRADICT nothing: a label year of 2019
    never matches a 2015 decree with the same number and body."""
    k = norm_key(ref)
    if not k: return set(), "none"
    if k in _DOC_SLUGS: return {k}, "exact"
    sg = doc_sig(ref)
    if not sg: return set(), "no number"
    n, y, b = sg
    best, cands = -1, set()
    for slug, cy, cb in _BY_NUM.get(n, ()):
        if (y and cy and y != cy) or (b and cb and not _body_ok(b, cb)):
            continue
        sc = bool(y and cy) + bool(b and cb)            # fields that positively AGREE
        if sc > best: best, cands = sc, {slug}
        elif sc == best: cands.add(slug)
    if best <= 0: return set(), ("number only" if cands else "not in corpus")
    if len(cands) > LABEL_MAX_DOC_AMBIG: return set(), f"ambiguous ({len(cands)} docs)"
    return cands, ("number/year/body" if best == 2 else "number + year or body")

def resolve_ref(doc, art, cid):
    """One citation → (positions, keys, source). Every route that succeeds is
    unioned: a chunk-id label ALSO gets its article, so a sibling chunk counts."""
    pos, keys, src = set(), set(), []
    if has_val(cid) and str(cid) in chunk_pos_all:
        pos.update(chunk_pos_all[str(cid)]); src.append("chunk_id")
    if has_val(art) and norm_key(art) in _CTX_IDX:
        pos.update(_CTX_IDX[norm_key(art)]); src.append("article_id")
    slugs, how = resolve_doc(doc) if has_val(doc) else (set(), "none")
    ak = art_key(art) if has_val(art) else None
    if slugs and ak:
        hit_ = [p for s in slugs for p in _ART_IDX.get((s, ak), [])]
        if hit_:
            pos.update(hit_); src.append(f"doc {how} + Điều")
    # article-level gold: every chunk resolved so far contributes its (doc, Điều) key
    keys.update(chunk_keys[p] for p in pos if chunk_keys[p][0] and chunk_keys[p][1])
    if slugs and not ak and not pos:           # the citation names a document only
        for s in slugs:
            keys.add((s, "")); pos.update(_DOC_POS.get(s, []))
        src.append(f"doc {how}, no Điều")
    return pos, keys, (src or [f"unresolved: doc {how}" + ("" if ak or not has_val(art) else ", Điều unreadable")])

if "citations" in labels.columns:
    _refs = [[(dict(c).get("document_number"), dict(c).get("article"), None)
              for c in (cits if cits is not None else [])] for cits in labels["citations"]]
else:
    _nul = pd.Series([None] * len(labels), index=labels.index)
    _refs = [[(d, a, c)] for d, a, c in zip(labels.get("positive_doc_name", _nul),
                                            labels.get("positive_article_id", _nul),
                                            labels.get("positive_chunk_id", _nul))]
_resolved = [[resolve_ref(*r) for r in rs] for rs in _refs]
labels["gold_positions"] = [set().union(*[x[0] for x in rr]) if rr else set() for rr in _resolved]
labels["gold_keys"]      = [set().union(*[x[1] for x in rr]) if rr else set() for rr in _resolved]
labels["gold_chunk_ids"] = [sorted({str(chunk_ids[p]) for p in ps})[:50] for ps in labels["gold_positions"]]
labels["res_src"]        = [sorted({s for x in rr for s in x[2]}) for rr in _resolved]
labels["resolvable"]     = [len(p) > 0 or len(k) > 0
                            for p, k in zip(labels["gold_positions"], labels["gold_keys"])]
if "query" not in labels.columns or labels["query"].isna().all():
    labels["query"] = [qa_by_id.at[q, "question"] if q in qa_by_id.index else ""
                       for q in labels["qa_id"]]

# ONE ROW PER QUESTION. A question citing three articles had three rows, and
# labels_by_qid.loc[q] then handed .iloc[0] — the FIRST citation only — to every
# consumer. Retrieving the second cited article scored as a miss.
# (a plain loop, not groupby.agg: agg's handling of set-valued results differs
#  between pandas 2 and 3, and 6k rows do not need vectorising)
_n_rows = len(labels)
_by_q = {}
_has_ans = "answer" in labels.columns
for _r in labels.itertuples(index=False):
    _q = _by_q.get(_r.qa_id)
    if _q is None:
        _by_q[_r.qa_id] = _q = {"qa_id": _r.qa_id, "query": "", "answer": "",
                                "gold_positions": set(), "gold_keys": set(), "res_src": set()}
    if not has_val(_q["query"]) and has_val(_r.query): _q["query"] = _r.query
    if _has_ans and not has_val(_q["answer"]) and has_val(_r.answer): _q["answer"] = _r.answer
    _q["gold_positions"] |= _r.gold_positions; _q["gold_keys"] |= _r.gold_keys
    _q["res_src"] |= set(_r.res_src)
labels = pd.DataFrame(list(_by_q.values()))
labels["gold_chunk_ids"] = [sorted({str(chunk_ids[p]) for p in ps})[:50] for ps in labels["gold_positions"]]
labels["res_src"]    = [sorted(s) for s in labels["res_src"]]
labels["resolvable"] = [len(p) > 0 or len(k) > 0
                        for p, k in zip(labels["gold_positions"], labels["gold_keys"])]
if not _has_ans: labels = labels.drop(columns=["answer"])

# ── resolution report — what CELL 6c's "labels_unresolved" measures ────
_src_rows = Counter(s for rr in _resolved for x in rr for s in x[2])
print(f"\n  LABEL RESOLUTION  {_n_rows:,} label rows → {len(labels):,} questions "
      f"({_n_rows - len(labels):,} extra citation rows merged)")
for _s, _n in sorted(_src_rows.items(), key=lambda kv: -kv[1]):
    print(f"    {_s:<44} {_n:>6,}")
_art_lvl = sum(1 for k in labels["gold_keys"] if any(a for _, a in k))
print(f"  resolved: {int(labels['resolvable'].sum()):,}/{len(labels):,} questions "
      f"({labels['resolvable'].mean():.1%}) | article-level gold for {_art_lvl:,}")
_unres = [r for rs, rr in zip(_refs, _resolved) for r, x in zip(rs, rr) if not x[0] and not x[1]]
for _d, _a, _c in _unres[:3]:
    print(f"    unresolved sample: doc={str(_d)[:60]!r} art={str(_a)[:30]!r} chunk={str(_c)[:30]!r}"
          f" → sig {doc_sig(_d)}")
LABEL_REPORT = {"rows": _n_rows, "questions": len(labels),
                "resolved": float(labels["resolvable"].mean()), "by_source": dict(_src_rows)}
labels_by_qid = labels.set_index("qa_id")

_sp = W("splits_v2.json")
if os.path.exists(_sp):
    splits = json.load(open(_sp))
else:
    _rng = np.random.default_rng(cfg.seed)
    _res = labels.loc[labels["resolvable"], "qa_id"].tolist()
    _rv = list(_rng.choice(_res, size=min(cfg.n_ret_val, len(_res)), replace=False)) if _res else []
    _pool = [q for q in qa["id"] if q not in set(_rv)]
    _gv = list(_rng.choice(_pool, size=min(cfg.n_gen_val, len(_pool)), replace=False))
    splits = {"ret_val": [str(x) for x in _rv], "gen_val": [str(x) for x in _gv]}
    json.dump(splits, open(_sp, "w"))
RET_VAL, GEN_VAL = set(splits["ret_val"]), set(splits["gen_val"])
print(f"\nchunks {len(chunks):,} | qa {len(qa):,} | test {len(test_data):,} | "
      f"known {len(known_by_q):,} | ret_val {len(RET_VAL)} | gen_val {len(GEN_VAL)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 4 — BM25 (sparse channel)
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['chunks', 'has_val', 'norm_key'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 3 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

import bm25s
from pyvi import ViTokenizer

def title_of(row):
    if has_val(row["document_title"]): return str(row["document_title"])
    t = re.sub(r"[-_]+", " ", str(row["name"] or "")).strip()
    return re.sub(r"\s*\d{4,}\s*$", "", t)

def chunk_header(pos):
    row = chunks.iloc[int(pos)]; t = title_of(row)
    if has_val(row["dieu"]):
        art = str(row["dieu"]).split(".")[0].strip()
        kh = f", khoản {row['khoan']}" if has_val(row["khoan"]) else ""
        return f"{art}{kh} — {t}"
    return t

def index_text(row):
    parts = [title_of(row)]
    if has_val(row["dieu"]): parts.append(str(row["dieu"]))
    parts.append(row["content"])
    return " . ".join(p for p in parts if p)

_IT = W("index_texts_v2.pkl.gz")
if os.path.exists(_IT):
    with gzip.open(_IT) as f: index_texts = pickle.load(f)
    print(f"loaded index_texts ({len(index_texts):,})")
else:
    index_texts = [index_text(r) for _, r in tqdm(chunks.iterrows(), total=len(chunks),
                                                  desc="compose")]
    with gzip.open(_IT, "wb") as f: pickle.dump(index_texts, f)
assert len(index_texts) == len(chunks)

bm25, BM25_OK = None, False
if PATHS["bm25"]:
    try:
        bm25 = bm25s.BM25.load(PATHS["bm25"], mmap=False); BM25_OK = True
        print(f"loaded prebuilt BM25 from {PATHS['bm25']}")
    except Exception as e:
        print(f"prebuilt BM25 load failed ({type(e).__name__}) — rebuilding")
if not BM25_OK:
    _ST = W("seg_texts_v2.pkl.gz")
    if os.path.exists(_ST):
        with gzip.open(_ST) as f: seg_texts = pickle.load(f)
    else:
        seg_texts = [ViTokenizer.tokenize(t.lower()) for t in tqdm(index_texts, desc="pyvi")]
        with gzip.open(_ST, "wb") as f: pickle.dump(seg_texts, f)
    bm25 = bm25s.BM25(k1=0.9, b=0.4)
    bm25.index(bm25s.tokenize(seg_texts, stopwords=None, show_progress=True))
    bm25.save(W("bm25_index_v2"))

def bm25_search(queries, k=None):
    k = k or cfg.cand_pool
    seg = [ViTokenizer.tokenize(str(q).lower()) for q in queries]
    idx, sc = bm25.retrieve(bm25s.tokenize(seg, stopwords=None, show_progress=False),
                            k=min(k, len(chunks)), show_progress=False)
    return np.asarray(idx), np.asarray(sc)

_i, _ = bm25_search(["giấy chứng nhận kiểm dịch động vật"], k=3)
assert _i.max() < len(chunks), "BM25 index out of sync with the chunk table"
print("BM25 smoke:", [str(chunk_ids[i]) for i in _i[0]])


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 5 — Dense index (cached embeddings: the base ones, or CELL 6d's)
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['chunks', 'PATHS', 'NativeEncoder', 'chunk_pos_all', 'CORPUS_FP'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 3 (and CELL 1) first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# Validity is decided by a QUERY-FREE test: rows whose chunk text is IDENTICAL
# must embed to cosine ≈ 1.0. Same text, same encoder, same vector. Anisotropy
# is only advisory — some encoders are deliberately whitened toward isotropy.
FORCE_REENCODE = False       # leave False: the rebuild costs ~50 min

def validate_embeddings(emb, n_probe=4000, max_pairs=400):
    rng = np.random.default_rng(0)
    sel = np.sort(rng.choice(len(emb), min(n_probe, len(emb)), replace=False))
    s = np.asarray(emb[sel], dtype=np.float32)
    s /= np.clip(np.linalg.norm(s, axis=1, keepdims=True), 1e-12, None)
    nan_r, zero_r = int(np.isnan(s).any(1).sum()), int((np.abs(s).sum(1) < 1e-8).sum())
    half = len(s)//2
    bg = s[:min(500, half)] @ s[half:half+2000].T
    # filter to identical content FIRST, then cap — capping first leaves ~10 pairs
    pairs = []
    for c, p in chunk_pos_all.items():
        if len(p) > 1 and chunks["content"].iat[p[0]] == chunks["content"].iat[p[1]]:
            pairs.append((p[0], p[1]))
            if len(pairs) >= max_pairs: break
    dup = None
    if pairs:
        A = np.asarray(emb[[a for a,_ in pairs]], dtype=np.float32)
        B = np.asarray(emb[[b for _,b in pairs]], dtype=np.float32)
        A /= np.clip(np.linalg.norm(A,axis=1,keepdims=True),1e-12,None)
        B /= np.clip(np.linalg.norm(B,axis=1,keepdims=True),1e-12,None)
        dup = float(np.mean(np.sum(A*B, axis=1)))
    print(f"  NaN {nan_r} | zero {zero_r} | background {bg.mean():+.4f} ± {bg.std():.4f}"
          f" (random would be {1/np.sqrt(emb.shape[1]):.4f})")
    if dup is not None:
        print(f"  IDENTICAL-TEXT pairs: {len(pairs)} | mean cosine {dup:.4f} (must be ≈1.00)")
    ok = nan_r == 0 and zero_r == 0 and (dup is None or dup > 0.95)
    print("  ✓ real semantic vectors" if ok else "  ✗ INVALID")
    return ok, {"dup_cosine": dup, "nan": nan_r, "zero": zero_r}

assert not FORCE_REENCODE, ("FORCE_REENCODE=True, but this pipeline has no rebuild "
    "path — re-encoding is a ~50-minute job that lives in the older notebook.")
# ── v11: the base encoder, or the fine-tuned one CELL 6d adopted ──────
# Encoder and corpus vectors are chosen as a PAIR, from one record. A query
# vector from one model against corpus vectors from another scores near chance
# and raises no error — so the two are never picked separately.
USE_FT_ENCODER = True          # False → base encoder even if CELL 6d adopted one
ENCODER_TAG, ENCODER_SIG = "base", f"base:{cfg.emb_model_id}"
ENC_ID, EMB_PATH = cfg.emb_model_id, PATHS["emb"]
_ADOPT = W("ENCODER_ADOPTED.json")
if USE_FT_ENCODER and os.path.exists(_ADOPT):
    try:
        with open(_ADOPT, encoding="utf-8") as _f:
            _a = json.load(_f)
        # v13.2: the record holds ABSOLUTE paths from the session that wrote it
        # (Colab: /content/drive/MyDrive/…). When the work dir moved — Drive to a
        # Modal volume — the same names are looked up under the current one.
        for _k in ("dir", "emb"):
            _here = W(os.path.basename(str(_a[_k]).rstrip("/\\")))
            if not os.path.exists(str(_a[_k])) and os.path.exists(_here):
                print(f"  {os.path.basename(_ADOPT)}: '{_k}' re-rooted to the current work dir → {_here}")
                _a[_k] = _here
        if os.path.isdir(_a["dir"]) and os.path.exists(_a["emb"]):
            ENCODER_TAG, ENCODER_SIG = _a["tag"], f"{_a['tag']}@{_a['time']}"
            ENC_ID, EMB_PATH = _a["dir"], _a["emb"]
        else:
            print(f"⚠ {os.path.basename(_ADOPT)} points at files that are gone — base encoder")
    except Exception as _ex:
        print(f"⚠ {os.path.basename(_ADOPT)} unreadable ({type(_ex).__name__}) — base encoder")
print(f"encoder: {ENCODER_TAG}  ({ENC_ID})")
assert os.path.exists(EMB_PATH), (
    f"{EMB_PATH} not found. This pipeline REUSES validated embeddings; "
    f"if they are gone, rebuild them with the older notebook first.")
_e = np.load(EMB_PATH, mmap_mode="r")
print(f"embeddings: shape={_e.shape} dtype={_e.dtype} "
      f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")
assert _e.shape[0] == len(chunks), f"{_e.shape[0]:,} rows != {len(chunks):,} chunks"
assert _e.shape[1] == cfg.emb_dim
_ok, EMB_REPORT = validate_embeddings(_e)
assert _ok, "embeddings failed the identical-text test — do not build an index on them"

# ── v13: a vector file is row-aligned to ONE corpus ───────────────────
# Each embeddings file is stamped with the corpus it was validated against. The
# same file under a different corpus — or an unstamped file right after CELL 3
# saw the corpus rows move — stops here instead of pointing every dense hit at
# the wrong chunk.
_CS = W("corpus_stamp.json")
try:
    _stamps = json.load(open(_CS, encoding="utf-8"))
except Exception:
    _stamps = {}
def _esig(p, span=1 << 20):
    n = os.path.getsize(p); h = hashlib.sha1()
    with open(p, "rb") as f:
        h.update(f.read(span))
        if n > span:
            f.seek(max(span, n - span)); h.update(f.read(span))
    return f"{n}:{h.hexdigest()[:12]}"
_ek, _es = os.path.basename(str(EMB_PATH)), _esig(EMB_PATH)
_st = _stamps.get(_ek, {})
_known = _st.get("emb_sig") == _es
if ((_known and _st.get("corpus_fp") != CORPUS_FP)
        or (not _known and globals().get("CORPUS_CHANGED"))) and not os.environ.get("DSC_ACCEPT_CORPUS"):
    raise RuntimeError(
        f"{_ek} was built for a DIFFERENT corpus than this legal_chunks.parquet "
        f"({_st.get('corpus_fp', 'the previous Kaggle copy') if _known else 'the previous Kaggle copy'}"
        f" → {CORPUS_FP}).\nIts rows follow the old chunk order, so every dense hit would land on the "
        f"wrong text.\nRe-encode the corpus (and rebuild BM25) for this version — or, if the change "
        f"is known to keep\nthe row order, set DSC_ACCEPT_CORPUS=1.")
_stamps[_ek] = {"emb_sig": _es, "corpus_fp": CORPUS_FP, "time": time.strftime("%Y-%m-%d %H:%M")}
with open(_CS, "w", encoding="utf-8") as _f:
    json.dump(_stamps, _f, indent=1)
print(f"  corpus {CORPUS_FP} ↔ {_ek}: {'same pairing as last run' if _known else 'stamped'}")

corpus_emb = np.ascontiguousarray(_e, dtype=np.float32)   # ~2.5 GB; FAISS copies it
_n = np.linalg.norm(corpus_emb[:1000], axis=1)
if not np.allclose(_n, 1.0, atol=1e-2):
    corpus_emb /= np.clip(np.linalg.norm(corpus_emb, axis=1, keepdims=True), 1e-12, None)

faiss_index = faiss.IndexFlatIP(corpus_emb.shape[1])
for i in range(0, len(corpus_emb), 200_000):      # block add keeps the peak flat
    faiss_index.add(corpus_emb[i:i+200_000])
assert faiss_index.ntotal == len(chunks)
print(f"FAISS IndexFlatIP dim={corpus_emb.shape[1]} ntotal={faiss_index.ntotal:,}")

query_encoder = NativeEncoder(ENC_ID, device="cuda", dtype=DTYPE,
                              max_len=cfg.query_max_len)

def prep_query(q):
    """Query prep MUST mirror the corpus side: pyvi segmentation, no lowercase.
    A mismatch here is silent — nothing errors, the scores just get worse."""
    return ViTokenizer.tokenize(str(q))

def dense_search(queries, k=None, index=None):
    k = k or cfg.cand_pool
    ix = index if index is not None else faiss_index
    q = np.ascontiguousarray(query_encoder.encode([prep_query(x) for x in queries],
                             batch_size=256, normalize_embeddings=True), dtype=np.float32)
    sc, idx = ix.search(q, min(k, len(chunks)))
    return idx, sc

# ── alignment guard: is the query encoder aimed at the same space? ─────
# top-1 is the MAX over N candidates, and the max of N noise draws already sits
# at ~sqrt(2 ln N) ≈ 5.2σ for N=800k. So the floor must be derived from N, not
# hardcoded — an arbitrary constant here once blocked a perfectly healthy index.
_probe = ["giấy chứng nhận kiểm dịch động vật", "thủ tục đăng ký kinh doanh",
          "mức xử phạt vi phạm hành chính giao thông", "điều kiện vay vốn học sinh sinh viên",
          "hồ sơ đề nghị cấp giấy phép xây dựng", "quyền và nghĩa vụ của người lao động"]
_idx, _sc = dense_search(_probe, k=5)
_qv = np.ascontiguousarray(query_encoder.encode([prep_query(x) for x in _probe]),
                           dtype=np.float32)
_bgm = _qv @ corpus_emb[np.sort(np.random.default_rng(0).choice(len(corpus_emb), 20000,
                                replace=False))].T
_zq = (_sc[:,0] - _bgm.mean(1)) / np.clip(_bgm.std(1), 1e-9, None)   # PER QUERY:
# pooling the std across probes folds between-query spread into σ and deflates z
_N = len(chunks); _a = math.sqrt(2*math.log(_N))
_zc = _a - (math.log(math.log(_N)) + math.log(4*math.pi)) / (2*_a)
print(f"\nalignment: top-1 {_sc[:,0].mean():.3f} | z median {np.median(_zq):.1f} "
      f"| chance ceiling z {_zc:.1f} | floor 7.0")
for q, ids, ss in zip(_probe, _idx, _sc):
    print(f"  {q[:40]:<40} → {str(chunk_ids[ids[0]])[:26]:<26} {ss[0]:.3f}")
if np.median(_zq) < 7.0 and not ((EMB_REPORT["dup_cosine"] or 0) > 0.95):
    print("  ⚠ weak alignment AND weak corpus evidence — inspect before trusting dense")
else:
    print("  ✓ aligned")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6 — Hybrid retrieval: RRF(BM25, dense [, exact refs]) + reranker
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['bm25_search', 'dense_search', 'chunk_keys', 'index_texts'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 4 and 5 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# RRF fuses RANKS, so no score calibration is needed — BM25 scores and cosines
# live on incomparable scales. That also makes adding a third arm safe: when it
# returns nothing, fusion is unchanged.
USE_LEXREF = True       # exact legal-reference arm
# The index cache is versioned (_LEXV below) and validated against the current
# _core() when it loads, so the empty cache the broken v2 extractor wrote is
# rebuilt automatically — nothing to delete by hand.

_DOC = re.compile(r"(?:nghị\s*định|thông\s*tư|quyết\s*định|nghị\s*quyết|pháp\s*lệnh"
                  r"|bộ\s*luật|luật)\s*(?:số\s*)?"
                  r"(\d{1,4}\s*[/\-]\s*\d{2,4}(?:\s*/\s*[A-Za-zĐđ0-9\-]+)*)", re.I)
_NUM = re.compile(r"\b(\d{1,4}\s*/\s*\d{2,4}(?:\s*/\s*[A-Za-zĐđ0-9\-]+)*)")
_ART = re.compile(r"\bđiều\s*(\d{1,3}[a-zđ]?)\b", re.I)

def _core(n):
    """Extract the stable (number, year) key from a document identifier.

        '100/2019/NĐ-CP'                            -> '100/2019'
        'Nghi-dinh-100-2019-ND-CP-xu-phat-16923'    -> '100/2019'
        'Quyet-dinh-405-QD-BNV-2021-...-468351'     -> '405/2021'
        'Bo-luat-Dan-su-1995-44-L-CTN-39391'        -> '44/1995'

    THE BUG THIS REPLACES: the old version was

        re.match(r"(\\d{1,4})\\s*[/\\-]\\s*(\\d{2,4})", str(n).strip())

    and re.match ANCHORS at position 0. Every document_number in this corpus is
    a slug beginning with a letter, so it returned None for all 801,863 rows.
    Cell 6 printed 'lexref index: 0 document numbers, 0 (doc,điều) pairs' and
    every lexref_search returned [] — the third RRF arm was dead while
    USE_LEXREF=True made it read as enabled."""
    s = unicodedata.normalize("NFC", str(n))
    # form A — canonical "100/2019" or "100-2019", anywhere in the string
    m = re.search(r"(?<!\d)(\d{1,4})\s*[/\-]\s*((?:19|20)\d{2})(?!\d)", s)
    if m:
        return f"{int(m.group(1))}/{m.group(2)}"
    # form B — slugs put the issuing body between the number and the year:
    #   Quyet-dinh-405-QD-BNV-2021 → 405 … 2021.  Take the 1-4 digit group
    #   nearest the year, preferring the one before it.
    y = re.search(r"(?<!\d)((?:19|20)\d{2})(?!\d)", s)
    if not y:
        return None
    before = re.findall(r"(?<!\d)(\d{1,4})(?!\d)", s[:y.start()])
    after  = re.findall(r"(?<!\d)(\d{1,4})(?!\d)", s[y.end():])
    num = before[-1] if before else (after[0] if after else None)
    return f"{int(num)}/{y.group(1)}" if num else None

# The cache filename carries the version of the CODE that wrote it. The v2 file
# was written by the broken _core() as an empty dict, and a bare filename let
# corrected code silently inherit it — cell 6 printed "0 document numbers" from
# a cache, not from the corpus, and then made you delete a file by hand. A guard
# that needs manual intervention is a guard that failed. This one validates what
# it loads against the current _core() and rebuilds when they disagree.
_LEXV = "v3"
_LEXP = W(f"lexref_index_{_LEXV}.pkl.gz")

def _lex_cache_ok(di, dai):
    """Trust a cache only if it is non-empty AND still agrees with the CURRENT
    _core() on a sample of the corpus. Emptiness alone is not the test: a cache
    built by a subtly different extractor would load fine and mis-key every
    lookup, which is harder to notice than an empty index."""
    if not di or not dai: return False
    _probe = [str(x) for x in chunks["document_number"].astype(str).head(500)]
    _want  = {k for k in (_core(p) for p in _probe) if k}
    return bool(_want) and len(_want & set(di)) >= max(1, len(_want) // 2)

doc_idx = docart_idx = None
if os.path.exists(_LEXP):
    try:
        with gzip.open(_LEXP) as f:
            doc_idx, docart_idx = pickle.load(f)
        if not _lex_cache_ok(doc_idx, docart_idx):
            print(f"  cached {os.path.basename(_LEXP)} disagrees with the current "
                  f"_core() — rebuilding")
            doc_idx = docart_idx = None
        else:
            print(f"  loaded {os.path.basename(_LEXP)} (validated against _core())")
    except Exception as _e:
        print(f"  cache unreadable ({type(_e).__name__}) — rebuilding")
        doc_idx = docart_idx = None

if doc_idx is None:
    for _old in Path(cfg.work_dir).glob("lexref_index_v*.pkl.gz"):
        if _old.name != os.path.basename(_LEXP):
            try: _old.unlink(); print(f"  removed stale {_old.name}")
            except Exception: pass
    doc_idx, docart_idx = defaultdict(list), defaultdict(list)
    _dn = chunks["document_number"].astype(str).to_numpy()
    _an = chunks["article_number"].astype(str).to_numpy()
    for i in tqdm(range(len(chunks)), desc="lexref index"):
        c = _core(_dn[i])
        if not c: continue
        doc_idx[c].append(i)
        a = norm_key(_an[i])
        if a: docart_idx[(c, a)].append(i)
    doc_idx, docart_idx = dict(doc_idx), dict(docart_idx)
    with gzip.open(_LEXP, "wb") as f: pickle.dump((doc_idx, docart_idx), f)
print(f"lexref index: {len(doc_idx):,} document numbers, {len(docart_idx):,} (doc,điều) pairs")
_cov = sum(len(v) for v in doc_idx.values())
print(f"  chunks reachable by exact document number: {_cov:,}/{len(chunks):,} ({_cov/len(chunks):.1%})")
def lexref_search(queries, k=None):
    k = k or cfg.cand_pool
    out = []
    for q in queries:
        s = unicodedata.normalize("NFC", str(q))
        d = _DOC.search(s) or _NUM.search(s)
        hits = []
        if d and _core(d.group(1)):
            key = _core(d.group(1)); a = _ART.search(s)
            if a: hits += docart_idx.get((key, norm_key(a.group(1))), [])
            seen = set(hits)
            hits += [p for p in doc_idx.get(key, []) if p not in seen]
        out.append(hits[:k])
    return out


if not doc_idx:
    print("\n  ⚠ THE LEXREF INDEX IS EMPTY even after a rebuild — _core() returned")
    print("    None for every row. Distinct sample document_number values:")
    for _v in list(dict.fromkeys(chunks["document_number"].astype(str)))[:8]:
        print(f"      {_v[:78]!r}  → _core = {_core(_v)!r}")
    print("    If those look like document identifiers, _core needs another form.")
assert doc_idx, "lexref index empty after rebuild — see the sample values above."

# Collision check. The key is (number, year), so 12/2019/TT-BTP and
# 12/2019/NĐ-CP collapse together. That is deliberate — the issuing-body suffix
# is spelled inconsistently across this corpus — but it is worth SEEING, because
# a key that names 40 documents is a weak retrieval signal, not an exact one.
_sizes = np.array([len(v) for v in doc_idx.values()])
print(f"  chunks per document key: median {np.median(_sizes):.0f} "
      f"p90 {np.percentile(_sizes,90):.0f} max {_sizes.max()}")
print(f"  (doc, điều) pairs: {len(docart_idx):,} — this is the precise arm; the")
print(f"  document-only tier is a fallback when the query names no Điều.")

# Functional smoke test: a query naming a statute must return chunks FROM it.
_probe_key = max(docart_idx, key=lambda k: len(docart_idx[k]))
_probe_q = f"Quy định tại Điều {_probe_key[1]} của văn bản số {_probe_key[0]}"
_hits = lexref_search([_probe_q], k=20)[0]
print(f"  smoke: {_probe_q[:58]!r}")
print(f"         → {len(_hits)} hits" + (f", first is {chunk_keys[_hits[0]]!r}" if _hits else ""))
assert _hits, ("lexref_search returned nothing for a query built from its own index — "
               "parse_ref and _core disagree on the key form.")

def rrf_fuse(rank_lists, k_out, k_rrf=None):
    """score(d) = Σ 1/(60 + rank)."""
    k_rrf = k_rrf or cfg.rrf_k
    sc = defaultdict(float)
    for ranks in rank_lists:
        for r, pos in enumerate(ranks): sc[int(pos)] += 1.0 / (k_rrf + r + 1)
    return [p for p, _ in sorted(sc.items(), key=lambda x: -x[1])[:k_out]]

def hybrid_search(queries, k=None):
    k = k or cfg.cand_pool
    b, _ = bm25_search(queries, k=cfg.cand_pool)
    d, _ = dense_search(queries, k=cfg.cand_pool)
    lx = lexref_search(queries, cfg.cand_pool) if USE_LEXREF else [[]]*len(queries)
    return [rrf_fuse([b[i], d[i], lx[i]], k_out=k) for i in range(len(queries))]

def diversify(positions, max_parts=None):
    """K slots should buy K distinct provisions, not one Điều repeated."""
    max_parts = max_parts or cfg.max_parts_per_article
    seen, out = defaultdict(int), []
    for p in positions:
        key = chunk_keys[int(p)]
        if seen[key] < max_parts: seen[key] += 1; out.append(int(p))
    return out

def reorder_lost_in_middle(positions):
    """Liu et al. 2023 — decoders attend to the START and END of long contexts.
    Rank 1 first, rank 2 LAST, rank 3 second… Free; never hurts."""
    head, tail = [], []
    for i, p in enumerate(positions): (head if i % 2 == 0 else tail).append(p)
    return head + tail[::-1]

reranker = NativeCrossEncoder(cfg.reranker_id, device="cuda", dtype=DTYPE, max_length=512)

# ── UPGRADE 3: front-load the reranker input ──────────────────────────
# bge-reranker-v2-m3 truncates the PAIR at 512 tokens. Passing
# index_texts[p][:2500] chars (~700-1000 tokens) means everything past roughly
# the first 1,800 characters is discarded BEFORE scoring — so when the
# answering clause is Khoản 4 or 5, the reranker never sees it and scores the
# passage on its opening boilerplate. This is a direct hit on hit@1, which is
# the measured bottleneck (recall 60.6%, need 68.3%).
#
# Fix: spend the 512-token budget deliberately — header first (it carries the
# Điều number and document name the query often states), then the clause window
# with the most query-term overlap, rather than whichever clause happens to
# come first.
from collections import Counter
RERANK_CHARS     = 1400      # ~450 tokens, leaving room for the query in the pair
RERANK_FRONTLOAD = True
RERANK_VARIANT   = "v1"      # "v1" = v11's front-loading (what every scored run used)
                             # "v2" = scaffold-free · IDF-weighted · intent-aware (CELL 6e A/Bs it)

_CLAUSE_SPLIT = re.compile(r"(?=(?:^|\n)\s*\d{1,2}\.\s)")

def _rerank_text_v1(pos, query):
    """Header + the most query-relevant window, capped so nothing silently
    falls off the 512-token cliff."""
    if not RERANK_FRONTLOAD:
        return index_texts[int(pos)][:2500]
    head = chunk_header(int(pos))
    body = str(chunks.iloc[int(pos)]["content"])
    if len(body) <= RERANK_CHARS - len(head):
        return f"{head}. {body}"
    terms = [w for w in unicodedata.normalize("NFC", str(query).lower()).split()
             if len(w) > 2]
    parts = [x for x in _CLAUSE_SPLIT.split(body) if x.strip()] or [body]
    scored = sorted(((sum(1 for t in terms if t in x.lower()) / max(1, len(x)) ** 0.25, j)
                     for j, x in enumerate(parts)), reverse=True)
    keep, used = set(), len(head) + 2
    for _, j in scored:                      # greedy by density, then restore order
        if used + len(parts[j]) > RERANK_CHARS and keep: continue
        keep.add(j); used += len(parts[j])
        if used >= RERANK_CHARS: break
    return f"{head}. " + " ".join(parts[j] for j in sorted(keep))


# ── v2: spend the 512-token window on law, not on markup ──────────────
# Three things v1 leaves on the table, each of which pushes the answering clause
# out of the window or out-scores it:
#   · SCAFFOLD. Short chunks were passed whole, '[DOCUMENT] <slug>' line included:
#     ~25 tokens of unaccented filename that match nothing in a Vietnamese query.
#   · FLAT TERM WEIGHTS. v1 counts substring hits, so 'của', 'được' and 'theo'
#     weigh as much as 'thai sản' or '100/2019'. v2 weights each query word by
#     its IDF over the corpus and adds bigrams, so legal phrases count as phrases.
#   · INTENT. 'Mức phạt bao nhiêu?' is answered by the clause holding the AMOUNT;
#     'thời hạn bao lâu' by the one holding the DURATION; 'hồ sơ gồm' by the list.
#     A clause carrying what the question asks for gets a bonus.
_RR_TAG_DOC = re.compile(r"\[DOCUMENT\]\s*\S*\s*")             # the tag and its slug, wherever it sits
_RR_TAG     = re.compile(r"\[(?:ARTICLE|CLAUSE)\]\s*")
_RR_WORD    = re.compile(r"[\w/]+", re.U)
_RR_INTENT  = [  # (question cue, clause evidence, bonus)
    (re.compile(r"bao nhiêu tiền|mức phạt|phạt bao nhiêu|lệ phí|mức phí|bao nhiêu đồng|mức lương|mức hưởng"),
     re.compile(r"\d{1,3}(?:\.\d{3})+|\bđồng\b|%"), 1.5),
    (re.compile(r"bao lâu|thời hạn|thời gian|mấy ngày|bao nhiêu ngày|khi nào"),
     re.compile(r"\b\d+\s*(?:ngày|tháng|năm|giờ)\b|thời hạn"), 1.0),
    (re.compile(r"hồ sơ|giấy tờ|thủ tục|cần những gì|gồm những"),
     re.compile(r"hồ sơ|bao gồm|\bgồm\b|bản sao|đơn đề nghị"), 1.0),
    (re.compile(r"điều kiện|trường hợp nào|được phép|có được"),
     re.compile(r"điều kiện|trường hợp|phải đáp ứng"), 0.7),
]
_RR_IDF = None

def _rr_tokens(s):
    return _RR_WORD.findall(unicodedata.normalize("NFC", str(s)).lower())

def _rr_idf():
    """IDF over a fixed 50k-chunk sample: stable across runs, ~10 s once."""
    global _RR_IDF
    if _RR_IDF is None:
        smp = chunks["content"].sample(n=min(50_000, len(chunks)), random_state=0)
        df = Counter()
        for t in smp: df.update(set(_rr_tokens(t)))
        n = len(smp)
        _RR_IDF = ({w: math.log((n + 1) / (c + 1)) + 1.0 for w, c in df.items()},
                   math.log(n + 1) + 1.0)
    return _RR_IDF

def _rr_clean(body):
    body = _RR_TAG_DOC.sub("", str(body))
    return _RR_TAG.sub("", body).strip()

_RR_SENT = re.compile(r"(?<=[.;:])\s+|\n+")

def _rr_units(body, cap):
    """Clauses, and any clause longer than `cap` cut into sentences (then words),
    so the budget can be filled with the relevant PART of a long clause instead
    of its opening — a 2,500-char chunk is often one unsplittable clause."""
    out = []
    for x in [x for x in _CLAUSE_SPLIT.split(body) if x.strip()] or [body]:
        if len(x) <= cap:
            out.append(x); continue
        for sent in [t for t in _RR_SENT.split(x) if t.strip()]:
            while len(sent) > cap:                      # a sentence longer than the cap
                cut = sent.rfind(" ", 0, cap)
                cut = cut if cut > cap // 2 else cap
                out.append(sent[:cut]); sent = sent[cut:].lstrip()
            if sent.strip(): out.append(sent)
    return out

def _rerank_text_v2(pos, query):
    head = chunk_header(int(pos))
    body = _rr_clean(chunks.iloc[int(pos)]["content"])
    if len(body) <= RERANK_CHARS - len(head):
        return f"{head}. {body}"
    idf, idf_max = _rr_idf()
    qt = [w for w in _rr_tokens(query) if len(w) > 1]
    qset = set(qt)
    qbig = {(a, b) for a, b in zip(qt, qt[1:])}
    ql = unicodedata.normalize("NFC", str(query).lower())
    cues = [(ev, bonus) for cue, ev, bonus in _RR_INTENT if cue.search(ql)]
    parts = _rr_units(body, max(200, (RERANK_CHARS - len(head)) // 2))
    scored = []
    for j, x in enumerate(parts):
        xt = _rr_tokens(x)
        xs = set(xt)
        s  = sum(idf.get(w, idf_max) for w in qset & xs)
        s += 0.5 * sum(idf.get(a, idf_max) + idf.get(b, idf_max)
                       for a, b in qbig & set(zip(xt, xt[1:])))
        xl = x.lower()
        s += sum(bonus * idf_max for ev, bonus in cues if ev.search(xl))
        scored.append((s / max(1, len(x)) ** 0.25, j))
    keep, used = set(), len(head) + 2
    for _, j in sorted(scored, reverse=True):
        if used + len(parts[j]) + 1 > RERANK_CHARS and keep: continue
        keep.add(j); used += len(parts[j]) + 1          # +1: the joining space
        if used >= RERANK_CHARS: break
    return f"{head}. " + " ".join(parts[j] for j in sorted(keep))


def rerank_text(pos, query):
    """What the cross-encoder reads for one candidate. CELL 7's fingerprint
    records RERANK_VARIANT, so switching it rebuilds the contexts."""
    return (_rerank_text_v2 if RERANK_VARIANT == "v2" else _rerank_text_v1)(pos, query)


def retrieve_k(queries, k=None):
    k = k or cfg.top_k_context
    out = []
    for q, cand in zip(tqdm(queries, desc="rerank", leave=False),
                       hybrid_search(queries, k=cfg.cand_pool)):
        sc = reranker.predict([(q, rerank_text(p, q)) for p in cand],
                              batch_size=PROFILE["rerank_batch"])
        ranked = [p for p, _ in sorted(zip(cand, sc), key=lambda x: -x[1])]
        out.append(reorder_lost_in_middle(diversify(ranked)[:k]))
    return out

_rr = reranker.predict([("giấy chứng nhận kiểm dịch động vật", index_texts[p][:2500])
                        for p in hybrid_search(["giấy chứng nhận kiểm dịch động vật"], k=20)[0]])
# np.ptp(a), NOT a.ptp(): NumPy 2.0 removed ptp from the ndarray class
_spread = float(np.ptp(_rr))
print(f"reranker spread {_spread:.2f} (min {_rr.min():.2f} max {_rr.max():.2f})")
assert _spread > 0.5, "reranker returned near-constant scores — check it loaded"

def hit(pos, row):
    if pos in row["gold_positions"]: return True
    dn, an = chunk_keys[int(pos)]
    return (dn, an) in row["gold_keys"] or (dn, "") in row["gold_keys"]

def eval_retrieval(fn, name, ks=(1,3,5,8), n=150):
    ids = [q for q in sorted(RET_VAL) if q in labels_by_qid.index][:n]   # sorted: same subset every session
    if not ids: print(f"{name}: no labelled val queries"); return {}
    val = labels_by_qid.loc[ids]
    ranked = fn(val["query"].tolist(), max(ks))
    hits, mrr = {k: 0 for k in ks}, 0.0
    for pos, (_, row) in zip(ranked, val.iterrows()):
        f = next((r for r, p in enumerate(pos) if hit(p, row)), None)
        if f is not None:
            mrr += 1.0/(f+1)
            for k in ks:
                if f < k: hits[k] += 1
    res = {f"hit@{k}": hits[k]/len(val) for k in ks}; res["MRR"] = mrr/len(val)
    print(f"{name:>26}: " + "  ".join(f"{m}={v:.3f}" for m, v in res.items()))
    return res

RETRIEVAL_METRICS = eval_retrieval(lambda q,k: [list(r) for r in hybrid_search(q,k)],
                                   "hybrid RRF")
_mr = eval_retrieval(retrieve_k, f"+ rerank K={cfg.top_k_context}")
if RETRIEVAL_METRICS and _mr:
    _gap = _mr.get("hit@8",0) - _mr.get("hit@1",0)
    print(f"\n  hit@8 − hit@1 = {_gap:.3f} — the right article is already in the")
    print(f"  candidate list for those, just misordered. Free score if the reranker")
    print(f"  improves; no new retrieval needed.")


# ── A/B the reranker change; it is the only upgrade aimed at recall ───
print("\n" + "="*68)
print("UPGRADE 3 A/B — reranker input truncation")
print("="*68)
_lens = [len(index_texts[p][:2500]) for p in range(0, len(chunks), max(1,len(chunks)//500))]
print(f"  old input: {np.mean(_lens):.0f} chars avg ≈ {np.mean(_lens)/3.5:.0f} tokens "
      f"(cap is 512 — the tail was being discarded)")
RERANK_FRONTLOAD = False
_off = eval_retrieval(retrieve_k, "rerank: raw [:2500]")
RERANK_FRONTLOAD = True
_on  = eval_retrieval(retrieve_k, "rerank: front-loaded")
if _off and _on:
    print()
    for _k in ("hit@1","hit@3","hit@8","MRR"):
        _d = _on.get(_k,0)-_off.get(_k,0)
        print(f"  {_k:<6} {_off.get(_k,0):.3f} → {_on.get(_k,0):.3f}  {_d:+.3f}"
              + ("  ← improvement" if _d > 0.002 else
                 "  ← REGRESSION, set RERANK_FRONTLOAD=False" if _d < -0.002 else ""))
    _d1 = _on.get("hit@1",0)-_off.get("hit@1",0)
    print(f"\n  hit@1 {_d1:+.3f} → roughly {_d1*0.30:+.3f} val METEOR "
          f"(hit/miss spread ≈0.30)")
    print(f"  CELL 7 keys its contexts by the retrieval configuration, so whichever")
    print(f"  setting you keep reaches generation on its own — nothing to delete.")
RETRIEVE_ROUTE = "hybrid+rerank (cell 6)"


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6b — RETRIEVAL LAB: acronym expansion + weighted RRF (A/B measured)
#  Runs AFTER cell 6, BEFORE cell 7 (which frees the retrieval stack).
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['hybrid_search', 'eval_retrieval', 'lexref_search'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 6 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# Both upgrades are retrieval-side, so eval_retrieval settles them in minutes —
# no generation needed. Neither is adopted until Δhit@1 says so.
USE_ACRONYM  = True      # expand Vietnamese legal shorthand before retrieval
USE_WEIGHTED = True      # bias RRF toward BM25 when the query names a statute

# ── A. acronym expansion ──────────────────────────────────────────────
# APPEND rather than replace: the corpus sometimes writes the acronym and
# sometimes the full phrase, and keeping both gives BM25 a shot at either.
ACRONYMS = {
    "vphc":"vi phạm hành chính", "xphc":"xử phạt vi phạm hành chính",
    "đkkd":"đăng ký kinh doanh", "đkdn":"đăng ký doanh nghiệp",
    "gcn":"giấy chứng nhận", "gcnqsdđ":"giấy chứng nhận quyền sử dụng đất",
    "nlđ":"người lao động", "nsdlđ":"người sử dụng lao động",
    "hđlđ":"hợp đồng lao động", "hđtv":"hội đồng thành viên",
    "hđqt":"hội đồng quản trị", "csgt":"cảnh sát giao thông",
    "gpkd":"giấy phép kinh doanh", "gplx":"giấy phép lái xe",
    "bhxh":"bảo hiểm xã hội", "bhyt":"bảo hiểm y tế", "bhtn":"bảo hiểm thất nghiệp",
    "ubnd":"ủy ban nhân dân", "hđnd":"hội đồng nhân dân",
    "tnhh":"trách nhiệm hữu hạn", "dnnn":"doanh nghiệp nhà nước",
    "atgt":"an toàn giao thông", "pccc":"phòng cháy chữa cháy",
    "vsattp":"vệ sinh an toàn thực phẩm", "tttt":"thông tin truyền thông",
    "qsdđ":"quyền sử dụng đất", "sxkd":"sản xuất kinh doanh",
}
_ACR_RE = re.compile(r"\b(" + "|".join(sorted(ACRONYMS, key=len, reverse=True)) + r")\b", re.I)

def expand_acronyms(q):
    """Append the expansion after each acronym. Case-insensitive match on the
    NFC-normalised form; the original token is preserved."""
    s = unicodedata.normalize("NFC", str(q))
    seen = set()
    def rep(m):
        k = m.group(1).lower()
        if k in seen: return m.group(0)
        seen.add(k)
        return f"{m.group(0)} {ACRONYMS[k]}"
    return _ACR_RE.sub(rep, s)

def prep_retrieval_query(q):
    return expand_acronyms(q) if USE_ACRONYM else str(q)

# coverage first: an upgrade that touches 2% of queries has a 2% ceiling
_vq = [labels_by_qid.at[q,"query"] for q in sorted(RET_VAL)[:400] if q in labels_by_qid.index]
_tq = [(test_data[q]["question"] if isinstance(test_data[q],dict) else str(test_data[q]))
       for q in list(test_data)]
for _nm, _qs in (("labelled val",_vq), ("test set",_tq)):
    _hit = [q for q in _qs if _ACR_RE.search(unicodedata.normalize("NFC",str(q)))]
    print(f"{_nm}: {len(_hit)}/{len(_qs)} ({len(_hit)/max(1,len(_qs)):.1%}) contain a known acronym")
    for _q in _hit[:3]:
        print(f"    {str(_q)[:56]}\n      → {expand_acronyms(_q)[:76]}")

# ── B. weighted RRF ───────────────────────────────────────────────────
# A query naming "Nghị định 100/2019" or "Điều 26" is a LEXICAL query: those
# tokens are low-IDF for BM25 but essentially noise for a dense encoder, which
# maps digit strings to near-arbitrary directions. Bias the fusion accordingly.
_LEXQ = re.compile(r"(?:\bđiều\s*\d|\bkhoản\s*\d|\bđiểm\s*[a-zđ]\b|\d{1,4}\s*/\s*\d{2,4}"
                   r"|\bnghị\s*định\b|\bthông\s*tư\b|\bluật\b)", re.I)
W_LEX   = {"bm25": 1.2, "dense": 0.8}
W_PLAIN = {"bm25": 1.0, "dense": 1.0}

def rrf_weights(q):
    return W_LEX if _LEXQ.search(unicodedata.normalize("NFC", str(q))) else W_PLAIN

def rrf_fuse_w(rank_lists, weights, k_out, k_rrf=None):
    k_rrf = k_rrf or cfg.rrf_k
    sc = defaultdict(float)
    for ranks, w in zip(rank_lists, weights):
        for r, pos in enumerate(ranks): sc[int(pos)] += w / (k_rrf + r + 1)
    return [p for p, _ in sorted(sc.items(), key=lambda x: -x[1])[:k_out]]

def hybrid_search_v2(queries, k=None):
    k = k or cfg.cand_pool
    qq = [prep_retrieval_query(q) for q in queries]
    b, _ = bm25_search(qq, k=cfg.cand_pool)
    d, _ = dense_search(qq, k=cfg.cand_pool)
    lx = lexref_search(qq, cfg.cand_pool) if USE_LEXREF else [[]]*len(queries)
    out = []
    for i, q in enumerate(queries):
        w = rrf_weights(q) if USE_WEIGHTED else W_PLAIN
        out.append(rrf_fuse_w([b[i], d[i], lx[i]],
                              [w["bm25"], w["dense"], 1.0], k_out=k))
    return out

_nlex = sum(1 for q in _tq if _LEXQ.search(unicodedata.normalize("NFC", str(q))))
print(f"\ntest set: {_nlex}/{len(_tq)} ({_nlex/max(1,len(_tq)):.1%}) look lexical "
      f"→ would get bm25 1.2 / dense 0.8")

# ── the A/B ───────────────────────────────────────────────────────────
print("\n" + "="*68)
print("RETRIEVAL A/B — same val questions, same reranker, retrieval only")
print("="*68)
_base = eval_retrieval(lambda q,k: [list(r) for r in hybrid_search(q,k)], "baseline (cell 6)")
_res = {}
_user_flags = (USE_ACRONYM, USE_WEIGHTED)   # the sweep below toggles both; YOUR setting is restored
for _a, _w, _lbl in ((False,False,"+ nothing (sanity)"), (True,False,"+ acronyms"),
                     (False,True,"+ weighted RRF"),      (True,True,"+ both")):
    USE_ACRONYM, USE_WEIGHTED = _a, _w
    _res[_lbl] = eval_retrieval(lambda q,k: [list(r) for r in hybrid_search_v2(q,k)], _lbl)
USE_ACRONYM, USE_WEIGHTED = _user_flags      # v10 forced both True here, ignoring lines 14-15

if _base:
    print(f"\n  {'variant':<24} {'Δhit@1':>8} {'Δhit@8':>8} {'ΔMRR':>8}")
    print("  " + "-"*52)
    for _lbl, _m in _res.items():
        if not _m: continue
        print(f"  {_lbl:<24} {_m.get('hit@1',0)-_base.get('hit@1',0):>+8.3f} "
              f"{_m.get('hit@8',0)-_base.get('hit@8',0):>+8.3f} "
              f"{_m.get('MRR',0)-_base.get('MRR',0):>+8.3f}")
    _best = max(_res.items(), key=lambda kv: (kv[1] or {}).get("hit@1", -1))
    print(f"\n  best on hit@1: {_best[0]}")
    print(f"  Set USE_ACRONYM / USE_WEIGHTED at the top of this cell to match. CELL 7")
    print(f"  keys contexts by the retrieval configuration, so the change reaches")
    print(f"  generation without deleting anything.")
    # Convert Δhit@1 into ΔMETEOR using the measured operating point.
    _d1 = _best[1].get("hit@1",0) - _base.get("hit@1",0)
    print(f"\n  rough value: your val METEOR ≈ p·(hit) + (1−p)·(miss). A Δhit@1 of "
          f"{_d1:+.3f}\n  moves it by roughly {_d1*0.30:+.3f} if the hit/miss spread is ~0.30.")


# ── wire the upgrades into the pipeline ───────────────────────────────
# cell 7 calls retrieve_k(), which calls hybrid_search(). Rebinding it here is
# what makes USE_ACRONYM / USE_WEIGHTED actually reach the contexts. With both
# False, hybrid_search_v2 is behaviourally identical to hybrid_search (weights
# 1.0/1.0, no expansion), so this override is safe either way.
# NOTE: re-running cell 6 reverts retrieve_k to the 2-arm version. If you do,
# re-run this cell too.
def retrieve_k(queries, k=None):
    k = k or cfg.top_k_context
    out = []
    for q, cand in zip(tqdm(queries, desc="rerank", leave=False),
                       hybrid_search_v2(queries, k=cfg.cand_pool)):
        # rerank_text(), NOT index_texts[p][:2500]. This override previously
        # reverted to the raw 2,500-char input — silently throwing away the
        # front-loading this very cell had just measured at +0.004 hit@1. A lab
        # whose winner the production path discards is worse than no lab.
        sc = reranker.predict([(q, rerank_text(p, q)) for p in cand],
                              batch_size=PROFILE["rerank_batch"])
        ranked = [p for p, _ in sorted(zip(cand, sc), key=lambda x: -x[1])]
        out.append(reorder_lost_in_middle(diversify(ranked)[:k]))
    return out

RETRIEVE_ROUTE = "hybrid_v2+rerank (cell 6b)"
print(f"\nretrieve_k now routes through hybrid_search_v2 "
      f"(acronyms={USE_ACRONYM}, weighted={USE_WEIGHTED}, lexref={USE_LEXREF})")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6c — RETRIEVAL AUTOPSY   (run after 6b, ~3 min, no generation)
# ══════════════════════════════════════════════════════════════════════
# THE NUMBER THAT CAPS EVERYTHING
#
# Your last run measured  hit@1 0.087 · hit@3 0.113 · hit@8 0.171.
# Eighty-three percent of questions do not have the right article ANYWHERE in
# the eight chunks the generator is shown. No citation strategy, no dedup, no
# prompt can recover an article that was never retrieved — cells 10c and 10d
# were optimising the packaging of a box that is usually empty.
#
# But 8.7% is also low enough to be suspicious. A BM25+dense hybrid over 800k
# Vietnamese legal chunks should not be at chance. So before spending a run on
# retrieval changes, this cell asks WHERE the number is lost, in order:
#
#   1. are the gold labels even resolving to positions?   (a label bug reads
#      exactly like a retrieval failure, and is far cheaper to fix)
#   2. is the right chunk in the 100-candidate pool at all? (recall)
#   3. if it is, does the reranker put it in the top 8?    (ordering)
#   4. which arm — BM25, dense, lexref — is carrying it?
_MISSING = [n for n in ['labels_by_qid','chunk_keys','hybrid_search','bm25_search',
                        'dense_search','lexref_search','reranker','rerank_text',
                        'RET_VAL','hit','chunks'] if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS 5 and 6 first.")

import numpy as np
from collections import Counter

_ids = [q for q in sorted(RET_VAL) if q in labels_by_qid.index][:200]   # sorted: same 200 every session
_rows = []
for q in _ids:
    r = labels_by_qid.loc[q]
    if isinstance(r, pd.DataFrame): r = r.iloc[0]
    _rows.append((q, r))
print(f"retrieval autopsy over {len(_rows)} labelled val questions\n")

# ══ 1. DO THE LABELS RESOLVE? ══════════════════════════════════════════
_npos = np.array([len(r["gold_positions"]) for _, r in _rows])
_nkey = np.array([len(r["gold_keys"])      for _, r in _rows])
print("1. LABEL RESOLUTION")
_rep = globals().get("LABEL_REPORT")
if _rep:
    print(f"   all labels (CELL 3): {_rep['resolved']:.1%} of {_rep['questions']:,} questions resolved; "
          f"by route: " + ", ".join(f"{k} {v:,}" for k, v in sorted(_rep["by_source"].items(),
                                                                    key=lambda kv: -kv[1])[:5]))
_art = np.array([any(a for _, a in r["gold_keys"]) for _, r in _rows])
print(f"   RET_VAL questions with ARTICLE-level gold: {_art.mean():.1%} — for these any chunk of the "
      f"cited Điều counts as a hit, which is what the generator needs")
print(f"   gold_positions per question: mean {_npos.mean():.0f}  "
      f"median {np.median(_npos):.0f}  zero for {int((_npos==0).sum())}/{len(_rows)}")
print(f"   gold_keys      per question: mean {_nkey.mean():.1f}  "
      f"zero for {int((_nkey==0).sum())}/{len(_rows)}")
_ex = next((r for _, r in _rows if r["gold_keys"]), None)
if _ex is not None:
    print(f"   sample gold_key : {list(_ex['gold_keys'])[0]!r}")
print(f"   sample chunk_key: {chunk_keys[0]!r}")
print(f"   → these two must have the SAME SHAPE. A gold_key of ('nghị định "
      f"100/2019','17')\n     against a chunk_key of ('nghi-dinh-100-2019-nd-cp-16923','17') "
      f"never\n     matches, and every hit() call returns False while retrieval is fine.")
if (_npos == 0).mean() > 0.5:
    print("   ⚠ MORE THAN HALF THE QUESTIONS HAVE NO RESOLVED GOLD POSITION.")
    print("     hit@k is then measuring label resolution, not retrieval. Fix cell 3's")
    print("     by_doc_art join before touching the retriever.")

# ══ 2. IS IT IN THE POOL AT ALL? ═══════════════════════════════════════
print("\n2. RECALL vs ORDERING")
_qs = [r["query"] for _, r in _rows]
_pool = hybrid_search(_qs, k=cfg.cand_pool)
_at = {}
for K in (1, 3, 8, 20, 50, 100):
    _at[K] = float(np.mean([any(hit(p, r) for p in pool[:K])
                            for pool, (_, r) in zip(_pool, _rows)]))
print(f"   fused pool, before reranking:")
for K in (1, 3, 8, 20, 50, 100):
    print(f"     hit@{K:<4} {_at[K]:.3f}  {'█'*int(_at[K]*50)}")
_ceiling = _at[100]
print(f"\n   → hit@100 = {_ceiling:.3f} is the CEILING the reranker works inside.")
if _ceiling < 0.35:
    print("     The right article is usually not retrieved at all. This is a RECALL")
    print("     problem: the reranker cannot fix it, and neither can a better prompt.")
else:
    print(f"     Recall is {_ceiling:.0%} but hit@8 after reranking was 0.171 — the")
    print(f"     article IS being found and then discarded. That is an ORDERING")
    print(f"     problem, and the reranker is where the headroom sits.")

# ══ 3. WHICH ARM IS CARRYING IT? ═══════════════════════════════════════
print("\n3. PER-ARM CONTRIBUTION (hit@100, each channel alone)")
_b, _ = bm25_search(_qs, k=cfg.cand_pool)
_d, _ = dense_search(_qs, k=cfg.cand_pool)
_l    = lexref_search(_qs, cfg.cand_pool)
for nm, arm in (("BM25", _b), ("dense", _d), ("lexref", _l)):
    cov = float(np.mean([len(a) > 0 for a in arm]))
    h   = float(np.mean([any(hit(int(p), r) for p in a) if len(a) else False
                         for a, (_, r) in zip(arm, _rows)]))
    print(f"   {nm:<8} fires on {cov:5.1%} of queries | hit@100 {h:.3f}")
print(f"   {'fused':<8} {'':16} hit@100 {_ceiling:.3f}")
print("   → an arm that fires on 0% is switched off or mis-keyed, not unhelpful. An empty arm adds")
print("     0 to every RRF score, so it cannot dilute the others; CELL 6e judges lexref where it fires.")

# ══ 4. WHAT WOULD A BIGGER POOL BUY? ═══════════════════════════════════
print("\n4. WOULD A DEEPER POOL HELP?")
_wide = hybrid_search(_qs, k=400)
for K in (100, 200, 400):
    h = float(np.mean([any(hit(p, r) for p in pool[:K])
                       for pool, (_, r) in zip(_wide, _rows)]))
    print(f"   hit@{K:<4} {h:.3f}" + ("   ← current cand_pool" if K == cfg.cand_pool else ""))
_gain = float(np.mean([any(hit(p, r) for p in pool[:400]) for pool, (_, r) in zip(_wide, _rows)])) - _ceiling
print(f"   → widening cand_pool 100→400 adds {_gain:+.3f} recall for ~4× reranker time.")
print(f"     Worth it only if the reranker can then order it — see section 2.")
RETRIEVAL_AUTOPSY = {"hit_at": _at, "ceiling": _ceiling,
                     "labels_unresolved": float((_npos == 0).mean())}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6d — RETRIEVER FINE-TUNING, ROUND 2  (your "Cell 2.4")   A100: ~25 min
#  Run AFTER 6 / 6b / 6c, BEFORE 6e / 7 (cell 7 frees the retrieval stack).
# ══════════════════════════════════════════════════════════════════════
# ROUND 2 = round 1's recipe, aimed at round 1's mistakes
#   · negatives are mined by the encoder CELL 5 loaded — ft_v1 once adopted — so
#     they are the errors ft_v1 still makes, not the ones it already fixed;
#   · training CONTINUES from ft_v1 (FT_INIT = "current") at half the LR;
#   · two negatives per question, each first vetted by the cross-encoder: a
#     candidate it scores ≥ the positive is dropped as a probable false negative;
#   · CELL 3's label fix feeds it more (and article-level) gold positives.
# The A/B then compares ft_v2 against what is loaded (ft_v1), and adoption needs
# the same paired McNemar evidence as round 1.
#
# WHY THIS IS THE CELL THAT CAN MOVE THE SCORE
#
# Cell 6 measured hit@1 0.087 · hit@8 0.171: for 83% of questions the right
# article never reaches the generator. Cells 10c/10d/10e then proved that
# post-processing and prompting are exhausted — every dedup threshold scored
# BELOW baseline, no prompt beat the current one. What remains is retrieval.
#
# WHAT FINE-TUNING CAN AND CANNOT DO — two common expectations are wrong:
#   ✗ It does NOT lift the 256-token window. PhoBERT has 258 learned position
#     embeddings (2 reserved by the RoBERTa padding offset). Asking for more
#     crashes; the window is architecture, not a setting.
#   ✗ It does NOT add vocabulary. The tokenizer is frozen; every legal term
#     is still spelled with the same sub-word pieces.
#   ✓ It DOES reshape the embedding space so that a legal QUESTION lands near
#     the CLAUSE that answers it — which is exactly what hit@1 measures, and
#     what a general-purpose Vietnamese encoder was never trained to do.
#   ✓ It recovers window budget indirectly: passages are encoded with the
#     chunker's [DOCUMENT]/[ARTICLE]/[CLAUSE] scaffolding stripped, so the 256
#     tokens are spent on law instead of on an unaccented filename slug.
#
# SAFETY
#   · Trains only on labels OUTSIDE RET_VAL and GEN_VAL, by id AND by question
#     text — otherwise the A/B below would score memorisation as retrieval.
#   · Hard negatives come from the current retriever's own mistakes, restricted
#     to documents the question does NOT cite, and never a near-copy of the
#     positive (amending decrees duplicate provisions across documents).
#   · The corpus is RE-ENCODED with the new model. Mixing a new query encoder
#     with old corpus vectors scores near chance and raises no error — so the
#     stored vectors are checked against the serving encoder (NativeEncoder) on
#     random passages, and cached vectors are stamped with the model that wrote
#     them.
#   · Adoption is decided by a paired McNemar test on held-out questions, and
#     rejected if the reranked top-1 gets significantly worse. On rejection the
#     original encoder objects are restored and nothing downstream changes.
#   · Parameter count is asserted identical to the base: the 4.0B budget holds.
#   · Resume-safe: the trained model, the corpus shards and the final matrix are
#     all cached on Drive; a disconnect costs one shard, not the run.
#   · It reaches the answers. CELL 7's context files are keyed by a fingerprint
#     of the retrieval stack (encoder included) and CELLS 10/11 key raw answers
#     by prompt hash — in v10 both were keyed by question id, which is why the
#     lexref fix never reached a single generated answer.
# ── fine-tuning core: pure functions, no notebook globals ─────────────
import os, re, math, time, json, hashlib, unicodedata
import numpy as np

_FT_DOC = re.compile(r"^\s*\[DOCUMENT\].*$", re.M)
_FT_CLA = re.compile(r"^\s*\[CLAUSE\]\s*\d*\.?\s*", re.M)
_FT_ART = re.compile(r"^\s*\[ARTICLE\]\s*", re.M)


def ft_strip_tags(s):
    """Chunker scaffolding out, legal text in. The [DOCUMENT] slug alone costs
    ~25 PhoBERT pieces of unaccented filename, and the window is 256 tokens."""
    s = _FT_DOC.sub("", str(s)); s = _FT_CLA.sub("", s); s = _FT_ART.sub("", s)
    return re.sub(r"\s+", " ", s).strip()


def ft_tokset(s):
    return set(unicodedata.normalize("NFC", str(s)).lower().split())


def ft_jaccard(a, b):
    return len(a & b) / max(1, len(a | b))


def ft_pick_positive(gold, answer, content, cap=200):
    """The gold chunk the reference answer actually draws from — most shared
    vocabulary with the answer. A document-level label resolves to EVERY chunk of
    that document, and an arbitrary one of them is a noisy positive."""
    ans, best, bs = ft_tokset(answer), None, -1
    for p in sorted(gold)[:cap]:
        s = len(ans & ft_tokset(content[p]))
        if s > bs:
            best, bs = int(p), s
    return best


def ft_mine_negative(ranked, gold_docs, pos_set, doc_of, content, max_jacc, rng, n_total):
    """Hardest candidate that is (a) from a document the question does NOT cite,
    and (b) not a near-copy of the positive.
    (a) Adjacent clauses of a cited document are often genuinely relevant;
        training the encoder to push them away teaches it the wrong thing.
    (b) Catches what (a) cannot: amending and consolidating decrees copy
        provisions verbatim, so the 'same' clause can live in another document."""
    for p in ranked:
        p = int(p)
        if doc_of[p] in gold_docs:
            continue
        if ft_jaccard(pos_set, ft_tokset(content[p])) > max_jacc:
            continue
        return p, "hard"
    for _ in range(100):                       # fallback: random, other document
        p = int(rng.integers(0, n_total))
        if doc_of[p] not in gold_docs:
            return p, "random"
    return None, "none"


def ft_hard_candidates(ranked, gold_docs, pos_set, doc_of, content, max_jacc, limit):
    """Round 2: up to `limit` candidates passing the SAME two rules as
    ft_mine_negative, in retrieval order — the pool the denoiser then filters."""
    out = []
    for p in ranked:
        p = int(p)
        if doc_of[p] in gold_docs:
            continue
        if ft_jaccard(pos_set, ft_tokset(content[p])) > max_jacc:
            continue
        out.append(p)
        if len(out) >= limit:
            break
    return out


def ft_random_negative(gold_docs, doc_of, rng, n_total, exclude=()):
    for _ in range(100):
        p = int(rng.integers(0, n_total))
        if doc_of[p] not in gold_docs and p not in exclude:
            return p
    return None


def ft_train(base_id, rows, dims, epochs, batch, lr, warmup, bf16, seed, max_len,
             device, tmp_dir):
    """SentenceTransformerTrainer + MatryoshkaLoss(MultipleNegativesRankingLoss).

    The model is ASSEMBLED from modules — Transformer → mean Pooling → Normalize —
    rather than loaded through the repo's modules.json. Two reasons:
      · sentence-transformers 3.x cannot read this checkpoint's modules.json;
      · the notebook encodes with NativeEncoder (masked MEAN pooling + L2), and
        the model must be trained with exactly the pooling it is served with.
    max_seq_length is set EXPLICITLY: PhoBERT has 258 position slots with a
    padding offset of 2, so 256 is the hard ceiling and 257+ raises
    'index out of range' inside the embedding layer."""
    from datasets import Dataset
    from sentence_transformers import (SentenceTransformer, SentenceTransformerTrainer,
                                       SentenceTransformerTrainingArguments, models, losses)
    from sentence_transformers.training_args import BatchSamplers
    wm = models.Transformer(base_id, max_seq_length=max_len)
    hid = wm.get_word_embedding_dimension()
    st = SentenceTransformer(modules=[wm, models.Pooling(hid, pooling_mode="mean"),
                                      models.Normalize()], device=device)
    dims = sorted({int(d) for d in dims if int(d) <= hid}, reverse=True) or [hid]
    # MNRL: every other positive AND negative in the batch is also a negative
    # for each anchor, so batch size is the lever — and duplicates inside a batch
    # would be false negatives, which NO_DUPLICATES rules out.
    loss = losses.MatryoshkaLoss(st, losses.MultipleNegativesRankingLoss(st),
                                 matryoshka_dims=dims)
    # (anchor, positive, negative_1, …, negative_n): MNRL treats every column
    # after the positive — and every other row's texts — as negatives
    cols = ["anchor", "positive"] + [f"negative_{i + 1}" for i in range(len(rows[0]) - 2)]
    ds = Dataset.from_dict({c: [r[j] for r in rows] for j, c in enumerate(cols)})
    args = SentenceTransformerTrainingArguments(
        output_dir=tmp_dir, num_train_epochs=epochs, per_device_train_batch_size=batch,
        learning_rate=lr, warmup_ratio=warmup, bf16=bf16, seed=seed,
        batch_sampler=BatchSamplers.NO_DUPLICATES, dataloader_drop_last=True,
        save_strategy="no", logging_steps=10, report_to="none")
    tr = SentenceTransformerTrainer(model=st, args=args, train_dataset=ds, loss=loss)
    tr.train()
    return st, dims, [h for h in tr.state.log_history if "loss" in h]


def ft_save_hf(st, out_dir):
    """Plain Hugging Face layout, so NativeEncoder / AutoModel load it unchanged."""
    os.makedirs(out_dir, exist_ok=True)
    st[0].auto_model.save_pretrained(out_dir)
    st[0].tokenizer.save_pretrained(out_dir)


def ft_embed_ids(model, id_lists, pad_id, device, dtype, batch=512):
    """Masked mean pooling + L2 — the SAME arithmetic as NativeEncoder.encode, run
    on token ids produced by the worker pool. Length-sorted to minimise padding."""
    import torch
    hid = model.config.hidden_size
    out = np.zeros((len(id_lists), hid), dtype=np.float16)
    order = np.argsort([-len(x) for x in id_lists], kind="stable")
    with torch.inference_mode():
        for s in range(0, len(order), batch):
            idx = order[s:s + batch]
            L = max(1, max(len(id_lists[i]) for i in idx))
            ids = torch.full((len(idx), L), int(pad_id), dtype=torch.long)
            att = torch.zeros((len(idx), L), dtype=torch.long)
            for r, i in enumerate(idx):
                x = id_lists[i]
                if len(x):
                    ids[r, :len(x)] = torch.as_tensor(x, dtype=torch.long)
                    att[r, :len(x)] = 1
            ids, att = ids.to(device), att.to(device)
            with torch.autocast(device_type=str(device).split(":")[0], dtype=dtype,
                                enabled=(str(device).startswith("cuda") and dtype != torch.float32)):
                h = model(input_ids=ids, attention_mask=att).last_hidden_state
            m = att.unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            v = torch.nn.functional.normalize(v.float(), p=2, dim=1)
            out[idx] = v.cpu().numpy().astype(np.float16)
    return out


FT_WORKER_SRC = '''
import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import numpy as np
_TOK = _SEG = _L = None
def init(tok_dir, max_len):
    global _TOK, _SEG, _L
    from transformers import AutoTokenizer
    from pyvi import ViTokenizer
    _TOK, _SEG, _L = AutoTokenizer.from_pretrained(tok_dir), ViTokenizer.tokenize, int(max_len)
def seg(texts):
    return [_SEG(t) if t else "" for t in texts]
def work(texts):
    ids = _TOK([_SEG(t) if t else "" for t in texts], truncation=True, max_length=_L)["input_ids"]
    return [np.asarray(x, dtype=np.int32) for x in ids]
'''


def ft_worker_module(dirpath):
    """pyvi is pure-Python CRF: 3.4 ms/passage measured, i.e. 45 min single-core
    for this corpus. The work goes to a SPAWN pool (not fork: the parent holds
    CUDA and OpenMP threads, and a forked child can inherit a held lock). Spawn
    needs an importable module, so the worker is written to disk."""
    import sys, importlib
    os.makedirs(dirpath, exist_ok=True)
    with open(os.path.join(dirpath, "_ft_worker.py"), "w", encoding="utf-8") as f:
        f.write(FT_WORKER_SRC)
    if dirpath not in sys.path:
        sys.path.insert(0, dirpath)
    import _ft_worker
    return importlib.reload(_ft_worker)


def ft_encode_corpus(texts, shard_dir, model, tok_dir, max_len, pad_id, device, dtype,
                     procs, wmod, shard=40000, part=500, batch=512, log=print):
    """Resume-safe: one .npy per shard, written atomically, skipped on re-run.
    A Colab disconnect costs one shard, not the whole corpus."""
    import multiprocessing as mp
    os.makedirs(shard_dir, exist_ok=True)
    n = len(texts); ns = max(1, math.ceil(n / shard))
    path = lambda s: os.path.join(shard_dir, f"{s:04d}.npy")
    todo = [s for s in range(ns) if not os.path.exists(path(s))]
    if todo:
        log(f"  encoding {len(todo)}/{ns} shards ({n:,} unique passages, {procs} pyvi workers)")
        t0 = time.time()
        with mp.get_context("spawn").Pool(procs, initializer=wmod.init,
                                          initargs=(tok_dir, max_len)) as pool:
            for k, s in enumerate(todo):
                lo, hi = s * shard, min(n, (s + 1) * shard)
                parts = [texts[i:min(hi, i + part)] for i in range(lo, hi, part)]
                ids = [x for chunk in pool.imap(wmod.work, parts) for x in chunk]
                emb = ft_embed_ids(model, ids, pad_id, device, dtype, batch)
                tmp = path(s) + ".tmp"
                with open(tmp, "wb") as f:
                    np.save(f, emb)
                os.replace(tmp, path(s))
                el = time.time() - t0
                log(f"    shard {s+1}/{ns}  {hi:,}/{n:,}  "
                    f"{el/60:5.1f} min elapsed, ~{el/(k+1)*(len(todo)-k-1)/60:5.1f} min left")
    return np.concatenate([np.load(path(s)) for s in range(ns)], axis=0)
# ══════════════════════════════════════════════════════════════════════
#  DRIVER — uses the notebook's globals
# ══════════════════════════════════════════════════════════════════════
RUN_RETRIEVER_FT   = True
FT_TAG             = "ft_v2"           # ROUND 2 (ft_v1 = round 1). Bump to retrain.
FT_INIT            = "current"         # "current": continue from the adopted encoder
                                       # (ft_v1) · "base": start again from the base
FT_EPOCHS          = 2
FT_BATCH           = 64                # MNRL: bigger batch = more in-batch negatives
FT_LR              = 2e-5              # from the base encoder
FT_LR_CONTINUE     = 1e-5              # from an already fine-tuned one: a smaller step
FT_WARMUP          = 0.10
FT_MAX_LEN         = 256               # PhoBERT's hard ceiling — see ft_train
MATRYOSHKA_DIMS    = [768, 512, 256]
FT_NEG_POOL        = 100               # hybrid candidates searched per question
FT_NEG_MAX_JACCARD = 0.60              # reject 'negatives' that copy the positive
FT_N_NEG           = 2                 # hard negatives per question (MNRL takes any number)
FT_DENOISE         = True              # cross-encoder veto on probable false negatives
FT_DENOISE_CANDS   = 12                # candidates scored per question
FT_DENOISE_MARGIN  = 0.0               # drop a candidate scoring ≥ positive − margin
FT_MIN_GAIN_HIT8   = 0.005             # adopt only on a measured hybrid hit@8 gain
FT_MAX_HIT1_LOSS   = 0.010             # …that does not cost the reranked top-1
FT_ADOPT_P         = 0.10              # paired McNemar, one-sided — see section 5
FT_RETEST          = False             # True → re-run the A/B for a tag rejected earlier
                                       # (e.g. after changing retrieval upstream)
FT_PROCS           = max(1, (os.cpu_count() or 2) - 1)
FT_LOCAL           = "/content" if os.path.isdir("/content") else os.path.expanduser("~")
FT_WORKER_DIR      = os.path.join(FT_LOCAL, ".ft_worker")   # its own path: nothing else cleans it
FT_DEVICE          = "cuda" if torch.cuda.is_available() else "cpu"
FT_DIR   = W(f"encoder_{FT_TAG}")
FT_EMB   = W(f"embeddings_{FT_TAG}.npy")
FT_META  = os.path.join(FT_DIR, "ft_meta.json")
FT_ADOPT = W("ENCODER_ADOPTED.json")
FT_REJECT = W(f"ENCODER_REJECTED_{FT_TAG}.json")   # a measured "no" is remembered too

_MISSING = [n for n in ['labels','labels_by_qid','RET_VAL','GEN_VAL','qa_by_id','chunks','chunk_keys','has_val',
                        'prep_query','NativeEncoder','query_encoder','faiss_index','dense_search',
                        'hybrid_search','retrieve_k','eval_retrieval','validate_embeddings',
                        'count_params','cfg','W','DTYPE','AutoConfig','hit','ViTokenizer']
                        + (['reranker', 'rerank_text', 'PROFILE'] if FT_DENOISE else [])
            if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\nRun CELLS 0-6 (and 6b) first. This cell must "
        f"run BEFORE CELL 7, which frees the retrieval stack.")

import sys, gc, json, random, functools, subprocess, importlib, importlib.metadata as _md
from collections import Counter
ENCODER_TAG = globals().get("ENCODER_TAG", "base")

def _ft_status():
    if not RUN_RETRIEVER_FT:
        return "skip", "RUN_RETRIEVER_FT = False"
    if not globals().get("USE_FT_ENCODER", True):
        return "skip", "USE_FT_ENCODER = False in CELL 5 (base encoder forced)"
    if os.path.exists(FT_ADOPT):
        _a = json.load(open(FT_ADOPT))
        if _a.get("tag") == FT_TAG and ENCODER_TAG == FT_TAG:
            return "skip", f"{FT_TAG} is already adopted and loaded by CELL 5"
    if os.path.exists(FT_REJECT) and not FT_RETEST:
        _r = json.load(open(FT_REJECT))
        return "skip", (f"{FT_TAG} was measured on {_r.get('time', '?')} and NOT adopted "
                        f"(hybrid hit@8 {_r.get('g8', 0):+.3f}, p = {_r.get('p8', 1):.3f}). "
                        f"Set FT_RETEST = True to measure again, or bump FT_TAG to retrain")
    return "run", ""

_state, _why = _ft_status()
if _state == "skip":
    print(f"retriever fine-tuning skipped — {_why}")
else:
    t_all = time.time()

    # ── 0. sentence-transformers WITHOUT moving the pinned transformers ──
    # This notebook pins transformers 4.49 because sentence-transformers 5.x
    # cannot run on it. A plain `pip install sentence-transformers` can pull a
    # newer transformers onto disk while 4.49 stays loaded in memory, and the
    # next import mixes the two. Pin, then CHECK the disk.
    _tv = transformers.__version__
    try:
        import sentence_transformers as _stm
        if not _stm.__version__.startswith("3."):
            raise ImportError(_stm.__version__)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "sentence-transformers==3.4.1", f"transformers=={_tv}"], check=True)
        importlib.invalidate_caches()
        import sentence_transformers as _stm
    import importlib.util as _iu
    for _dep in ("datasets", "accelerate"):          # Trainer needs both; Colab ships them
        if _iu.find_spec(_dep) is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", _dep,
                            f"transformers=={_tv}"], check=True)
    _disk = _md.version("transformers")
    assert _disk == _tv, (
        f"pip moved transformers on disk to {_disk} while {_tv} is loaded. "
        f"Run: pip install transformers=={_tv}, then Runtime → Restart session.")
    print(f"sentence-transformers {_stm.__version__} | transformers {_tv} (unchanged)")

    # ── 1. passages, exactly as they will be encoded ──────────────────
    _content = chunks["content"].astype(str).to_numpy()
    _dtitle  = chunks["document_title"].astype(str).to_numpy()
    _dname   = chunks["name"].astype(str).to_numpy()
    _doc_of  = [k[0] for k in chunk_keys]

    @functools.lru_cache(maxsize=None)
    def _title(dt, nm):
        if has_val(dt): return str(dt)
        t = re.sub(r"[-_]+", " ", str(nm or "")).strip()
        return re.sub(r"\s*\d{4,}\s*$", "", t)

    def ft_passage_raw(i):
        """Document title (capped) + scaffold-stripped chunk text. The SAME
        function builds the training positives, the negatives and all 801,863
        corpus passages — train and serve cannot drift apart."""
        t = _title(_dtitle[i], _dname[i])[:160]
        b = ft_strip_tags(_content[i])
        return f"{t} . {b}" if t else b

    # The token ceiling comes from the checkpoint, not from a constant. RoBERTa-
    # family models (PhoBERT is one) start positions at padding_idx + 1, so
    # 258 slots hold 256 tokens; ask for 257 and the embedding lookup raises
    # "index out of range". Deriving it keeps this cell safe if the encoder is
    # ever swapped for one with a different window.
    _ecfg = AutoConfig.from_pretrained(cfg.emb_model_id)
    _off = ((_ecfg.pad_token_id or 0) + 1
            if _ecfg.model_type in ("roberta", "xlm-roberta", "camembert") else 0)
    _cap = int(_ecfg.max_position_embeddings) - _off
    if FT_MAX_LEN > _cap:
        print(f"FT_MAX_LEN {FT_MAX_LEN} exceeds this checkpoint's window — clamped to {_cap}")
        FT_MAX_LEN = _cap
    print(f"encoder window: {FT_MAX_LEN} tokens ({_ecfg.model_type}, "
          f"{_ecfg.max_position_embeddings} position slots)")

    wmod = ft_worker_module(FT_WORKER_DIR)

    # ── 2 + 3. triplets and training — only when this tag has no model yet ──
    # (mining re-queries the retriever for every labelled question; on a re-run
    #  with the model already on Drive none of it is needed)
    if os.path.exists(FT_META):
        print(f"\n{FT_DIR} already trained — reusing it (bump FT_TAG to retrain)")
    else:
        # ── 2. triplets from held-in labels only ──────────────────────────
        # RET_VAL is the set the A/B below is scored on, GEN_VAL feeds the METEOR
        # gate. Training on either — by id OR by an identical question under another
        # id — would make the A/B report memorisation as retrieval quality.
        _nq = lambda s: re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).lower().strip())
        _hold_ids = {str(q) for q in RET_VAL} | {str(q) for q in GEN_VAL}
        _hold_txt = set()
        for q in _hold_ids:
            if q in labels_by_qid.index:
                _r = labels_by_qid.loc[q]
                _r = _r.iloc[0] if isinstance(_r, pd.DataFrame) else _r
                _hold_txt.add(_nq(_r.get("query", "")))
            if q in qa_by_id.index:
                _hold_txt.add(_nq(qa_by_id.at[q, "question"]))

        _cand = []
        for _, r in labels.iterrows():
            q = str(r["qa_id"]); qt = str(r.get("query") or "")
            if q in _hold_ids or not qt.strip() or _nq(qt) in _hold_txt: continue
            gold = {int(p) for p in (r["gold_positions"] or [])}
            if not gold: continue
            ans = r.get("answer") if "answer" in labels.columns else None
            if not has_val(ans) and q in qa_by_id.index: ans = qa_by_id.at[q, "answer"]
            if not has_val(ans): continue
            pos = ft_pick_positive(gold, str(ans), _content, cap=1000)
            gdocs = {_doc_of[p] for p in gold} | {k[0] for k in (r["gold_keys"] or []) if k and k[0]}
            _cand.append((qt, pos, gdocs))
        print(f"\ntraining questions: {len(_cand):,}  "
              f"(held out {len(_hold_ids):,} val ids + {len(_hold_txt):,} val question texts)")
        assert len(_cand) >= 500, "fewer than 500 usable labelled questions — check CELL 3's label resolution"

        # ── ROUND-2 HARD NEGATIVES ────────────────────────────────────────
        # Mined by the retriever that is LOADED now. After CELL 5 that is the
        # adopted encoder (ft_v1), so these are ITS mistakes — harder than the
        # base model's, which is the point of a second round.
        # A stronger retriever also surfaces more FALSE negatives: relevant clauses
        # copied into amending or consolidated documents that the labels don't
        # cite. Pushing those away teaches the encoder the wrong thing. So every
        # candidate is first shown to the cross-encoder, and one it scores at or
        # above the question's own positive is dropped as probably relevant
        # (RocketQA's "denoised hard negatives").
        _rng = np.random.default_rng(cfg.seed)
        _qs = [c[0] for c in _cand]; _ranked = []
        for i in tqdm(range(0, len(_qs), 256), desc="mining negatives"):
            _ranked += hybrid_search(_qs[i:i+256], k=FT_NEG_POOL)
        _pools = [ft_hard_candidates(ranked, gdocs, ft_tokset(_content[pos]), _doc_of, _content,
                                     FT_NEG_MAX_JACCARD, FT_DENOISE_CANDS if FT_DENOISE else FT_N_NEG)
                  for (qt, pos, gdocs), ranked in zip(_cand, _ranked)]
        _kinds = Counter()
        if FT_DENOISE:
            _flat = [(i, p) for i, ((qt, pos, _), pl) in enumerate(zip(_cand, _pools)) for p in [pos] + pl]
            _pairs = [(_cand[i][0], rerank_text(p, _cand[i][0])) for i, p in _flat]
            _ord = sorted(range(len(_pairs)), key=lambda j: len(_pairs[j][1]))   # less padding
            _s = np.empty(len(_pairs), dtype=np.float32)
            _s[_ord] = reranker.predict([_pairs[j] for j in _ord], batch_size=PROFILE["rerank_batch"])
            _score = {}
            for (i, p), v in zip(_flat, _s): _score[(i, p)] = float(v)
        _trip = []
        for i, ((qt, pos, gdocs), pl) in enumerate(zip(_cand, _pools)):
            if FT_DENOISE:
                keep = [p for p in pl if _score[(i, p)] < _score[(i, pos)] - FT_DENOISE_MARGIN]
                _kinds["vetoed as probable false negatives"] += len(pl) - len(keep)
            else:
                keep = pl
            negs = keep[:FT_N_NEG]; _kinds["hard"] += len(negs)
            while len(negs) < FT_N_NEG:                       # nothing clean left: random other-doc
                r = ft_random_negative(gdocs, _doc_of, _rng, len(chunks), exclude=set(negs) | {pos})
                if r is None: break
                negs.append(r); _kinds["random (no clean hard one)"] += 1
            if len(negs) == FT_N_NEG: _trip.append((qt, pos, *negs))
        print(f"  negatives: {dict(_kinds)}  (hard = top-{FT_NEG_POOL} hybrid, uncited document, "
              f"not a copy{', cross-encoder-checked' if FT_DENOISE else ''})")

        # segmentation through the pool; anchors go through prep_query so training
        # sees queries exactly as dense_search will send them
        _need = sorted({p for t in _trip for p in t[1:]})
        import multiprocessing as _mp
        with _mp.get_context("spawn").Pool(FT_PROCS, initializer=wmod.init,
                                           initargs=(cfg.emb_model_id, FT_MAX_LEN)) as _pool:
            _raw = [ft_passage_raw(p) for p in _need]
            _seg = [s for part in _pool.imap(wmod.seg, [_raw[i:i+400] for i in range(0, len(_raw), 400)])
                    for s in part]
        _pseg = dict(zip(_need, _seg))
        FT_ROWS = [(prep_query(t[0]),) + tuple(_pseg[p] for p in t[1:]) for t in _trip]
        print(f"  {len(FT_ROWS):,} (anchor, positive, {FT_N_NEG} negatives) rows ready "
              f"({time.time()-t_all:.0f}s)")

        # ── 3. train ──────────────────────────────────────────────────────
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        _t0 = time.time()
        _enc_id = str(globals().get("ENC_ID", ""))
        _init = (_enc_id if FT_INIT == "current" and ENCODER_TAG != "base" and os.path.isdir(_enc_id)
                 else cfg.emb_model_id)
        _lr = FT_LR_CONTINUE if _init != cfg.emb_model_id else FT_LR
        print(f"  initialising from {'the adopted ' + ENCODER_TAG if _init != cfg.emb_model_id else 'the base encoder'}"
              f" · lr {_lr:g}")
        _st, _dims, _hist = ft_train(_init, FT_ROWS, MATRYOSHKA_DIMS, FT_EPOCHS,
                                     FT_BATCH, _lr, FT_WARMUP, bool(BF16_OK), cfg.seed,
                                     FT_MAX_LEN, FT_DEVICE, os.path.join(FT_LOCAL, "ft_tmp"))
        _L = [h["loss"] for h in _hist]
        print(f"\ntrained in {(time.time()-_t0)/60:.1f} min | matryoshka {_dims} | "
              f"loss {_L[0]:.3f} → {_L[-1]:.3f}")
        ft_save_hf(_st, FT_DIR)
        # the served encoder must BE the trained one — pooling, weights, tokenizer
        _probe = [r[1] for r in FT_ROWS[:48]] + [r[0] for r in FT_ROWS[:48]]
        _a = _st.encode(_probe, normalize_embeddings=True, convert_to_numpy=True, batch_size=64)
        _b = NativeEncoder(FT_DIR, device=FT_DEVICE, dtype=torch.float32,
                           max_len=FT_MAX_LEN).encode(_probe)
        _c = (_a * _b).sum(1)
        print(f"  NativeEncoder round trip: cosine min {_c.min():.5f}")
        assert _c.min() > 0.999, "NativeEncoder does not reproduce the trained model — do not adopt"
        _np0, _np1 = count_params(cfg.emb_model_id), count_params(FT_DIR)
        assert _np0 == _np1, f"parameter count changed {_np0} → {_np1}"
        print(f"  parameters {_np1/1e6:.1f}M — identical to the base, the 4.0B budget is unchanged")
        json.dump({"tag": FT_TAG, "base": cfg.emb_model_id, "init": _init, "dims": _dims,
                   "epochs": FT_EPOCHS, "batch": FT_BATCH, "lr": _lr, "rows": len(FT_ROWS),
                   "n_neg": FT_N_NEG, "denoise": FT_DENOISE, "negatives": dict(_kinds),
                   "loss": _L, "pooling": "mean+l2", "max_len": FT_MAX_LEN},
                  open(FT_META, "w"), indent=1)
        del _st; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    # ── 4. re-encode the corpus with the fine-tuned model ─────────────
    # Mandatory, not optional: a query from the NEW encoder against corpus
    # vectors from the OLD one scores near chance, and nothing would error.
    # Cached vectors are trusted only if they were written by THIS model: the
    # stamp holds a hash of ft_meta.json (weights' training record) and the row
    # count. Delete the model dir to retrain under the same tag, and the old
    # matrix and any half-finished shards are discarded instead of reused.
    _meta_sha = hashlib.sha1(open(FT_META, "rb").read()).hexdigest()[:12]
    _STAMP, _SHARDS = FT_EMB + ".stamp", W(f"emb_shards_{FT_TAG}")
    _want = f"{_meta_sha}:{len(chunks)}"
    if os.path.exists(FT_EMB) and not (os.path.exists(_STAMP) and open(_STAMP).read().strip() == _want):
        print(f"\n{os.path.basename(FT_EMB)} was written by a different model — re-encoding")
        os.remove(FT_EMB)
    if os.path.exists(FT_EMB):
        E_ft = np.load(FT_EMB, mmap_mode="r")
        assert E_ft.shape[0] == len(chunks), f"{E_ft.shape[0]:,} rows != {len(chunks):,} chunks"
        print(f"\n{os.path.basename(FT_EMB)} exists {E_ft.shape} — reusing it (stamp {_want})")
    else:
        _raw_all = [ft_passage_raw(i) for i in range(len(chunks))]
        _uniq, _inv = {}, np.empty(len(_raw_all), dtype=np.int64)
        for i, t in enumerate(_raw_all):
            _inv[i] = _uniq.setdefault(t, len(_uniq))
        _utexts = list(_uniq)
        print(f"\nre-encoding {len(_raw_all):,} passages → {len(_utexts):,} unique "
              f"(duplicates encoded once)")
        _sst = os.path.join(_SHARDS, "STAMP")
        if os.path.isdir(_SHARDS) and not (os.path.exists(_sst) and
                                            open(_sst).read().strip() == f"{_meta_sha}:{len(_utexts)}"):
            shutil.rmtree(_SHARDS, ignore_errors=True)       # shards from another model
        os.makedirs(_SHARDS, exist_ok=True)
        with open(_sst, "w") as f: f.write(f"{_meta_sha}:{len(_utexts)}")
        _em = AutoModel.from_pretrained(FT_DIR, torch_dtype=DTYPE).to(FT_DEVICE).eval()
        _tk = AutoTokenizer.from_pretrained(FT_DIR)
        _Eu = ft_encode_corpus(_utexts, _SHARDS, _em, FT_DIR, FT_MAX_LEN,
                               _tk.pad_token_id, FT_DEVICE, DTYPE, FT_PROCS, wmod,
                               batch=PROFILE.get("emb_batch", 512))
        E_ft = _Eu[_inv]
        with open(FT_EMB + ".tmp", "wb") as f:
            np.save(f, E_ft)
        os.replace(FT_EMB + ".tmp", FT_EMB)
        with open(_STAMP, "w") as f: f.write(_want)
        shutil.rmtree(_SHARDS, ignore_errors=True)
        del _em, _Eu, _raw_all, _uniq, _utexts, _inv; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    _ok, _rep = validate_embeddings(E_ft)
    assert _ok, "fine-tuned embeddings failed the identical-text test"
    # SERVING PARITY — the check that matters. Identical-text pairs are encoded
    # once (deduplicated), so the test above cannot see a pipeline bug. This one
    # re-encodes random rows through the SAME path queries use at serve time
    # (NativeEncoder) and requires the stored vectors to match.
    _smp = np.sort(np.random.default_rng(1).choice(len(chunks), 48, replace=False))
    _ne  = NativeEncoder(FT_DIR, device=FT_DEVICE, dtype=DTYPE, max_len=FT_MAX_LEN)
    _ref = _ne.encode([ViTokenizer.tokenize(ft_passage_raw(int(i))) for i in _smp])
    _got = np.asarray(E_ft[_smp], dtype=np.float32)
    _got /= np.clip(np.linalg.norm(_got, axis=1, keepdims=True), 1e-12, None)
    _cs  = (_ref * _got).sum(1)
    print(f"  serving parity (stored rows vs NativeEncoder, 48 random passages): "
          f"cosine min {_cs.min():.4f}")
    assert _cs.min() > 0.99, "stored corpus vectors do not match the serving encoder — do not adopt"
    del _ne

    # ── 5. A/B on held-out RET_VAL — same questions, same reranker ────
    # Decided by a PAIRED test, not a raw delta. OLD and NEW are scored on the
    # same questions, so only the questions where they DISAGREE carry any
    # information: b = NEW finds the article and OLD does not, c = the reverse.
    # Under "no real difference" b ~ Binomial(b+c, ½). A bare "+0.005 hit@8"
    # rule adopts pure noise: on 392 questions that is two questions.
    # (@3/@5 on retrieve_k are distorted by reorder_lost_in_middle, which moves
    # rank 2 LAST — so the reranked path is judged on @1 and @8 only.)
    from math import comb
    _ids_all = [q for q in sorted(RET_VAL) if q in labels_by_qid.index]   # sorted: same subset every session
    _ids_rr  = _ids_all[:150]                      # the reranker is the slow arm

    def _first_hits(fn, ids, K=8):
        """Rank of the first gold chunk per question (None = not in top K),
        using the notebook's own hit() so the criterion matches eval_retrieval."""
        val = labels_by_qid.loc[ids]
        ranked = fn(val["query"].tolist(), K)
        return [next((r for r, p in enumerate(list(pos)[:K]) if hit(p, row)), None)
                for pos, (_, row) in zip(ranked, val.iterrows())]

    def _summary(v):
        n = max(1, len(v))
        return {"hit@1": sum(f == 0 for f in v) / n, "hit@8": sum(f is not None for f in v) / n,
                "MRR": sum(1.0 / (f + 1) for f in v if f is not None) / n}

    def _mcnemar(win, lose):
        """One-sided exact McNemar: P(≥ b wins out of b+c discordant | p = ½)."""
        b = sum(1 for w, l in zip(win, lose) if w and not l)
        c = sum(1 for w, l in zip(win, lose) if l and not w)
        m = b + c
        return b, c, (sum(comb(m, k) for k in range(b, m + 1)) / 2 ** m) if m else 1.0

    _dense = lambda q, k: [list(map(int, r)) for r in dense_search(q, k)[0]]
    _hyb   = lambda q, k: [list(r) for r in hybrid_search(q, k)]
    print("\n" + "=" * 70 + f"\nA/B — {len(_ids_all)} held-out RET_VAL questions, "
          f"never seen in training\n" + "=" * 70)
    OLD_V = {"dense": _first_hits(_dense, _ids_all), "hybrid": _first_hits(_hyb, _ids_all),
             "rerank": _first_hits(retrieve_k, _ids_rr)}
    _saved = (query_encoder, faiss_index, globals().get("corpus_emb"))
    _E32 = np.ascontiguousarray(E_ft, dtype=np.float32)
    _ix = faiss.IndexFlatIP(_E32.shape[1])
    for i in range(0, len(_E32), 200_000): _ix.add(_E32[i:i+200_000])
    query_encoder = NativeEncoder(FT_DIR, device=FT_DEVICE, dtype=DTYPE, max_len=cfg.query_max_len)
    faiss_index, corpus_emb = _ix, _E32          # dense_search reads these at call time
    NEW_V = {"dense": _first_hits(_dense, _ids_all), "hybrid": _first_hits(_hyb, _ids_all),
             "rerank": _first_hits(retrieve_k, _ids_rr)}
    OLD = {k: _summary(v) for k, v in OLD_V.items()}
    NEW = {k: _summary(v) for k, v in NEW_V.items()}

    print(f"  {'':<8} {'metric':<7} {'OLD':>7} {'NEW':>7} {'Δ':>8}")
    for arm, keys in (("dense", ("hit@1", "hit@8", "MRR")), ("hybrid", ("hit@1", "hit@8", "MRR")),
                      ("rerank", ("hit@1", "hit@8"))):
        for k in keys:
            print(f"  {arm:<8} {k:<7} {OLD[arm][k]:>7.3f} {NEW[arm][k]:>7.3f} "
                  f"{NEW[arm][k]-OLD[arm][k]:>+8.3f}")

    _o8 = [f is not None for f in OLD_V["hybrid"]]; _n8 = [f is not None for f in NEW_V["hybrid"]]
    _b8, _c8, _p8 = _mcnemar(_n8, _o8)
    _o1 = [f == 0 for f in OLD_V["rerank"]];         _n1 = [f == 0 for f in NEW_V["rerank"]]
    _bh, _ch, _ph = _mcnemar(_o1, _n1)               # HARM test: does OLD beat NEW at top-1?
    _g8 = NEW["hybrid"]["hit@8"] - OLD["hybrid"]["hit@8"]
    _g1 = NEW["rerank"]["hit@1"] - OLD["rerank"]["hit@1"]
    print(f"\n  hybrid hit@8  NEW-only {_b8} vs OLD-only {_c8} questions → one-sided p = {_p8:.3f}")
    print(f"  rerank hit@1  OLD-only {_bh} vs NEW-only {_ch} questions → harm p = {_ph:.3f}")
    ADOPT = (_g8 >= FT_MIN_GAIN_HIT8 and _p8 <= FT_ADOPT_P
             and _g1 >= -FT_MAX_HIT1_LOSS and not _ph <= FT_ADOPT_P)

    if ADOPT:
        _when = time.strftime("%Y-%m-%d %H:%M")
        json.dump({"tag": FT_TAG, "dir": FT_DIR, "emb": FT_EMB, "old": OLD, "new": NEW,
                   "time": _when}, open(FT_ADOPT, "w"), indent=1, default=str)
        # the same four names CELL 5 sets from ENCODER_ADOPTED.json after a
        # restart — so CELL 7's retrieval fingerprint is identical either way
        ENCODER_TAG, ENCODER_SIG = FT_TAG, f"{FT_TAG}@{_when}"
        ENC_ID, EMB_PATH = FT_DIR, FT_EMB
        print(f"\n✓ ADOPTED {FT_TAG}: hybrid hit@8 {_g8:+.3f} (p = {_p8:.3f}), "
              f"reranked hit@1 {_g1:+.3f}")
        print(f"  dense_search now uses the fine-tuned encoder for the rest of this session;")
        print(f"  CELL 5 will load it (with its matching embeddings) after a restart.")
        print(f"  CELL 7's context cache is keyed by the retrieval config, so it rebuilds")
        print(f"  on its own, and CELLS 10/11 regenerate every answer whose prompt changed.")
    else:
        query_encoder, faiss_index, corpus_emb = _saved
        print(f"\n✗ NOT ADOPTED: hybrid hit@8 {_g8:+.3f} at p = {_p8:.3f} "
              f"(need ≥ {FT_MIN_GAIN_HIT8:+.3f} and p ≤ {FT_ADOPT_P}); "
              f"reranked hit@1 {_g1:+.3f}, harm p = {_ph:.3f}")
        print(f"  The base encoder stays in place; nothing downstream changes.")
        json.dump({"tag": FT_TAG, "g8": _g8, "p8": _p8, "g1": _g1, "ph": _ph, "old": OLD, "new": NEW,
                   "time": time.strftime("%Y-%m-%d %H:%M")}, open(FT_REJECT, "w"), indent=1, default=str)
        print(f"  Recorded in {os.path.basename(FT_REJECT)} — the next Run All skips this cell.")
    # Drop every EXTRA reference to the two retrieval stacks. Whichever one is live
    # is held by query_encoder / faiss_index / corpus_emb alone, so CELL 7's
    # free_vram() really frees it (~5 GB of RAM otherwise stays pinned here).
    del _saved, _ix, _E32, E_ft; gc.collect()
    RETRIEVER_FT = {"tag": FT_TAG, "adopted": ADOPT, "old": OLD, "new": NEW}
    print(f"\ncell total {(time.time()-t_all)/60:.1f} min")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6e — RETRIEVAL CONFIG LAB: pool depth · reranker input · lexref · context K
#  Run AFTER 6d (it measures the encoder you will ship) and BEFORE 7.  A100 ≈ 6-10 min
# ══════════════════════════════════════════════════════════════════════
# HOW EACH KNOB REACHES m      M = (1 − pen) · m / (0.9·|ref| + 0.1·|hyp|)
#
# The generator can only copy what the prompt shows it. So every retrieval knob
# acts on m through two measurable quantities, computed below on the SAME
# held-out questions for every configuration:
#   hit@K         P(a cited article is among the K chunks in the prompt) — labels
#   ctx-recall@K  IDF-weighted share of the REFERENCE ANSWER's tokens present
#                 anywhere in those K chunks — no labels needed. It is the ceiling
#                 on m for copied text: the retrieval number METEOR actually feels.
#
#   cand_pool 100 → 250   raises the ceiling hit@P; pays only if the reranker can
#                         then order the extra candidates — hit@8 is the proof.
#   RERANK_VARIANT v2     changes what the cross-encoder reads: ordering only.
#   lexref on/off         an arm that returns [] adds 0 to EVERY RRF score, so it
#                         cannot dilute the other arms. It matters only on the
#                         queries where it fires — so that is where it is judged.
#   top_k_context 8→10/12 adds (hit@K − hit@8) of questions whose gold article
#                         was just out of view — but every added chunk is also a
#                         DISTRACTOR for the questions whose article was already
#                         in view, and that cost cannot be measured on this val
#                         (its answers are in the SFT data). So the rule is
#                         conservative: at least K_GAIN_PER_CHUNK of hit per
#                         added chunk (K=10 needs +2 points, K=12 needs +4).
#
# A change is ADOPTED only when a paired one-sided sign test on the same
# questions says it is better (pool depth: not worse — it only costs compute),
# and never when reranked hit@1 gets significantly worse. Decisions go to
# RETRIEVAL_LAB.json; the next Run All re-applies them in a second unless the
# encoder, the labels or the retrieval code changed.
_MISSING = [n for n in ['hybrid_search_v2', 'lexref_search', 'reranker', '_rerank_text_v1',
                        '_rerank_text_v2', 'diversify', 'hit', 'labels_by_qid', 'RET_VAL',
                        'GEN_VAL', 'qa_by_id', 'chunks', 'chunk_header', 'cfg', 'W', 'PROFILE',
                        'test_data', '_rr_idf', '_rr_tokens', 'prep_retrieval_query', 'has_val']
            if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS 6 and 6b (and 6d) first; "
                       f"this cell must run BEFORE CELL 7, which frees the retrieval stack.")

import json, hashlib, time
from math import comb
from collections import Counter

RUN_RET_LAB     = True
FORCE_RET_LAB   = False       # re-measure even if RETRIEVAL_LAB.json matches this stack
LAB_POOLS       = (100, 250)  # first = what v11 shipped
LAB_VARIANTS    = ("v1", "v2")
LAB_KS          = (8, 10, 12) # first = what v11 shipped
LAB_N_GEN       = 120         # GEN_VAL questions added for the label-free metric
LAB_P           = 0.10        # one-sided sign-test level
K_GAIN_PER_CHUNK = 0.01       # hit@K − hit@8 needed PER ADDED CHUNK before the prompt grows
LEXREF_MIN_FIRE = 0.01        # below this fire rate on the TEST questions, lexref is dead
_LABF = W("RETRIEVAL_LAB.json")


# ── shared lab machinery (CELL 6f uses it too) ────────────────────────
def lab_sign_test(new, old):
    """One-sided exact sign test on PAIRED values: P(≥ b wins of the b + c
    non-tied pairs | p = ½), b = questions where NEW is better. For 0/1 values
    this is McNemar's test. Ties carry no information and are dropped."""
    b = sum(1 for n, o in zip(new, old) if n > o + 1e-12)
    c = sum(1 for n, o in zip(new, old) if o > n + 1e-12)
    m = b + c
    return b, c, (sum(comb(m, k) for k in range(b, m + 1)) / 2 ** m) if m else 1.0

def lab_fsig(fn):
    """Fingerprint of a function's code: bytecode + names + constants, walking
    nested code objects BY CONTENT. repr(co_consts) alone is not stable: on
    Python ≤ 3.11 every list comprehension is a nested code object whose repr
    carries its memory address, so the signature would change on every re-run
    of the defining cell and the fast path below would never fire."""
    def walk(co):
        parts = [co.co_code.hex(), ",".join(co.co_names)]
        for k in co.co_consts:
            if hasattr(k, "co_code"):
                parts.append(walk(k))
            elif isinstance(k, frozenset):
                parts.append(repr(sorted(map(repr, k))))     # set order varies per session
            else:
                parts.append(repr(k))
        return hashlib.sha1("|".join(parts).encode("utf-8")).hexdigest()[:10]
    co = getattr(getattr(fn, "__func__", fn), "__code__", None)
    return walk(co) if co else "?"

_mean = lambda v: float(np.mean(v)) if len(v) else 0.0
_lab_row = lambda q: (lambda r: r.iloc[0] if isinstance(r, pd.DataFrame) else r)(labels_by_qid.loc[q])

class RetrievalLabSet:
    """The held-out questions, and everything needed to score a ranked list for
    them. Reranker scores are cached per (question, chunk, variant), so a second
    configuration only pays for the candidates the first one never saw."""
    def __init__(self, n_gen=LAB_N_GEN):
        self.ret_q  = sorted(q for q in RET_VAL if q in labels_by_qid.index)
        self.retset = set(self.ret_q)
        self.rows   = {q: _lab_row(q) for q in self.ret_q}
        gen = [q for q in sorted(GEN_VAL) if q in qa_by_id.index and q not in self.retset][:n_gen]
        self.Q = {q: str(self.rows[q]["query"]) for q in self.ret_q}
        self.Q.update({q: str(qa_by_id.at[q, "question"]) for q in gen})
        self.qids = list(self.Q); self.qtx = [self.Q[q] for q in self.qids]
        self.idf, self.idf_max = _rr_idf()
        ref = {q: str(qa_by_id.at[q, "answer"]) for q in self.qids
               if q in qa_by_id.index and has_val(qa_by_id.at[q, "answer"])}
        self.refc = {q: Counter(_rr_tokens(r)) for q, r in ref.items()}
        self.refw = {q: sum(self.idf.get(w, self.idf_max) * c for w, c in rc.items()) or 1.0
                     for q, rc in self.refc.items()}
        self._pt, self._sc = {}, {}

    def ptoks(self, p):
        """Tokens of exactly what the PROMPT shows for chunk p."""
        t = self._pt.get(p)
        if t is None:
            t = self._pt[p] = Counter(_rr_tokens(
                f"{chunk_header(p)} {str(chunks.iloc[p]['content'])[:cfg.max_chunk_chars]}"))
        return t

    def ctx_recall(self, q, positions):
        have = Counter()
        for p in positions: have.update(self.ptoks(p))
        return sum(self.idf.get(w, self.idf_max) * min(c, have[w])
                   for w, c in self.refc[q].items()) / self.refw[q]

    def rank(self, cands, variant):
        """Cross-encoder order, then diversify — the SET retrieve_k keeps for any K."""
        need = [(i, p) for i, cl in enumerate(cands) for p in cl
                if (self.qids[i], p, variant) not in self._sc]
        if need:
            fn = _rerank_text_v2 if variant == "v2" else _rerank_text_v1
            pairs = [(self.qtx[i], fn(p, self.qtx[i])) for i, p in need]
            order = sorted(range(len(pairs)), key=lambda j: len(pairs[j][1]))   # less padding
            sc = reranker.predict([pairs[j] for j in order], batch_size=PROFILE["rerank_batch"])
            for j, s in zip(order, sc):
                i, p = need[j]; self._sc[(self.qids[i], p, variant)] = float(s)
        return [diversify(sorted(cl, key=lambda p: -self._sc[(self.qids[i], p, variant)]))
                for i, cl in enumerate(cands)]

    def metrics(self, lists, ks=LAB_KS):
        """Per-question vectors, so every comparison can be PAIRED."""
        first = [next((j for j, p in enumerate(lst) if hit(p, self.rows[q])), None)
                 for q, lst in zip(self.qids, lists) if q in self.retset]
        m = {f"hit@{K}": [f is not None and f < K for f in first] for K in (1,) + tuple(ks)}
        m["rr"] = [1.0 / (f + 1) if f is not None else 0.0 for f in first]
        for K in ks:
            m[f"cr@{K}"] = [self.ctx_recall(q, lst[:K]) for q, lst in zip(self.qids, lists)
                            if q in self.refc]
        m["ceiling"] = [any(hit(p, self.rows[q]) for p in lst)
                        for q, lst in zip(self.qids, lists) if q in self.retset]
        return m

def lab_pool_lists(qtexts, pool, lexref, search=None):
    """Fused candidates at a pool depth. Each arm reads cfg.cand_pool."""
    global USE_LEXREF
    saved = (cfg.cand_pool, USE_LEXREF)
    cfg.cand_pool, USE_LEXREF = pool, lexref
    try:
        return [list(map(int, r)) for r in (search or hybrid_search_v2)(qtexts, k=pool)]
    finally:
        cfg.cand_pool, USE_LEXREF = saved

def lab_row(label, m, base=None, k0=LAB_KS[0], k1=LAB_KS[-1]):
    cols = ["hit@1", f"hit@{k0}", f"hit@{k1}", "rr", f"cr@{k0}"]
    s = f"  {label:<30}" + "".join(f"{_mean(m[c]):>{12 if c.startswith('cr@') else 9}.3f}" for c in cols)
    if base is not None:
        b, c, p = lab_sign_test(m["rr"], base["rr"])
        s += f"   ΔMRR {_mean(m['rr']) - _mean(base['rr']):+.3f} ({b}↑ {c}↓, p {p:.3f})"
    return s

LAB_HEADER = (f"\n  {'config':<30}{'hit@1':>9}{'hit@' + str(LAB_KS[0]):>9}{'hit@' + str(LAB_KS[-1]):>9}"
              f"{'MRR':>9}{'ctx-rec@' + str(LAB_KS[0]):>12}")


# ── the lab ────────────────────────────────────────────────────────────
_ret_ids = sorted(q for q in RET_VAL if q in labels_by_qid.index)
_lab_sig = hashlib.sha1(json.dumps({
    "enc": globals().get("ENCODER_SIG", "base"), "ret": _ret_ids,
    "gold": hashlib.sha1(repr([sorted(map(str, _lab_row(q)["gold_keys"])) for q in _ret_ids])
                         .encode()).hexdigest(),
    "grid": [LAB_POOLS, LAB_VARIANTS, LAB_KS, LAB_N_GEN, LAB_P, K_GAIN_PER_CHUNK, LEXREF_MIN_FIRE],
    "flags": [globals().get("USE_ACRONYM"), globals().get("USE_WEIGHTED"), globals().get("_LEXV"),
              RERANK_CHARS, cfg.max_parts_per_article, cfg.max_chunk_chars, cfg.max_seq_len],
    "code": [lab_fsig(f) for f in (hybrid_search_v2, _rerank_text_v1, _rerank_text_v2, _rr_units,
                                   diversify, lexref_search, hit)],
}, sort_keys=True, default=str).encode()).hexdigest()[:12]

def _lab_apply(d, how):
    global RERANK_VARIANT, USE_LEXREF
    cfg.cand_pool, RERANK_VARIANT = int(d["cand_pool"]), d["rerank_variant"]
    USE_LEXREF, cfg.top_k_context = bool(d["use_lexref"]), int(d["top_k_context"])
    print(f"retrieval config ({how}): cand_pool {cfg.cand_pool} · rerank {RERANK_VARIANT} · "
          f"lexref {USE_LEXREF} · top_k_context {cfg.top_k_context}")

_fire = lambda qs: _mean([len(x) > 0 for x in lexref_search([prep_retrieval_query(s) for s in qs],
                                                            cfg.cand_pool)])
_tq_of = lambda q: test_data[q]["question"] if isinstance(test_data[q], dict) else str(test_data[q])

_rec = None
if os.path.exists(_LABF) and not FORCE_RET_LAB:
    try: _rec = json.load(open(_LABF, encoding="utf-8"))
    except Exception: _rec = None
if not RUN_RET_LAB:
    print("retrieval lab skipped (RUN_RET_LAB = False) — cell 2/6 settings stand")
elif _rec and _rec.get("sig") == _lab_sig:
    _d = dict(_rec["decisions"])
    # v13: rule 3 has a TEST-SET half (the fire rate on the test questions), and
    # the test set changed between phases — so that half is re-read on this one.
    _ft = _fire([_tq_of(q) for q in test_data])
    if _d["use_lexref"] and _ft < LEXREF_MIN_FIRE:
        _d["use_lexref"] = False
        print(f"  lexref fires on {_ft:.2%} of THIS test set (< {LEXREF_MIN_FIRE:.0%}): dead arm → OFF")
    elif not _d["use_lexref"] and _ft >= LEXREF_MIN_FIRE and _rec.get("lexref_harm_p", 0.0) > LAB_P:
        _d["use_lexref"] = True
        print(f"  lexref fires on {_ft:.2%} of THIS test set, and showed no harm on val → back ON")
    else:
        print(f"  lexref fires on {_ft:.2%} of this test set → decision unchanged")
    _lab_apply(_d, f"measured {_rec.get('time', '?')}, re-applied")
    RETRIEVAL_METRICS = _rec.get("final", globals().get("RETRIEVAL_METRICS"))
else:
    _t0 = time.time()
    RLAB = RetrievalLabSet()
    print(f"retrieval lab: {len(RLAB.ret_q)} labelled RET_VAL questions (hit@K, MRR) + "
          f"{len(RLAB.refc)} with reference answers (ctx-recall)")
    _P0, _V0, _K0 = LAB_POOLS[0], LAB_VARIANTS[0], LAB_KS[0]
    _cands = {P: lab_pool_lists(RLAB.qtx, P, True) for P in LAB_POOLS}
    _R = {(P, V): RLAB.metrics(RLAB.rank(_cands[P], V)) for P in LAB_POOLS for V in LAB_VARIANTS}
    print(LAB_HEADER)
    for (P, V), m in _R.items():
        print(lab_row(f"pool {P} · rerank {V}" + ("  (v11)" if (P, V) == (_P0, _V0) else ""),
                      m, None if (P, V) == (_P0, _V0) else _R[(_P0, _V0)]))
    print("  pool ceiling (gold anywhere in the pool): " +
          " · ".join(f"@{P} {_mean(_R[(P, _V0)]['ceiling']):.3f}" for P in LAB_POOLS))

    # ── 1. reranker input: must be BETTER ────────────────────────────────
    _dec = {"rerank_variant": _V0}
    if "v2" in LAB_VARIANTS:
        _b, _c, _p = lab_sign_test(_R[(_P0, "v2")]["rr"], _R[(_P0, _V0)]["rr"])
        _, _, _ph = lab_sign_test(_R[(_P0, _V0)]["hit@1"], _R[(_P0, "v2")]["hit@1"])
        if _p <= LAB_P and _ph > LAB_P:
            _dec["rerank_variant"] = "v2"
        print(f"\n  1. reranker input  v2 vs v1: MRR better on {_b}, worse on {_c} questions "
              f"(p {_p:.3f}); hit@1 harm p {_ph:.3f} → {_dec['rerank_variant']}")

    # ── 2. pool depth: must be NOT WORSE (it only costs compute) ─────────
    _V = _dec["rerank_variant"]; _P1 = LAB_POOLS[-1]
    _mb, _mc = _R[(_P0, _V)], _R[(_P1, _V)]
    _, _, _pr = lab_sign_test(_mb["rr"], _mc["rr"])
    _, _, _p1 = lab_sign_test(_mb["hit@1"], _mc["hit@1"])
    _ok = (_mean(_mc["rr"]) >= _mean(_mb["rr"]) and _mean(_mc[f"hit@{_K0}"]) >= _mean(_mb[f"hit@{_K0}"])
           and _pr > LAB_P and _p1 > LAB_P)
    _dec["cand_pool"] = _P1 if _ok else _P0
    print(f"  2. pool depth      {_P1} vs {_P0}: ceiling {_mean(_mb['ceiling']):.3f} → "
          f"{_mean(_mc['ceiling']):.3f}, hit@{_K0} {_mean(_mb[f'hit@{_K0}']):.3f} → "
          f"{_mean(_mc[f'hit@{_K0}']):.3f}, harm p (MRR {_pr:.3f}, hit@1 {_p1:.3f}) → {_dec['cand_pool']}")

    # ── 3. lexref: judged where it fires ─────────────────────────────────
    _P = _dec["cand_pool"]; _ON = _R[(_P, _V)]
    _OFF = RLAB.metrics(RLAB.rank(lab_pool_lists(RLAB.qtx, _P, False), _V))
    _tq = [(test_data[q]["question"] if isinstance(test_data[q], dict) else str(test_data[q]))
           for q in test_data]
    _fire = lambda qs: _mean([len(x) > 0 for x in lexref_search([prep_retrieval_query(s) for s in qs],
                                                                cfg.cand_pool)])
    _fv, _ft = _fire([RLAB.Q[q] for q in RLAB.ret_q]), _fire(_tq)
    _b, _c, _pb = lab_sign_test(_ON["rr"], _OFF["rr"])
    _, _, _pw = lab_sign_test(_OFF["rr"], _ON["rr"])
    _dec["use_lexref"] = not (_ft < LEXREF_MIN_FIRE or _pw <= LAB_P)
    print(f"  3. lexref          fires on {_fv:.1%} of val / {_ft:.1%} of test queries; ON vs OFF: "
          f"better on {_b}, worse on {_c} (help p {_pb:.3f}, harm p {_pw:.3f}) → "
          f"{'keep' if _dec['use_lexref'] else 'OFF'}")
    _F = _ON if _dec["use_lexref"] else _OFF

    # ── 4. context size: the gold article must actually come into view ───
    _dec["top_k_context"] = _K0
    try:
        _qtok = AutoTokenizer.from_pretrained(cfg.llm_id)
    except Exception as _ex:
        _qtok = None; print(f"  (prompt-length estimate skipped: {type(_ex).__name__})")
    _flists = RLAB.rank(lab_pool_lists(RLAB.qtx, _P, _dec["use_lexref"]), _V)
    _BT = {}
    def _btok(i, p):
        if (i, p) not in _BT:
            _BT[(i, p)] = len(_qtok(f"[Văn bản {i}] {chunk_header(p)}\n"
                                    f"{str(chunks.iloc[p]['content'])[:cfg.max_chunk_chars]}",
                                    add_special_tokens=False).input_ids)
        return _BT[(i, p)]
    print(f"  4. context K       gain over K={_K0}:")
    for K in LAB_KS[1:]:
        _dh = _mean(_F[f"hit@{K}"]) - _mean(_F[f"hit@{_K0}"])
        _, _, _pc = lab_sign_test(_F[f"cr@{K}"], _F[f"cr@{_K0}"])
        _dcr = _mean(_F[f"cr@{K}"]) - _mean(_F[f"cr@{_K0}"])
        _fit = ""
        if _qtok is not None:
            _est = [300 + len(_qtok(RLAB.Q[q], add_special_tokens=False).input_ids)
                    + sum(_btok(i + 1, p) for i, p in enumerate(lst[:K]))
                    for q, lst in zip(RLAB.qids, _flists)]
            _over = _mean([e > cfg.max_seq_len - 32 for e in _est])
            _fit = (f" | prompt ≈ p50 {np.percentile(_est, 50):.0f} / p95 {np.percentile(_est, 95):.0f}"
                    f" tok, {_over:.1%} over {cfg.max_seq_len} (CELL 9 drops their lowest-ranked chunks)")
        _take = _dh >= K_GAIN_PER_CHUNK * (K - _K0) and _pc <= LAB_P
        if _take: _dec["top_k_context"] = K
        print(f"     K={K:<3} hit {_dh:+.3f} (needs ≥ {K_GAIN_PER_CHUNK * (K - _K0):+.3f})  "
              f"ctx-recall {_dcr:+.3f} (p {_pc:.3f})"
              f"{'  ✓' if _take else ''}{_fit}")
    print(f"     → top_k_context {_dec['top_k_context']}"
          + ("" if _dec["top_k_context"] == _K0 else
             "  (every answer regenerates in CELLS 10/11: their prompts changed)"))

    _lab_apply(_dec, "measured now")
    _fin = RLAB.metrics(RLAB.rank(lab_pool_lists(RLAB.qtx, cfg.cand_pool, USE_LEXREF), RERANK_VARIANT))
    RETRIEVAL_METRICS = {k: _mean(v) for k, v in _fin.items()}
    _base = {k: _mean(v) for k, v in _R[(_P0, _V0)].items()}
    print(f"\n  shipped config vs v11:  hit@1 {_base['hit@1']:.3f} → {RETRIEVAL_METRICS['hit@1']:.3f} | "
          f"hit@{_K0} {_base[f'hit@{_K0}']:.3f} → {RETRIEVAL_METRICS[f'hit@{_K0}']:.3f} | "
          f"MRR {_base['rr']:.3f} → {RETRIEVAL_METRICS['rr']:.3f} | ctx-recall@K "
          f"{_base[f'cr@{_K0}']:.3f} → {RETRIEVAL_METRICS[f'cr@{cfg.top_k_context}']:.3f}")
    json.dump({"sig": _lab_sig, "decisions": _dec, "final": RETRIEVAL_METRICS, "v11": _base,
               "lexref_harm_p": _pw, "lexref_fire_test": _ft,
               "time": time.strftime("%Y-%m-%d %H:%M")}, open(_LABF, "w", encoding="utf-8"), indent=1)
    print(f"  lab {(time.time() - _t0) / 60:.1f} min; decisions saved to {os.path.basename(_LABF)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 6f — HyDE LAB: a hypothetical provision as a second query   (A100 ≈ 5-10 min)
#  Run AFTER 6e (it reuses 6e's scored candidates) and BEFORE 7.
# ══════════════════════════════════════════════════════════════════════
# WHY IT COULD RAISE m
# A question is phrased the way a CITIZEN talks ("tôi có bị phạt không nếu…");
# the article that answers it is phrased the way a DRAFTER writes ("Phạt tiền từ
# … đồng đối với hành vi …"). The encoder has to bridge that gap from the
# question alone. HyDE lets the generator bridge it first: Qwen writes a short
# provision-style paragraph that WOULD answer the question — its facts may be
# wrong, but its vocabulary and register are the corpus's — and the query becomes
#       v = normalise(v_question + λ · v_draft)
# (and, optionally, BM25 also reads the draft: 'query2doc'). If the gold article
# enters the top K more often, m rises exactly as in cell 6e: through hit@K and
# ctx-recall@K, which is what is measured here.
#
# Same Qwen2.5-3B the pipeline already counts (the 4.0B budget is unchanged),
# loaded here and FREED before CELL 7. A stage-1 screen compares variants on the
# fused candidate pool (cheap); only the best one pays for reranking, and it is
# adopted only if a paired sign test on MRR says it wins without costing hit@1.
_MISSING = [n for n in ['RetrievalLabSet', 'lab_sign_test', 'lab_pool_lists', 'lab_row', 'lab_fsig',
                        'hybrid_search_v2', 'bm25_search', 'lexref_search', 'rrf_fuse_w',
                        'rrf_weights', 'W_PLAIN', 'prep_retrieval_query', 'prep_query',
                        'query_encoder', 'faiss_index', 'reranker', 'rerank_text', 'diversify',
                        'reorder_lost_in_middle', 'test_data', 'GEN_VAL', 'cfg', 'W', 'DTYPE',
                        '_lab_row', '_rerank_text_v1', '_rerank_text_v2']
            if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS 6, 6b and 6e first; "
                       f"this cell must run BEFORE CELL 7, which frees the retrieval stack.")

import json, hashlib, time, gc

RUN_HYDE_LAB   = True
FORCE_HYDE_LAB = False
HYDE_LAMBDAS   = (0.5, 1.0)          # v = normalise(v_q + λ·v_draft)
HYDE_SCOPES    = ("all", "short")    # "short": questions of ≤ HYDE_SHORT_TOK words only
HYDE_SHORT_TOK = 20
HYDE_SPARSE    = (False, True)       # also append the draft to the BM25 query
HYDE_MAX_NEW   = 96
HYDE_BATCH     = 32
HYDE_P         = 0.10
HYDE_SYS  = "Bạn là chuyên gia soạn thảo văn bản quy phạm pháp luật Việt Nam."
HYDE_USER = ("Viết MỘT đoạn ngắn (1–2 câu) đúng văn phong điều khoản của văn bản pháp luật "
             "Việt Nam, có nội dung trả lời câu hỏi dưới đây. Chỉ viết nội dung điều khoản; "
             "không giải thích, không nêu số hiệu văn bản.\n\nCâu hỏi: {q}")
_HYDE_FILE, _HYDE_DRAFTS_FILE = W("HYDE_LAB.json"), W("hyde_drafts.json")
_HYDE_PSIG = hashlib.sha1(f"{cfg.llm_id}|{HYDE_SYS}|{HYDE_USER}|{HYDE_MAX_NEW}".encode()).hexdigest()[:10]

USE_HYDE    = globals().get("USE_HYDE", False)
HYDE        = globals().get("HYDE", {"lambda": 1.0, "scope": "all", "sparse": False})
HYDE_DRAFTS = globals().get("HYDE_DRAFTS", {})


# ── drafts: cached on disk by (prompt signature, question) ─────────────
_HY = {"lm": None, "tok": None}

def _hyde_free():
    if _HY["lm"] is not None:
        _HY["lm"] = _HY["tok"] = None
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f"  HyDE generator freed — VRAM free {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")

def hyde_drafts(questions):
    """{question: draft} — generated once per question and prompt, then read
    from hyde_drafts.json. The LLM is loaded only if something is missing."""
    cache = {}
    if os.path.exists(_HYDE_DRAFTS_FILE):
        try: cache = json.load(open(_HYDE_DRAFTS_FILE, encoding="utf-8"))
        except Exception: cache = {}
    key = lambda q: hashlib.sha1(f"{_HYDE_PSIG}\n{q}".encode("utf-8")).hexdigest()[:16]
    todo = sorted({q for q in questions if key(q) not in cache}, key=len)
    if todo:
        if _HY["lm"] is None:
            gc.collect(); torch.cuda.empty_cache()
            _HY["tok"] = AutoTokenizer.from_pretrained(cfg.llm_id)
            _HY["tok"].padding_side = "left"
            if _HY["tok"].pad_token is None: _HY["tok"].pad_token = _HY["tok"].eos_token
            _HY["lm"] = AutoModelForCausalLM.from_pretrained(cfg.llm_id, torch_dtype=DTYPE).to("cuda").eval()
            print(f"  HyDE generator loaded ({cfg.llm_id}) — VRAM free "
                  f"{torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")
        tok, lm = _HY["tok"], _HY["lm"]
        for i in tqdm(range(0, len(todo), HYDE_BATCH), desc="HyDE drafts", leave=False):
            qs = todo[i:i + HYDE_BATCH]
            prompts = [tok.apply_chat_template([{"role": "system", "content": HYDE_SYS},
                                                {"role": "user", "content": HYDE_USER.format(q=q)}],
                                               tokenize=False, add_generation_prompt=True) for q in qs]
            enc = tok(prompts, return_tensors="pt", padding=True,
                      return_token_type_ids=False).to("cuda")
            with torch.inference_mode():
                out = lm.generate(**enc, do_sample=False, max_new_tokens=HYDE_MAX_NEW,
                                  pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
            for q, t in zip(qs, tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                                 skip_special_tokens=True)):
                cache[key(q)] = re.sub(r"\s+", " ", t).strip()
            del enc, out
        _empty = sum(1 for q in todo if not cache[key(q)])
        if _empty:                    # stored anyway: an empty draft means "plain query" (_hyde_on)
            print(f"  ⚠ {_empty}/{len(todo)} new drafts are empty — those questions keep the plain query")
        with open(_HYDE_DRAFTS_FILE + ".tmp", "w", encoding="utf-8") as f:
            json.dump(cache, f, ensure_ascii=False)
        os.replace(_HYDE_DRAFTS_FILE + ".tmp", _HYDE_DRAFTS_FILE)
    return {q: cache[key(q)] for q in questions}


# ── the production path (inert until adopted) ──────────────────────────
def _hyde_on(q, hy):
    return bool(HYDE_DRAFTS.get(q)) and (hy["scope"] == "all" or len(str(q).split()) <= HYDE_SHORT_TOK)

def dense_search_fused(queries, drafts, lam, k):
    """dense_search, with v = normalise(v_q + λ·v_draft) wherever a draft exists."""
    qv = np.asarray(query_encoder.encode([prep_query(x) for x in queries], batch_size=256,
                                         normalize_embeddings=True), dtype=np.float32)
    idx = [i for i, d in enumerate(drafts) if d]
    if idx and lam:
        dv = np.asarray(query_encoder.encode([prep_query(drafts[i]) for i in idx], batch_size=256,
                                             normalize_embeddings=True), dtype=np.float32)
        qv[idx] = qv[idx] + lam * dv
        qv[idx] /= np.clip(np.linalg.norm(qv[idx], axis=1, keepdims=True), 1e-12, None)
    sc, ix = faiss_index.search(np.ascontiguousarray(qv), min(k, len(chunks)))
    return ix, sc

def hybrid_search_hyde(queries, k=None, hy=None):
    """hybrid_search_v2 with the HyDE draft in the dense arm (and BM25 if sparse)."""
    hy, k = hy or HYDE, k or cfg.cand_pool
    qq = [prep_retrieval_query(q) for q in queries]
    dr = [HYDE_DRAFTS.get(q) if _hyde_on(q, hy) else None for q in queries]
    bq = [f"{a} {d}" if (d and hy["sparse"]) else a for a, d in zip(qq, dr)]
    b, _ = bm25_search(bq, k=cfg.cand_pool)
    d, _ = dense_search_fused(qq, dr, hy["lambda"], cfg.cand_pool)
    lx = lexref_search(qq, cfg.cand_pool) if USE_LEXREF else [[]] * len(queries)
    out = []
    for i, q in enumerate(queries):
        w = rrf_weights(q) if USE_WEIGHTED else W_PLAIN
        out.append(rrf_fuse_w([b[i], d[i], lx[i]], [w["bm25"], w["dense"], 1.0], k_out=k))
    return out

def _route_retrieve_k():
    """Rebind retrieve_k through the HyDE path — only once HyDE is adopted, so a
    rejected lab leaves CELL 6b's retrieve_k (and CELL 7's cache key) untouched."""
    def retrieve_k(queries, k=None):
        k = k or cfg.top_k_context
        cands = hybrid_search_hyde(queries, k=cfg.cand_pool)
        out = []
        for q, cand in zip(tqdm(queries, desc="rerank", leave=False), cands):
            sc = reranker.predict([(q, rerank_text(p, q)) for p in cand],
                                  batch_size=PROFILE["rerank_batch"])
            ranked = [p for p, _ in sorted(zip(cand, sc), key=lambda x: -x[1])]
            out.append(reorder_lost_in_middle(diversify(ranked)[:k]))
        return out
    globals()["retrieve_k"] = retrieve_k
    globals()["RETRIEVE_ROUTE"] = "hybrid_v2+HyDE+rerank (cell 6f)"


# ── the lab ────────────────────────────────────────────────────────────
_ret_ids = sorted(q for q in RET_VAL if q in labels_by_qid.index)
_all_q = [(test_data[q]["question"] if isinstance(test_data[q], dict) else str(test_data[q]))
          for q in test_data]
_val_q = [str(qa_by_id.at[q, "question"]) for q in sorted(GEN_VAL) if q in qa_by_id.index][:120]
_hy_sig = hashlib.sha1(json.dumps({
    "enc": globals().get("ENCODER_SIG", "base"), "ret": _ret_ids, "psig": _HYDE_PSIG,
    "gold": hashlib.sha1(repr([sorted(map(str, _lab_row(q)["gold_keys"])) for q in _ret_ids])
                         .encode()).hexdigest(),
    "grid": [HYDE_LAMBDAS, HYDE_SCOPES, HYDE_SHORT_TOK, HYDE_SPARSE, HYDE_P],
    "cfg": [cfg.cand_pool, RERANK_VARIANT, bool(USE_LEXREF), cfg.top_k_context,
            globals().get("USE_ACRONYM"), globals().get("USE_WEIGHTED"), globals().get("_LEXV"),
            RERANK_CHARS, cfg.max_parts_per_article],
    "code": [lab_fsig(f) for f in (hybrid_search_hyde, dense_search_fused, hybrid_search_v2,
                                   _rerank_text_v1, _rerank_text_v2, _rr_units, diversify,
                                   lexref_search, hit)],
}, sort_keys=True, default=str).encode()).hexdigest()[:12]

_rec = None
if os.path.exists(_HYDE_FILE) and not FORCE_HYDE_LAB:
    try: _rec = json.load(open(_HYDE_FILE, encoding="utf-8"))
    except Exception: _rec = None
if not RUN_HYDE_LAB:
    print("HyDE lab skipped (RUN_HYDE_LAB = False) — retrieval unchanged")
elif _rec and _rec.get("sig") == _hy_sig:
    USE_HYDE, HYDE = bool(_rec["adopted"]), dict(_rec["hyde"])
    if USE_HYDE:
        HYDE_DRAFTS = hyde_drafts(_all_q + _val_q)      # cached; loads the LLM only if a draft is missing
        _route_retrieve_k()
    _hyde_free()
    print(f"HyDE ({_rec.get('time', '?')}, re-applied): "
          + (f"ADOPTED λ={HYDE['lambda']} scope={HYDE['scope']} sparse={HYDE['sparse']}"
             if USE_HYDE else "not adopted — retrieval unchanged"))
else:
    _t0 = time.time()
    RLAB = globals().get("RLAB") if isinstance(globals().get("RLAB"), RetrievalLabSet) else RetrievalLabSet()
    HYDE_DRAFTS = hyde_drafts(RLAB.qtx)
    _ex = next(((q, d) for q, d in HYDE_DRAFTS.items() if d), ("-", "(every draft is empty)"))
    print(f"  sample draft  Q: {_ex[0][:90]}\n                D: {_ex[1][:160]}")
    _P, _V = cfg.cand_pool, RERANK_VARIANT
    _base_c = lab_pool_lists(RLAB.qtx, _P, USE_LEXREF)
    _mb0 = RLAB.metrics(_base_c)
    _base_ceiling = _mean(_mb0["ceiling"])

    # ── stage 1: screen every variant on the fused pool (no reranking) ────
    print(f"\n  stage 1 — gold anywhere in the fused pool of {_P} (baseline {_base_ceiling:.3f}, "
          f"fused MRR {_mean(_mb0['rr']):.3f}):")
    _scr = {}
    for lam in HYDE_LAMBDAS:
        for scope in HYDE_SCOPES:
            for sparse in HYDE_SPARSE:
                hy = {"lambda": lam, "scope": scope, "sparse": sparse}
                c = lab_pool_lists(RLAB.qtx, _P, USE_LEXREF,
                                   search=lambda qs, k, hy=hy: hybrid_search_hyde(qs, k=k, hy=hy))
                _m = RLAB.metrics(c)
                _scr[(lam, scope, sparse)] = (_mean(_m["ceiling"]), _mean(_m["rr"]), c)
                print(f"     λ={lam:<4} scope={scope:<6} bm25+draft={str(sparse):<5} "
                      f"ceiling {_scr[(lam, scope, sparse)][0]:.3f}   fused MRR {_scr[(lam, scope, sparse)][1]:.3f}")
    # the reranker re-orders whatever the pool holds, so the pool CEILING decides;
    # the fused MRR only breaks ties
    _best = max(_scr, key=lambda v: (_scr[v][0], _scr[v][1]))
    _hy_best = {"lambda": _best[0], "scope": _best[1], "sparse": _best[2]}

    # ── stage 2: rerank the best variant against the baseline ─────────────
    _mB = RLAB.metrics(RLAB.rank(_base_c, _V))
    _mH = RLAB.metrics(RLAB.rank(_scr[_best][2], _V))
    print(LAB_HEADER)
    print(lab_row("baseline (6e config)", _mB))
    print(lab_row(f"HyDE λ={_best[0]} {_best[1]} sparse={_best[2]}", _mH, _mB))
    _b, _c, _p = lab_sign_test(_mH["rr"], _mB["rr"])
    _, _, _ph = lab_sign_test(_mB["hit@1"], _mH["hit@1"])
    _k = cfg.top_k_context
    _adopt = (_p <= HYDE_P and _ph > HYDE_P and _mean(_mH[f"hit@{_k}"]) >= _mean(_mB[f"hit@{_k}"]))
    print(f"\n  MRR better on {_b}, worse on {_c} questions → p {_p:.3f}; hit@1 harm p {_ph:.3f}"
          f"; hit@{_k} {_mean(_mB[f'hit@{_k}']):.3f} → {_mean(_mH[f'hit@{_k}']):.3f}")
    USE_HYDE, HYDE = bool(_adopt), _hy_best
    if USE_HYDE:
        print(f"  ✓ ADOPTED — generating drafts for the {len(_all_q)} test + {len(_val_q)} val "
              f"questions CELL 7 will retrieve for")
        # exactly the drafts CELL 7 needs — the same dict the re-applied path builds
        HYDE_DRAFTS = hyde_drafts(_all_q + _val_q)
        _route_retrieve_k()
    else:
        print("  ✗ not adopted — retrieve_k is untouched, CELL 7's contexts are unaffected")
    _hyde_free()
    json.dump({"sig": _hy_sig, "adopted": USE_HYDE, "hyde": HYDE,
               "screen": {f"{k[0]}|{k[1]}|{k[2]}": v[:2] for k, v in _scr.items()},
               "baseline": {k: _mean(v) for k, v in _mB.items()},
               "hyde_metrics": {k: _mean(v) for k, v in _mH.items()},
               "time": time.strftime("%Y-%m-%d %H:%M")},
              open(_HYDE_FILE, "w", encoding="utf-8"), indent=1)
    print(f"  HyDE lab {(time.time() - _t0) / 60:.1f} min")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 7 — Build test + val contexts (keyed by the retrieval config), free VRAM
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['retrieve_k', 'test_data', 'qa_by_id', 'W', 'cfg', 'chunks',
                        'GEN_VAL', 'free_vram'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 2, 3 and 6 (and 6b / 6d) first — scroll to ITS output to see why it "
        "stopped. A NameError here is a symptom, not the cause.")

# WHY THE FILENAME CHANGED
# v10 cached contexts as test_ctx_k8.parquet — keyed by K and nothing else. The
# lexref fix, the front-loaded reranker and every 6b change were computed,
# printed, and then REPLACED by contexts loaded from a file written before any
# of them existed. Your 0.5495 run printed "loaded test_ctx (1000, 2)" and
# "all answers cached": no answer in it saw any retrieval change since v6.
#
# The filename now carries a fingerprint of everything that decides which
# chunks a question gets — the configuration AND the bytecode of every function
# on the retrieval path. Change any of it (including adopting the CELL 6d
# encoder) and a new file is built; change nothing and the cached one is reused.
# The v10 files stay untouched on Drive: CELLS 10/11 use them to prove which old
# answers are still valid.
import hashlib

def _stable_code(fn):
    """Bytecode + constants + names + defaults, recursively. Stable across
    sessions and across re-runs of the defining cell — unlike marshal, which
    embeds the cell's temp-file name and line numbers."""
    def walk(co):
        parts = [co.co_code.hex(), ",".join(co.co_names)]
        for k in co.co_consts:
            if hasattr(k, "co_code"):
                parts.append(walk(k))
            elif isinstance(k, frozenset):
                parts.append(repr(sorted(map(repr, k))))     # set order varies per session
            else:
                parts.append(repr(k))
        return hashlib.sha1("|".join(parts).encode("utf-8")).hexdigest()[:12]
    f = getattr(fn, "__func__", fn)
    c = getattr(f, "__code__", None)
    return (walk(c) + repr(getattr(f, "__defaults__", None))) if c else "?"

_g = globals()
_RET_FUNCS = ["retrieve_k", "hybrid_search", "hybrid_search_v2", "rerank_text", "diversify",
              "reorder_lost_in_middle", "lexref_search", "_core", "rrf_fuse", "rrf_fuse_w",
              "rrf_weights", "expand_acronyms", "prep_retrieval_query", "dense_search",
              "prep_query", "bm25_search", "chunk_header", "title_of", "norm_key",
              "_rerank_text_v1", "_rerank_text_v2", "_rr_clean", "_rr_tokens",      # v12: CELL 6
              "_rr_units", "_rr_idf"]
# v13: the QUESTION SETS are inputs, like the chunks: they key the context files
# next to the configuration. v12 named test contexts by the configuration alone,
# so a new test file under an unchanged configuration would have LOADED the
# public contexts, and CELL 12 would have packaged 1,000 public answers.
_tq_of = lambda q: test_data[q]["question"] if isinstance(test_data[q], dict) else str(test_data[q])
_TIDS, _TQS = [str(q) for q in test_data], [_tq_of(q) for q in test_data]
_VIDS = [q for q in sorted(GEN_VAL) if q in qa_by_id.index][:120]   # sorted: the same 120 every session
_VQS  = [str(qa_by_id.at[q, "question"]) for q in _VIDS]
if _g.get("USE_HYDE"):
    # HyDE's functions enter the fingerprint only when HyDE is ON: a rejected or
    # skipped 6f must leave the fingerprint (and the cached contexts) untouched.
    _RET_FUNCS += ["hybrid_search_hyde", "dense_search_fused", "_hyde_on"]           # v12: CELL 6f
    _HQ = _TQS + _VQS                     # the questions this cell retrieves for
    # every question needs an ENTRY; an empty draft is a legitimate entry (_hyde_on
    # then keeps the plain query for it), a missing one means 6f never saw it
    _miss = [q for q in _HQ if q not in HYDE_DRAFTS]
    assert not _miss, (f"HyDE is adopted but {len(_miss)} questions have no draft — re-run CELL 6f "
                       f"(it generates the missing ones) or set USE_HYDE = False")
    _empty = sum(1 for q in _HQ if not HYDE_DRAFTS[q])
    if _empty:
        print(f"  HyDE: {_empty}/{len(_HQ)} drafts are empty — those questions retrieve with the plain query")
RET_CFG = {
    "K": cfg.top_k_context, "cand_pool": cfg.cand_pool, "rrf_k": cfg.rrf_k,
    "max_parts": cfg.max_parts_per_article, "query_max_len": cfg.query_max_len,
    "encoder": _g.get("ENCODER_SIG", f"base:{cfg.emb_model_id}"),
    "emb_file": os.path.basename(str(_g.get("EMB_PATH", PATHS["emb"]))),
    "reranker": cfg.reranker_id, "route": _g.get("RETRIEVE_ROUTE", "?"),
    "rerank_variant": _g.get("RERANK_VARIANT", "v1"),
    "hyde": ({"on": True, **_g.get("HYDE", {}), "psig": _g.get("_HYDE_PSIG"),
              "short_tok": _g.get("HYDE_SHORT_TOK")}          # the drafts key the SET files below
             if _g.get("USE_HYDE") else {"on": False}),
    "lexref": bool(_g.get("USE_LEXREF", False)), "lexref_v": _g.get("_LEXV"),
    "frontload": bool(_g.get("RERANK_FRONTLOAD", False)), "rerank_chars": _g.get("RERANK_CHARS"),
    "acronym": bool(_g.get("USE_ACRONYM", False)), "weighted": bool(_g.get("USE_WEIGHTED", False)),
    "w_lex": _g.get("W_LEX"), "w_plain": _g.get("W_PLAIN"),
    "acronyms": sorted(_g.get("ACRONYMS", {}).items()),
    "patterns": {n: getattr(_g.get(n), "pattern", None)
                 for n in ("_DOC", "_NUM", "_ART", "_LEXQ", "_ACR_RE", "_CLAUSE_SPLIT",
                           "_RR_TAG_DOC", "_RR_TAG", "_RR_WORD", "_RR_SENT")},
    "rr_intent": [(a.pattern, b.pattern, w) for a, b, w in _g.get("_RR_INTENT", [])],
    "n_chunks": len(chunks),
    "code": {n: _stable_code(_g[n]) for n in _RET_FUNCS if n in _g},
    "code_cls": {n: _stable_code(getattr(_g[c], m)) for n, c, m in
                 (("rerank", "NativeCrossEncoder", "predict"), ("encode", "NativeEncoder", "encode"))
                 if c in _g},
}
RET_FP = hashlib.sha1(json.dumps(RET_CFG, sort_keys=True, default=str).encode()).hexdigest()[:10]

def _set_fp(ids, qs):
    """Fingerprint of the questions a context file answers — and, under HyDE,
    of the drafts it retrieved with. Order-free: a reshuffled file is the same set."""
    _hy = bool(_g.get("USE_HYDE"))
    rows = sorted([q, t, HYDE_DRAFTS.get(t) if _hy else None] for q, t in zip(ids, qs))
    return hashlib.sha1(json.dumps(rows, ensure_ascii=False).encode("utf-8")).hexdigest()[:8]
TEST_CTX_FP, VAL_CTX_FP = _set_fp(_TIDS, _TQS), _set_fp(_VIDS, _VQS)
_K = cfg.top_k_context
_CTX  = W(f"test_ctx_k{_K}_{RET_FP}_{TEST_CTX_FP}.parquet")
_VCTX = W(f"val_ctx_k{_K}_{RET_FP}_{VAL_CTX_FP}.parquet")
_LCTX, _LVCTX = W(f"test_ctx_k{_K}.parquet"), W(f"val_ctx_k{_K}.parquet")     # v10 names
with open(W(f"ctx_{RET_FP}.json"), "w", encoding="utf-8") as _f:          # what built them
    json.dump(RET_CFG, _f, ensure_ascii=False, indent=1, default=str)
print(f"retrieval fingerprint {RET_FP} | encoder {RET_CFG['encoder']} | pool {cfg.cand_pool} | "
      f"K {_K} | rerank {RET_CFG['rerank_variant']} | lexref {RET_CFG['lexref']} "
      f"({RET_CFG['lexref_v']}) | HyDE {RET_CFG['hyde']['on']} | acronym {RET_CFG['acronym']} "
      f"| weighted {RET_CFG['weighted']}")
print(f"test set: {len(_TIDS):,} questions (set {TEST_CTX_FP}) | val: {len(_VIDS)} (set {VAL_CTX_FP})")

def _load_ctx(path):
    d = pd.read_parquet(path); d["qa_id"] = d["qa_id"].astype(str)
    return d.set_index("qa_id")

def _save_ctx(df, path):
    df.to_parquet(path + ".tmp")          # atomic: a killed runtime never leaves
    os.replace(path + ".tmp", path)       # a truncated file that looks complete

if os.path.exists(_CTX):
    test_ctx = _load_ctx(_CTX); print(f"loaded test_ctx {test_ctx.shape} (same fingerprint)")
else:
    qids, questions = _TIDS, _TQS
    pos = []
    for i in tqdm(range(0, len(questions), 128), desc=f"retrieve K={_K}"):
        pos += retrieve_k(questions[i:i+128])
    _df = pd.DataFrame({"qa_id": qids, "question": questions,
                        "ctx_positions": [list(map(int, p)) for p in pos]})
    _save_ctx(_df, _CTX); test_ctx = _df.set_index("qa_id")
    print(f"built test_ctx {test_ctx.shape}")

if os.path.exists(_VCTX):
    val_ctx = _load_ctx(_VCTX)
else:
    vq, vqs = _VIDS, _VQS
    vpos = []
    for i in tqdm(range(0, len(vq), 128), desc="retrieve val"):
        vpos += retrieve_k(vqs[i:i+128])
    _df = pd.DataFrame({"qa_id": vq, "question": vqs,
                        "ctx_positions": [list(map(int, p)) for p in vpos]})
    _save_ctx(_df, _VCTX); val_ctx = _df.set_index("qa_id")
print(f"test_ctx {test_ctx.shape} | val_ctx {val_ctx.shape}")
# v13: the contexts must answer EXACTLY the test file CELL 3 loaded — ids and text
_xt = set(map(str, test_ctx.index))
assert len(test_ctx) == len(_TIDS) and _xt == set(_TIDS), (
    f"test_ctx ≠ the test file: {len(set(_TIDS) - _xt)} ids missing, {len(_xt - set(_TIDS))} extra. "
    f"Delete {os.path.basename(_CTX)} and re-run this cell.")
_bad = [q for q, t in zip(_TIDS, _TQS) if str(test_ctx.at[q, "question"]) != t]
assert not _bad, f"{len(_bad)} test_ctx questions differ from the test file (e.g. ids {_bad[:3]})"

# How much did retrieval actually change since the contexts your scored runs used?
# A question whose chunks changed gets a fresh answer in CELL 11; the others can
# keep theirs (CELL 11 compares the whole prompt, not just the chunk list). If
# this says 0, nothing you changed upstream reached the contexts.
CTX_DELTA = {}
for _nm, _new, _old in (("test", test_ctx, _LCTX), ("val", val_ctx, _LVCTX)):
    if not os.path.exists(_old):
        continue
    _o = _load_ctx(_old)
    _common = [q for q in _new.index if q in _o.index             # v13: same id AND same
               and str(_o.at[q, "question"]) == str(_new.at[q, "question"])]   # question
    _same = sum(1 for q in _common if list(map(int, _o.at[q, "ctx_positions"]))
                == list(map(int, _new.at[q, "ctx_positions"])))
    _top1 = sum(1 for q in _common
                if int(_o.at[q, "ctx_positions"][0]) != int(_new.at[q, "ctx_positions"][0]))
    CTX_DELTA[_nm] = {"changed": len(_common) - _same, "top1_changed": _top1, "n": len(_common)}
    print(f"  vs v10 {_nm} contexts ({os.path.basename(_old)}): {len(_common)-_same}/{len(_common)} "
          f"questions see different chunks, {_top1} a different top-1")

free_vram("reranker", "query_encoder", "faiss_index", "corpus_emb")
_f, _t = torch.cuda.mem_get_info()
print(f"VRAM free {_f/1e9:.1f}/{_t/1e9:.0f} GB")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL T4 — Load Qwen2.5-3B + LoRA adapter, merged to native bf16
# ══════════════════════════════════════════════════════════════════════
# T1/T2/T3 DO NOT EXIST IN THIS NOTEBOOK. There is no SFTTrainer, no dataset
# builder, no optimizer — nothing here can start a training run, so Run All is
# safe and cannot silently burn an hour of A100 time. The trainable variant is
# the separate SFT notebook.
USE_TRAINED_ADAPTER = False     # False → the pre-trained adapter resolved in CELL 0
                                #         (dangphuc2109/legalqa-qwen2.5-3b-adapter)
                                # True  → a locally trained adapter, which this
                                #         notebook cannot produce; it will hard-fail
                                #         rather than quietly fall back.

_MISSING = [n for n in ['ADAPTER_DIR', 'cfg', 'vram', 'DTYPE'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 0, 1 and 2 first — scroll to THEIR output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

if USE_TRAINED_ADAPTER:
    _local = W("checkpoints/generator/hf_adapter")
    if not os.path.exists(os.path.join(_local, "adapter_config.json")):
        raise RuntimeError(
            f"USE_TRAINED_ADAPTER=True but no adapter at {_local}.\n"
            "This notebook has no training cells, so nothing can have written one. "
            "Set USE_TRAINED_ADAPTER=False to use the pre-trained adapter from CELL 0.")
    ADAPTER_IN_USE = _local
    print(f"adapter: LOCALLY TRAINED → {ADAPTER_IN_USE}")
else:
    ADAPTER_IN_USE = ADAPTER_DIR
    print(f"adapter: PRE-TRAINED (CELL 0) → {ADAPTER_IN_USE}")
assert os.path.exists(os.path.join(ADAPTER_IN_USE, "adapter_config.json")), \
    f"no adapter_config.json under {ADAPTER_IN_USE} — re-run CELL 0"


def load_tokenizer(adapter_dir, base_id):
    """This adapter's tokenizer_config.json stores extra_special_tokens as a
    LIST of 13 Qwen control tokens; transformers expects a MAPPING and calls
    .keys() on it. The field is redundant (all 13 are already in
    added_tokens_decoder), so overriding it with {} as a kwarg — which wins over
    the file — loses nothing and needs no file surgery."""
    if os.path.exists(os.path.join(adapter_dir, "tokenizer_config.json")):
        for tag, kw in (("as-is", {}), ("extra_special_tokens={}", {"extra_special_tokens": {}})):
            try:
                return AutoTokenizer.from_pretrained(adapter_dir, **kw), f"adapter [{tag}]"
            except Exception as ex:
                print(f"  adapter tokenizer [{tag}] failed — {type(ex).__name__}: {str(ex)[:80]}")
        _a = json.load(open(os.path.join(adapter_dir, "adapter_config.json"), encoding="utf-8"))
        _t = list(_a.get("modules_to_save") or []) + list(_a.get("target_modules") or [])
        if any(("embed" in str(m)) or ("lm_head" in str(m)) for m in _t):
            print("  ⚠ adapter trains embed_tokens/lm_head — base tokenizer may not match")
        else:
            print("  (adapter touches only attention/MLP projections → vocab unchanged,"
                  " base tokenizer is exact)")
    return AutoTokenizer.from_pretrained(base_id), "base model"

tokenizer, _src = load_tokenizer(ADAPTER_IN_USE, cfg.llm_id)
print(f"tokenizer: {_src} | vocab {len(tokenizer):,}")
tokenizer.padding_side, tokenizer.truncation_side = "left", "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# The adapter ships chat_template.jinja as a STANDALONE file and its
# tokenizer_config.json has no chat_template key. If the tokenizer comes back
# without one, apply_chat_template() emits a bare concatenation with no
# <|im_start|> markers — off-distribution from the fine-tune, and silent.
if not getattr(tokenizer, "chat_template", None):
    _j = os.path.join(ADAPTER_IN_USE, "chat_template.jinja")
    if os.path.exists(_j):
        tokenizer.chat_template = open(_j, encoding="utf-8").read()
        print("  chat_template: from the adapter's chat_template.jinja")
    else:
        tokenizer.chat_template = AutoTokenizer.from_pretrained(cfg.llm_id).chat_template
        print("  chat_template: borrowed from the base model")
else:
    print("  chat_template: present ✓")

_probe = tokenizer.apply_chat_template(
    [{"role": "system", "content": "S"}, {"role": "user", "content": "U"}],
    tokenize=False, add_generation_prompt=True)
assert "<|im_start|>" in _probe, f"chat template emits no ChatML markers: {_probe[:120]!r}"
print(f"  chat probe ok: {_probe[:56]!r}…")

# On an A100, 4-bit is the wrong trade: NF4 dequantizes every tile on every
# forward pass to buy memory we already have. merge_and_unload also folds away
# 252 extra LoRA matmuls per forward (7 projections × 36 layers).
# Re-running this cell without freeing first strands the previous model: the
# old `model` stays referenced until the new assignment completes, so peak holds
# TWO copies, and CUDA memory is not returned until gc + empty_cache run.
for _nm in ("model", "reranker", "query_encoder", "faiss_index", "corpus_emb"):
    if _nm in globals(): del globals()[_nm]
gc.collect(); torch.cuda.empty_cache()
_free = vram("before load")
assert _free > 12.0, (
    f"only {_free:.1f} GB free before loading a ~6.2 GB model plus activations. "
    f"Runtime → Restart session, then run cells 0-T4 again.")

model = AutoModelForCausalLM.from_pretrained(cfg.llm_id, torch_dtype=DTYPE,
                                             attn_implementation=ATTN_IMPL,
                                             device_map={"": 0})
model = PeftModel.from_pretrained(model, ADAPTER_IN_USE, torch_dtype=DTYPE)
model = model.merge_and_unload()
model.eval(); model.config.use_cache = True
for _k in ("temperature", "top_p", "top_k"):      # silence sampling warnings:
    setattr(model.generation_config, _k, None)    # do_sample=False ignores them
_devs = {p.device.type for p in model.parameters()}
assert _devs == {"cuda"}, f"CPU offloading detected: {_devs} — refusing to run slow"
gc.collect(); torch.cuda.empty_cache()     # drop the pre-merge PeftModel wrapper

# ── COMPLIANCE GATE: ≤ 4.0B parameters, counted AFTER the merge ───────
# This is the count that disqualifies a submission, and the merge is the only
# moment it can be measured for real: before it, the LoRA matrices are separate
# tensors; after it, they are folded into the base weights. Qwen2.5-3B-Instruct
# is 3.086B, so the merge must not change the total at all — LoRA folds into
# existing matrices rather than adding any.
_NP = sum(p.numel() for p in model.parameters())
print(f"\nPARAMETER BUDGET  {_NP/1e9:.3f}B / 4.000B limit  ({100*_NP/4e9:.1f}% used)")
assert _NP <= 4.0e9, (
    f"MERGED MODEL IS {_NP/1e9:.3f}B — OVER THE 4B LIMIT. Any submission from this "
    f"model is disqualified. (Your 0.5487 run was a >4B model; this is the check "
    f"that would have caught it.)")
print(f"model merged on GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
vram("after merge")
ADAPTER_IN_USE = ADAPTER_IN_USE


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 9 — Prompt, citation builder, metrics
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['tokenizer', 'chunks', 'chunk_keys', 'SIB', 'chunk_header'] if n not in globals()]
import hashlib
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 4 and 8 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

SYS_B = (
    "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. Nhiệm vụ của bạn là trả lời "
    "câu hỏi của người dùng DỰA TRÊN các trích đoạn văn bản pháp luật được cung cấp.\n"
    "Yêu cầu bắt buộc về cách trả lời:\n"
    "- Mở đầu bằng việc nêu căn cứ pháp lý (ví dụ: \"Căn cứ khoản 3 Điều 17 Nghị định 90/2017/NĐ-CP...\").\n"
    "- Trích dẫn ĐẦY ĐỦ, nguyên văn TẤT CẢ các khoản, điểm có liên quan đến câu hỏi "
    "(kể cả khi có nhiều khoản), giữ nguyên cách đánh số 1., 2., a), b) như trong văn bản. "
    "Không tóm tắt, không rút gọn nội dung điều luật. Sau khi trích dẫn đầy đủ mới đưa ra kết luận.\n"
    "- Trả lời mạch lạc bằng văn xuôi tiếng Việt, không bịa đặt điều khoản không có "
    "trong ngữ cảnh, không nhắc đến việc bạn được cung cấp ngữ cảnh."
)

# v11: the ORIGINAL prompt format is the default again. v9/v10 stripped the
# chunker scaffold inside the prompt, but no answer was ever generated from that
# prompt: the raw caches were keyed by question id, so both scored runs (0.5486,
# 0.5495) shipped answers generated from THIS format. It is also the format
# CELL T1 built the adapter's SFT data with — stripping it is an unmeasured
# shift in the model's input. The CITATION block stays scaffold-stripped (STRIP_SCAFFOLD
# below); that one was measured. Set True only to A/B it through cell 10.
PROMPT_STRIP_SCAFFOLD = False

def _render_prompt(question, positions, strip):
    blocks = []
    for i, p in enumerate(positions, 1):
        p = int(p)
        head, body = chunk_header(p), str(chunks.iloc[p]["content"])
        if strip:
            head, body = clean_header(head), strip_scaffold(body)
        blocks.append(f"[Văn bản {i}] {head}\n{body[:cfg.max_chunk_chars]}")
    user = ("Các trích đoạn văn bản pháp luật:\n\n" + "\n\n".join(blocks) +
            f"\n\nCâu hỏi: {str(question).strip()}\n\nTrả lời:")
    return tokenizer.apply_chat_template(
        [{"role":"system","content":SYS_B},{"role":"user","content":user}],
        tokenize=False, add_generation_prompt=True)

# ── v12: a prompt that does not fit loses its WEAKEST chunks, not its head ──
# CELL T4 sets truncation_side = "left", so a prompt longer than cfg.max_seq_len
# used to lose its START: the system instructions and, because
# reorder_lost_in_middle puts rank 1 first, the BEST chunk. At K = 8 that almost
# never happens; at K = 10-12 it can. Now the lowest-ranked chunks are dropped
# until the prompt fits. A prompt that already fits is byte-identical to v11,
# so every cached answer stays valid.
PROMPT_FIT        = True
PROMPT_FIT_MARGIN = 32          # tokens kept free below cfg.max_seq_len

def _rank_order(positions):
    """Invert reorder_lost_in_middle: [r1, r3, r5, …, r6, r4, r2] → [r1, r2, r3, …]."""
    n, h = len(positions), (len(positions) + 1) // 2
    head, tail = list(positions[:h]), list(positions[h:])[::-1]
    return [head[i // 2] if i % 2 == 0 else tail[i // 2] for i in range(n)]

def to_infer_text(question, positions, strip=None):
    strip = PROMPT_STRIP_SCAFFOLD if strip is None else strip
    positions = [int(p) for p in positions]
    prompt = _render_prompt(question, positions, strip)
    if not PROMPT_FIT:
        return prompt
    budget = cfg.max_seq_len - PROMPT_FIT_MARGIN
    n_tok = lambda s: len(tokenizer(s, add_special_tokens=False).input_ids)
    if n_tok(prompt) <= budget:
        return prompt
    keep = list(positions)
    for p in reversed(_rank_order(positions)[1:]):    # lowest rank first; rank 1 always stays
        keep.remove(p)
        prompt = _render_prompt(question, keep, strip)
        if n_tok(prompt) <= budget:
            break
    return prompt

# ── clause splitter (used only by cite_mode="sniper") ─────────────────
_MARK = re.compile(r"(?:^|(?<=\n)|(?<=[.;:])\s|(?<=\s))\s*((?:\d{1,2}\.)|(?:[a-zA-ZđĐ]\)))(?=\s+\S)")
def split_clauses(text):
    """Markers accepted only in ASCENDING runs, so '5.000.000 đồng' can never be
    mistaken for Khoản 5 (which would shred the article)."""
    marks = [(m.start(1), m.group(1)) for m in _MARK.finditer(text)]
    keep, last_n, last_a = [], 0, ""
    for s, g in marks:
        if g[0].isdigit():
            n = int(g[:-1])
            if n == last_n + 1 or (n == 1 and last_n == 0):
                keep.append((s,g)); last_n = n; last_a = ""
        else:
            ch = g[0].lower()
            if last_a == "" or ord(ch) > ord(last_a): keep.append((s,g)); last_a = ch
    if not keep: return [(None, text)]
    out = []
    if keep[0][0] > 0: out.append((None, text[:keep[0][0]].strip()))
    for i,(s,g) in enumerate(keep):
        end = keep[i+1][0] if i+1 < len(keep) else len(text)
        out.append((g, text[s:end].strip()))
    return [(m,t) for m,t in out if t]

_VI_STOP = set("""và của có là được các những cho với trong khi thì mà này đó nếu về như theo
tại từ đến ra vào một hai người việc phải sẽ đã không hay hoặc bị do nào gì thế bao nhiêu ai
sao quy định trường hợp thực hiện đối tượng bạn tôi hỏi xin cảm ơn ạ vậy còn cũng nên""".split())
def q_terms(q):
    t = re.sub(r"[^\w\s/\-]", " ", unicodedata.normalize("NFC", str(q).lower()))
    return [w for w in t.split() if w not in _VI_STOP and len(w) > 1]
def _seg_score(seg, terms):
    s = unicodedata.normalize("NFC", seg.lower())
    h = sum(1 for t in terms if t in s)
    return (h/max(1,len(set(terms)))) * (1 + 0.15*np.log1p(h)) / (1 + len(seg)/4000)
def sniper_extract_clauses(text, question, budget):
    segs = split_clauses(text)
    if len(segs) == 1 and segs[0][0] is None: return text[:budget]
    terms = q_terms(question)
    scored = sorted(((_seg_score(s,terms) + (10.0 if m is None else 0.0), i)
                     for i,(m,s) in enumerate(segs)), reverse=True)
    chosen, used = set(), 0
    for sc, i in scored:
        seg = segs[i][1]
        if used + len(seg) > budget and chosen: continue
        chosen.add(i); used += len(seg)
        if used >= budget: break
    return "\n".join(segs[i][1] for i in sorted(chosen))

# ── citation: WHOLE ARTICLE by default ────────────────────────────────
# METEOR's F_mean = m / (0.9·|ref| + 0.1·|hyp|). Each matched token adds 1.0 to
# the numerator, each token of any kind adds 0.1 to the denominator — so at
# α=0.9 recall dominates. The rest of the SAME Điều is both the wording the
# reference quotes and CONTIGUOUS, so it lifts recall without paying the
# fragmentation penalty a second, different article would.
MAX_SIB = 12      # a real Điều is a handful of chunks, not hundreds

def article_text(pos, budget):
    k = chunk_keys[int(pos)]
    sibs = SIB.get(k, [])
    if not k[0] or not k[1] or len(sibs) > MAX_SIB:
        sibs = [int(pos)]                       # unparseable key → top-1 only
    parts, used = [], 0
    for i in sorted(sibs):                      # document order: LCS depends on it
        c = str(chunks.iloc[i]["content"])
        if used + len(c) > budget and parts:
            parts.append(c[:max(0, budget-used)]); break
        parts.append(c); used += len(c)
        if used >= budget: break
    return "\n".join(p for p in parts if p)

# ── chunk scaffolding: the chunker wrote machine tags INTO the text ───
# Real chunk content looks like this:
#     [DOCUMENT] Quyet-dinh-405-QD-BNV-2021-phe-duyet-Dieu-le-...-468351
#     [ARTICLE]  Điều 11. Thủ tục,
#     [CLAUSE]   1.
#     <the actual Vietnamese legal text>
# and the [DOCUMENT]/[ARTICLE] pair REPEATS for every sibling chunk, so a
# whole-article citation carries it ~4×. Measured on the 0.5224 submission:
# 54 tokens per answer of scaffolding.
#
# The slug is an unaccented filename; a reference answer writes "Quyết định
# 405/QĐ-BNV" WITH diacritics, so those tokens match nothing and are pure
# denominator. The [ARTICLE] title IS real Vietnamese and can match — but only
# once, because METEOR aligns each reference token a single time, so repeats 2-4
# are also pure denominator. Cell 10c measures this against val references
# instead of assuming it.
STRIP_SCAFFOLD = True

_SC_DOC  = re.compile(r"^\s*\[DOCUMENT\]")
_SC_ART  = re.compile(r"^\s*\[ARTICLE\]\s*")
_SC_CLA0 = re.compile(r"^\s*\[CLAUSE\]\s*\d*\.?\s*$")     # bare marker, no text
_SC_CLA  = re.compile(r"^\s*\[CLAUSE\]\s*\d*\.?\s*")      # marker + text
_SC_SLUG = re.compile(r"\b[A-Za-z0-9]+(?:-[A-Za-z0-9]+){3,}\b")

def strip_scaffold(text, keep_first_article=True):
    """Remove chunker tags. Keeps the FIRST [ARTICLE] title (real text, may
    match) and drops its repeats, every [DOCUMENT] slug line, and bare [CLAUSE]
    markers. Deletion only — never slices inside a line, so it cannot truncate
    a number the way an earlier regex turned '800.000 đồng' into '000 đồng'."""
    if not STRIP_SCAFFOLD: return text
    out, seen = [], not keep_first_article
    for line in str(text).split("\n"):
        if _SC_DOC.match(line) or _SC_CLA0.match(line): continue
        if _SC_ART.match(line):
            t = _SC_ART.sub("", line).strip()
            if seen or not t: continue
            seen = True; out.append(t); continue
        line = _SC_CLA.sub("", line)
        if line.strip(): out.append(line)
    return re.sub(r"\n{3,}", "\n\n", "\n".join(out)).strip()

def clean_header(h):
    """The 'Căn cứ …' line ends in the same slug. Drop it, keep the Điều/khoản.

    Falls back to the ORIGINAL header when stripping would empty it. A chunk
    with no parseable Điều/khoản has a header that is ONLY the slug, and the
    last run shipped 61 answers reading 'Căn cứ :' with nothing after the colon
    — worse than the slug it replaced, because a bare dangling colon tells the
    reader (and the grader) nothing at all."""
    if not STRIP_SCAFFOLD: return h
    c = _SC_SLUG.sub("", str(h))
    c = re.sub(r"\s*[—–-]\s*(?=:|$)", "", c)
    c = re.sub(r"[ \t]{2,}", " ", c).strip(" -—–,:")
    return c if c else str(h).strip()

# ── the dedup unit splitter, shared by cells 10c, 10d and 12 ──────────
# Vietnamese statutes number sub-points with LOWERCASE letters — a) b) c) đ) —
# so a capital-only sentence splitter swallows a whole run of them as ONE unit,
# and a run that is half-duplicated survives a duplication threshold it should
# have failed. Measured cost of getting this wrong: 0.0025 METEOR.
_UNIT = re.compile(
    r"(?<=[.;:])\s+(?="
    r"[A-ZÐĐÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚĂĨŨƠƯ]"     # capitalised sentence
    r"|[a-zđ]\)"                       # a)  b)  đ)
    r"|\d{1,2}[.)]\s"                 # 1.  2)
    r"|[-+\u2022*]\s"                 # bullets
    r")")

def split_units(text):
    """Blank-line blocks, then sentence/sub-point units inside each. Contiguity
    matters — METEOR's fragmentation penalty is 1 − 0.5·(chunks/matches)³ — so
    units stay clause-sized rather than being shredded to tokens."""
    for block in re.split(r"\n\s*\n", str(text)):
        block = block.strip()
        if not block: continue
        for s in _UNIT.split(block):
            s = s.strip()
            if s: yield s

def build_citation(qid, ctx, mode=None, budget=None, n_chunks=None):
    mode   = mode     or cfg.cite_mode
    budget = budget   if budget   is not None else cfg.cite_budget
    n      = n_chunks if n_chunks is not None else cfg.cite_n_chunks
    if budget <= 0 or n <= 0: return ""
    per, blocks = max(200, budget // max(1, n)), []
    for p in [int(x) for x in ctx.at[qid, "ctx_positions"][:n]]:
        body = (article_text(p, per) if mode == "article"
                else str(chunks.iloc[p]["content"])[:per] if mode == "chunk"
                else sniper_extract_clauses(chunks.iloc[p]["content"],
                                            ctx.at[qid,"question"], per))
        body = strip_scaffold(body)
        if body.strip():
            blocks.append(f"Căn cứ {clean_header(chunk_header(p))}:\n{body}")
    return "\n\n".join(blocks)

_CLEAN = [(re.compile(r"<\|im_(?:start|end)\|>.*?$", re.S), ""),
          (re.compile(r"^\s*(?:assistant|system|user)\s*[:：]\s*", re.I), ""),
          (re.compile(r"^\s*```[a-zA-Z]*\s*|\s*```\s*$"), ""),
          (re.compile(r"^\s*(?:Trả lời|Câu trả lời)\s*[:：]\s*"), ""),
          (re.compile(r"\n{3,}"), "\n\n"), (re.compile(r"[ \t]{2,}"), " ")]
# Deletes ONLY the preamble phrase plus one trailing punctuation mark.
# The previous version ended with [^.\n]*[.\n] — "run to the first period" —
# which is catastrophic in Vietnamese, where the period is the THOUSANDS
# SEPARATOR: "Dựa trên tài liệu được cung cấp, mức phạt là 800.000 đồng."
# became "000 đồng.". No wildcard here, so it cannot reach into content.
_PRE = re.compile(r"^\s*(?:Dựa\s+(?:trên|vào)|Căn\s+cứ\s+vào)\s+(?:các\s+)?"
                  r"(?:tài liệu|thông tin|ngữ cảnh|trích đoạn|văn bản)"
                  r"(?:\s+(?:được\s+cung\s+cấp|trên|sau))?\s*[,.:;]?\s*", re.I)
# ── decoding loops: collapse_loops (v11 lives HERE, not in cell 12) ────
# Defined next to clean_answer and called INSIDE it, so every val lab (10, 10b,
# 10c, 10d, 10e) measures exactly the text cell 12 ships. v10 defined it in
# cell 12, where a def dropped into the emission loop truncated the loop body.
COLLAPSE_LOOPS = True
_LOOP_TOK = re.compile(r"\n|[^\s]+")


def collapse_loops(text, min_p=3, max_p=30, min_copies=3):
    """Collapse an immediately repeated block of whole tokens — any period from
    min_p to max_p — down to ONE copy. `min_copies` or more back-to-back copies
    is a decoding loop; two is a refrain and is left alone.

    Why every period: the stutters in the 0.5486 file looped on 5-, 7- and
    9-token phrases ("vạch chỉ dẫn, vạch chỉ dẫn, …" ×235); a fixed-width
    detector misses most of them. Why it is safe: under METEOR's 1-to-1
    alignment a repeat matches nothing and still adds +0.1 to the denominator.

    Whole tokens only — no regex reaches inside one, so "139/2007/NĐ-CP"
    cannot become "139/2007/". Newlines are kept as tokens, so an answer with a
    loop keeps its line structure. Text with no loop is returned untouched,
    byte for byte."""
    s = str(text)
    t = _LOOP_TOK.findall(s)
    n = len(t)
    if n < min_p * min_copies:
        return s
    out, i, changed = [], 0, False
    while i < n:
        hit = False
        for p in range(min_p, min(max_p, (n - i) // 2) + 1):
            if t[i:i + p] != t[i + p:i + 2 * p]:
                continue
            j = i + p
            while j + p <= n and t[j:j + p] == t[i:i + p]:
                j += p
            if j - i >= min_copies * p:
                out.extend(t[i:i + p]); i = j; hit = changed = True
                break
        if not hit:
            out.append(t[i]); i += 1
    if not changed:
        return s
    res = []
    for tok in out:
        if tok == "\n":
            res.append("\n")
        else:
            if res and res[-1] != "\n":
                res.append(" ")
            res.append(tok)
    return "".join(res).strip()


def clean_answer(s):
    """Deleting k tokens only pays if their overlap with the reference is BELOW
    0.1×F_mean (≈5%) — a very low bar, so aggressive cleaning loses. These
    target ~0% text: chat scaffolding, fences, prompt echo, and decoding loops
    (a repeat matches nothing under 1-to-1 alignment). NOT 'Căn cứ…', which is
    high-density legal register the references use themselves."""
    s = unicodedata.normalize("NFC", str(s or ""))
    s = _PRE.sub("", s)
    for p, r in _CLEAN: s = p.sub(r, s)
    s = s.strip()
    return collapse_loops(s) if COLLAPSE_LOOPS else s

def finalize(raw, qid, ctx):
    a = clean_answer(raw)
    c = build_citation(qid, ctx)
    return (a + "\n\nTrích dẫn quy định:\n" + c).strip() if c else a

# ── metrics (NLTK METEOR minus the English-only stem/synonym stages, which
#    fire on ZERO Vietnamese tokens — identical here, and no nltk download
#    that can fail mid-run) ──────────────────────────────────────────────
def vi_tokens(s): return unicodedata.normalize("NFC", str(s).lower()).split()
def _align(h, r):
    he, rr, m = list(enumerate(h)), list(enumerate(r)), []
    for i in range(len(he)-1,-1,-1):
        for j in range(len(rr)-1,-1,-1):
            if he[i][1]==rr[j][1]: m.append((he[i][0],rr[j][0])); he.pop(i); rr.pop(j); break
    return sorted(m)
def _nchunks(m):
    if not m: return 0
    c = 1
    for i in range(len(m)-1):
        if not (m[i+1][0]==m[i][0]+1 and m[i+1][1]==m[i][1]+1): c += 1
    return c
def meteor(hyp, ref, alpha=0.9, beta=3.0, gamma=0.5):
    h, r = vi_tokens(hyp), vi_tokens(ref)
    if not h or not r: return 0.0
    m = _align(h, r); mm = len(m)
    if not mm: return 0.0
    P, R = mm/len(h), mm/len(r)
    return (P*R/(alpha*P+(1-alpha)*R)) * (1 - gamma*(_nchunks(m)/mm)**beta)
def rouge_l(hyp, ref):
    h, r = vi_tokens(hyp), vi_tokens(ref)
    if not h or not r: return 0.0
    prev = [0]*(len(r)+1)
    for i in range(1, len(h)+1):
        cur = [0]*(len(r)+1)
        for j in range(1, len(r)+1):
            cur[j] = prev[j-1]+1 if h[i-1]==r[j-1] else max(prev[j], cur[j-1])
        prev = cur
    p, rc = prev[-1]/len(h), prev[-1]/len(r)
    return 0.0 if p+rc == 0 else 2*p*rc/(p+rc)

@torch.inference_mode()
def generate_answers(prompts, max_new_tokens=None, batch_size=None):
    """Greedy, length-sorted, OOM-adaptive, ORDER-PRESERVING.

    · repetition_penalty stays 1.0: HF gathers the penalty over input_ids — the
      FULL sequence, prompt included — and the prompt IS the retrieved statute,
      so >1.0 suppresses exactly what we want quoted verbatim.
    · batches are sorted longest-first so each batch pads to a similar width,
      and so an OOM surfaces on batch 1 rather than 40 minutes in.
    · on OOM the batch halves and retries; results are written back by original
      index, so callers can still zip() the output against their input list.
    """
    mnt = max_new_tokens or cfg.max_new_tokens
    bs  = int(batch_size or PROFILE["gen_batch"])
    order = sorted(range(len(prompts)), key=lambda i: -len(prompts[i]))
    out, i, oom = [None]*len(prompts), 0, 0
    pbar = tqdm(total=len(prompts), desc="generate", leave=False)
    while i < len(order):
        idx = order[i:i+bs]
        try:
            enc = tokenizer([prompts[j] for j in idx], return_tensors="pt", padding=True,
                            truncation=True, max_length=cfg.max_seq_len).to(model.device)
            gen = model.generate(**enc, do_sample=False, max_new_tokens=mnt,
                                 repetition_penalty=cfg.repetition_penalty,
                                 pad_token_id=tokenizer.pad_token_id,
                                 eos_token_id=tokenizer.eos_token_id)
            dec = tokenizer.batch_decode(gen[:, enc.input_ids.shape[1]:],
                                         skip_special_tokens=True)
            for j, t in zip(idx, dec): out[j] = t.strip()
            del enc, gen
            i += len(idx); pbar.update(len(idx))
        except torch.cuda.OutOfMemoryError:
            enc = gen = None            # drop the partial tensors before collecting
            gc.collect(); torch.cuda.empty_cache()
            oom += 1
            if bs == 1:
                pbar.close()
                raise RuntimeError(
                    f"OOM even at batch size 1 after {oom} reductions. The prompt is "
                    f"{max(len(prompts[j]) for j in idx)} chars; lower cfg.max_seq_len or "
                    f"cfg.top_k_context, or restart the runtime to clear stranded memory.")
            bs = max(1, bs // 2)
            print(f"  OOM → batch {bs*2} → {bs}, retrying")
    pbar.close()
    if oom: print(f"  completed after {oom} batch reduction(s); final batch size {bs}")
    assert all(o is not None for o in out), "some prompts produced no output"
    return out

# ── v11: raw-answer caches keyed by the PROMPT, not the question id ────
# val_raw_cache.jsonl and gen_raw_v6.jsonl stored answers by qa_id alone. Change
# retrieval, the prompt or the adapter and cells 10/11 still found "an answer
# for this id" and reused it — the 0.5495 run generated nothing, so no answer in
# it saw the lexref fix. Every record now carries a hash of the exact prompt
# plus the adapter weights and decoding settings, and is reused only while that
# hash matches.
TRUST_LEGACY_CACHE = True    # reuse a v10 answer ONLY if its prompt is provably identical

def _file_sig(path, span=1 << 20):
    """Size + first/last MiB: tells two adapters apart without hashing 100+ MB."""
    try:
        n = os.path.getsize(path)
        with open(path, "rb") as f:
            h = hashlib.sha1(f.read(span))
            if n > span:
                f.seek(max(span, n - span)); h.update(f.read(span))
        return f"{n}:{h.hexdigest()[:12]}"
    except OSError:
        return "missing"

def gen_signature():
    """Everything besides the prompt text that decides what greedy decoding emits."""
    ad = str(globals().get("ADAPTER_IN_USE") or "")
    w = next((os.path.join(ad, f) for f in ("adapter_model.safetensors", "adapter_model.bin")
              if ad and os.path.exists(os.path.join(ad, f))), None)
    return "|".join([cfg.llm_id, _file_sig(w) if w else "no-adapter", str(DTYPE),
                     f"mnt={cfg.max_new_tokens}", f"rep={cfg.repetition_penalty}",
                     f"msl={cfg.max_seq_len}", "greedy"])

def prompt_fp(prompt, sig):
    return hashlib.sha1(f"{sig}\n{prompt}".encode("utf-8")).hexdigest()[:16]

def _read_jsonl(path):
    out = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            try:
                out.append(json.loads(line))
            except json.JSONDecodeError:
                pass                                  # a line cut short by a disconnect
    return out

def load_raw_cache(path, prompts, legacy=None):
    """prompts {qid: prompt} → (valid {qid: raw}, {qid: fp}, stats).

    A record in `path` is valid iff its fp equals today's. `legacy` is
    (v10 jsonl, v10 context parquet): an id-keyed v10 answer is accepted only if
    the prompt REBUILT from the contexts it was generated with — in the original
    format every v10 answer used — hashes identically to today's prompt.
    Accepted ones are written into `path` with their fp, so this runs once."""
    sig = gen_signature()
    fps = {q: prompt_fp(p, sig) for q, p in prompts.items()}
    have, stale, by_fp = {}, set(), {}
    if os.path.exists(path):
        for r in _read_jsonl(path):
            if r.get("raw") is not None and r.get("fp"):
                by_fp[r["fp"]] = r["raw"]              # v13: every answer, by its exact prompt
            q = str(r.get("qa_id"))
            if q not in fps or r.get("raw") is None:
                continue
            if r.get("fp") == fps[q]:
                have[q] = r["raw"]
            else:
                stale.add(q)
    # v13: the same PROMPT under another id — a public question repeated in the
    # private set, or a re-numbered one — has the same greedy answer. fp covers the
    # prompt, the adapter and the decoding settings, so this is as safe as an id match.
    n_fp = 0
    for q, f in fps.items():
        if q not in have and f in by_fp:
            have[q] = by_fp[f]; n_fp += 1
    n_leg = n_rej = 0
    if legacy and TRUST_LEGACY_CACHE and all(os.path.exists(x) for x in legacy):
        old = {}
        for r in _read_jsonl(legacy[0]):
            if r.get("raw") is not None:
                old[str(r["qa_id"])] = r["raw"]          # last record wins, as v10 read it
        oc = pd.read_parquet(legacy[1]); oc["qa_id"] = oc["qa_id"].astype(str)
        oc = oc.set_index("qa_id")
        add = []
        for q in fps:
            if q in have or q not in old or q not in oc.index:
                continue
            lp = to_infer_text(oc.at[q, "question"], oc.at[q, "ctx_positions"], strip=False)
            if prompt_fp(lp, sig) == fps[q]:
                have[q] = old[q]; add.append(q)
            else:
                n_rej += 1
        if add:
            with open(path, "a", encoding="utf-8") as f:
                for q in add:
                    f.write(json.dumps({"qa_id": q, "raw": have[q], "fp": fps[q], "src": "v10"},
                                       ensure_ascii=False) + "\n")
        n_leg = len(add)
    return have, fps, {"reused": len(have), "legacy": n_leg, "legacy_rejected": n_rej, "by_prompt": n_fp,
                       "stale": len(stale - set(have)), "sig": sig}

print("prompt, citation, metrics and generation helpers ready")

# ── smoke test: catch a broken adapter in 30 seconds, not after 90 minutes ──
_sm = generate_answers([to_infer_text(val_ctx.at[q, "question"],
                                      val_ctx.at[q, "ctx_positions"])
                        for q in list(val_ctx.index)[:2]], max_new_tokens=160)
for _t in _sm:
    print(f"\n  sample ({len(vi_tokens(_t))} tok): {_t[:200]}…")
assert all(len(vi_tokens(t)) > 5 for t in _sm), (
    "the adapter produces near-empty output — do NOT run inference with it")
print("\n  ✓ adapter generates sane text")
ADAPTER_IN_USE = globals().get("ADAPTER_IN_USE") or ADAPTER_DIR   # set by CELL T4 — v10
                                                                  # overwrote it here


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 10 — Validation gate + citation-mode sweep  (measure before you spend)
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['val_ctx', 'generate_answers', 'build_citation', 'qa_by_id', 'meteor'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 7 and 9 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# Generates the val answers ONCE and caches the RAW text, then sweeps citation
# modes over the cache for free. Never spend a submission on an unmeasured
# configuration.
RUN_GATE  = True
_VCACHE   = W("val_raw_v11.jsonl")                               # prompt-keyed (v11)
_VLEGACY  = (W("val_raw_cache.jsonl"),                           # v10: keyed by id only
             W(f"val_ctx_k{cfg.top_k_context}.parquet"))

if RUN_GATE and len(val_ctx):
    _vids = [q for q in val_ctx.index if q in qa_by_id.index]
    _vpr  = {q: to_infer_text(val_ctx.at[q, "question"], val_ctx.at[q, "ctx_positions"])
             for q in _vids}
    _raw, _vfp, _vst = load_raw_cache(_VCACHE, _vpr, legacy=_VLEGACY)
    _todo = [q for q in _vids if q not in _raw]
    print(f"val answers: {_vst['reused']} reusable ({_vst['legacy']} carried over from v10 "
          f"with a byte-identical prompt) | {len(_todo)} to generate")
    if _todo:
        _t0 = time.time()
        _h = generate_answers([_vpr[q] for q in _todo])
        with open(_VCACHE, "a", encoding="utf-8") as f:
            for q, a in zip(_todo, _h):
                _raw[q] = a
                f.write(json.dumps({"qa_id": q, "raw": a, "fp": _vfp[q]},
                                   ensure_ascii=False) + "\n")
        print(f"  {(time.time()-_t0)/60:.1f} min")

    _refs = {q: str(qa_by_id.at[q,"answer"]) for q in _vids}
    _lr = np.array([len(vi_tokens(_refs[q])) for q in _vids])
    _lh = np.array([len(vi_tokens(_raw[q]))  for q in _vids])
    print(f"\nLENGTH  reference mean {_lr.mean():.0f} p90 {np.percentile(_lr,90):.0f} | "
          f"raw output mean {_lh.mean():.0f} p90 {np.percentile(_lh,90):.0f}")

    print(f"\n{'citation mode':<34} {'METEOR':>8} {'ROUGE-L':>8} {'len':>7}")
    print("-"*62)
    _rows = []
    for _mode, _bud in (("sniper",3000), ("chunk",4000),
                        ("article",2500), ("article",4000), ("article",6000), ("article",9000)):
        _out = [(clean_answer(_raw[q]) + "\n\nTrích dẫn quy định:\n"
                 + build_citation(q, val_ctx, _mode, _bud, 1)).strip() for q in _vids]
        _M  = float(np.mean([meteor(o,_refs[q])  for o,q in zip(_out,_vids)]))
        _RL = float(np.mean([rouge_l(o,_refs[q]) for o,q in zip(_out,_vids)]))
        _L  = float(np.mean([len(vi_tokens(o)) for o in _out]))
        _rows.append((_mode,_bud,_M,_RL,_L))
        _tag = "  ← cfg default" if (_mode==cfg.cite_mode and _bud==cfg.cite_budget) else ""
        print(f"  {_mode+' @'+str(_bud):<32} {_M:>8.4f} {_RL:>8.4f} {_L:>7.0f}{_tag}")
    _b = max(_rows, key=lambda r: r[2])
    print("-"*62)
    print(f"  best: {_b[0]} @{_b[1]}  →  METEOR {_b[2]:.4f}")
    # Calibration from two prior submissions: val runs ~0.020 optimistic.
    print(f"  projected Codabench ≈ {_b[2]-globals().get('CB_GAP',0.002):.3f}")
    if (_b[0], _b[1]) != (cfg.cite_mode, cfg.cite_budget):
        print(f"\n  ⚠ cfg says {cfg.cite_mode}@{cfg.cite_budget} but {_b[0]}@{_b[1]} scores "
              f"higher.\n    Set cfg.cite_mode/cfg.cite_budget before cell 11.")
    VAL_BEST = _b
else:
    VAL_BEST = None; print("gate skipped")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 10b — CITATION LAB: combinations never yet measured
#  Free — pure post-processing on cell 10's cached raw answers.
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['_raw', 'build_citation', 'val_ctx'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 10 (with RUN_GATE=True) first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# Earlier sweeps tested n_chunks with the SNIPER citation, and the whole-article
# citation with n_chunks=1. Whole-article × n_chunks=2 has never been run, and
# the reason n=2 hurt before (fragmentation from a different article's wording)
# may not apply once each block is a complete, contiguous Điều.
assert globals().get('_raw'), "run cell 10 first (RUN_GATE=True) — this reuses its val cache"
_ids  = [q for q in val_ctx.index if q in qa_by_id.index and q in _raw]
_ref  = {q: str(qa_by_id.at[q,"answer"]) for q in _ids}

def _ev(mode, bud, n, keep_raw=True):
    outs = []
    for q in _ids:
        c = build_citation(q, val_ctx, mode, bud, n)
        outs.append(((clean_answer(_raw[q]) + "\n\nTrích dẫn quy định:\n" + c).strip()
                     if keep_raw else c) if c else clean_answer(_raw[q]))
    M  = float(np.mean([meteor(o,_ref[q])  for o,q in zip(outs,_ids)]))
    RL = float(np.mean([rouge_l(o,_ref[q]) for o,q in zip(outs,_ids)]))
    L  = float(np.mean([len(vi_tokens(o)) for o in outs]))
    return M, RL, L

print(f"CITATION SWEEP  (n={len(_ids)} val questions)")
print(f"  {'mode':<10} {'budget':>7} {'n':>3} {'METEOR':>8} {'ROUGE-L':>8} {'len':>7}")
print("  " + "-"*50)
rows=[]
for n in (1, 2, 3):
    for bud in (4000, 6000, 9000, 12000):
        if n == 1 and bud > 9000: continue
        M, RL, L = _ev("article", bud, n)
        rows.append(("article", bud, n, M, RL, L))
        tag = "  ← current" if (n==cfg.cite_n_chunks and bud==cfg.cite_budget) else ""
        print(f"  {'article':<10} {bud:>7} {n:>3} {M:>8.4f} {RL:>8.4f} {L:>7.0f}{tag}")
_b = max(rows, key=lambda r: r[3])
_cur = next((r for r in rows if r[1]==cfg.cite_budget and r[2]==cfg.cite_n_chunks), None)
print("  " + "-"*50)
print(f"  best: article @{_b[1]} n={_b[2]} → {_b[3]:.4f}  (proj. Codabench {_b[3]-globals().get('CB_GAP',0.002):.3f})")
if _cur: print(f"  vs current: {_b[3]-_cur[3]:+.4f}")

# ── what max_new_tokens=320 would actually cost ───────────────────────
print(f"\nLENGTH CALIBRATION (Upgrade C)")
_sub = [len(tokenizer(_raw[q]).input_ids) for q in _ids]
print(f"  raw prose, SUBWORD tokens: mean {np.mean(_sub):.0f} median {np.median(_sub):.0f} "
      f"p90 {np.percentile(_sub,90):.0f}")
print(f"  {'cap':>6} {'answers truncated':>18} {'METEOR':>9}")
for cap in (320, 512, 768, 1400):
    _t = sum(1 for s in _sub if s > cap)
    _o = []
    for q in _ids:
        _ids_ = tokenizer(_raw[q]).input_ids[:cap]
        _p = tokenizer.decode(_ids_, skip_special_tokens=True)
        _c = build_citation(q, val_ctx, cfg.cite_mode, cfg.cite_budget, cfg.cite_n_chunks)
        _o.append((clean_answer(_p) + "\n\nTrích dẫn quy định:\n" + _c).strip())
    _M = float(np.mean([meteor(o,_ref[q]) for o,q in zip(_o,_ids)]))
    print(f"  {cap:>6} {f'{_t}/{len(_ids)}':>18} {_M:>9.4f}"
          + ("   ← Upgrade C proposes this" if cap==320 else ""))

# ── known-QA coverage: exact matches score ~1.0 and are free ──────────
def _nq(s): return re.sub(r"\s+"," ",unicodedata.normalize("NFC",str(s)).lower().strip())
_k = {_nq(k): v for k, v in known_by_q.items()}
_hitv = [q for q in _ids if _nq(val_ctx.at[q,"question"]) in _k]
_hitt = sum(1 for q in test_ctx.index if _nq(test_ctx.at[q,"question"]) in _k)
print(f"\nKNOWN-QA COVERAGE")
print(f"  val : {len(_hitv)}/{len(_ids)}   test: {_hitt}/{len(test_ctx)} "
      f"({_hitt/max(1,len(test_ctx)):.1%})")
if _hitv:
    _km = float(np.mean([meteor(str(_k[_nq(val_ctx.at[q,'question'])]), _ref[q]) for q in _hitv]))
    print(f"  METEOR of the override answers on val: {_km:.4f}  "
          f"(should be ≈1.0 — both come from organiser data)")
    if _km < 0.9:
        print(f"  ⚠ below 0.9 — the lookup is matching the WRONG answers. Each bad override")
        print(f"    costs a near-perfect score, so verify before trusting it on test.")
print(f"  → each additional exact match is worth ~{(1.0-_b[3])/max(1,len(test_ctx)):.5f} "
      f"METEOR on the full test set")


# ══════════════════════════════════════════════════════════════════════
#  UPGRADE 7 — currency form: which way do the REFERENCES write it?
# ══════════════════════════════════════════════════════════════════════
# "5.000.000 đồng" and "5 triệu đồng" share no unigrams, so if the model
# writes one form and the reference the other, every amount is a miss.
# Normalising toward the WRONG form doubles the damage — so measure first.
_DIGIT = re.compile(r"\b\d{1,3}(?:\.\d{3})+\s*(?:đồng|vnđ|vnd)\b", re.I)
_WORD  = re.compile(r"\b\d+(?:[.,]\d+)?\s*(?:triệu|tỷ|nghìn|ngàn)\s*(?:đồng)?\b", re.I)

_rf = {q: str(qa_by_id.at[q,"answer"]) for q in _ids}
_d_ref = sum(len(_DIGIT.findall(v)) for v in _rf.values())
_w_ref = sum(len(_WORD.findall(v))  for v in _rf.values())
_d_hyp = sum(len(_DIGIT.findall(_raw[q])) for q in _ids)
_w_hyp = sum(len(_WORD.findall(_raw[q]))  for q in _ids)
print(f"\nCURRENCY FORM over {len(_ids)} val pairs")
print(f"  {'':<12} {'digit (5.000.000 đồng)':>24} {'word (5 triệu đồng)':>22}")
print(f"  {'references':<12} {_d_ref:>24} {_w_ref:>22}")
print(f"  {'model output':<12} {_d_hyp:>24} {_w_hyp:>22}")
_tot = _d_ref + _w_ref
if _tot < 20:
    print(f"  → only {_tot} currency mentions in the references; not worth normalising")
else:
    _pref = "digit" if _d_ref >= _w_ref else "word"
    print(f"  → references prefer the {_pref.upper()} form "
          f"({max(_d_ref,_w_ref)/_tot:.0%} of mentions)")
    _mis = (_w_hyp if _pref=="digit" else _d_hyp)
    print(f"  → model uses the other form {_mis} times ≈ {_mis/len(_ids):.2f} per answer")
    if _mis / max(1,len(_ids)) < 0.15:
        print(f"     below 0.15/answer — the upside is a few tokens. Skip it.")
    else:
        print(f"     worth converting; each conversion recovers ~2-4 matched unigrams")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 10c — DEDUPLICATION LAB   (free: post-processing on cell 10's cache)
# ══════════════════════════════════════════════════════════════════════
# WHY LENGTH IS NOT THE TARGET
#
# METEOR at α=0.9 is   M = (1 − pen) · m / (0.9·|ref| + 0.1·|hyp|).
# Appending a token that matches with probability ρ is profitable iff
#
#       ρ  >  0.1 · F_mean       ≈ 0.1 × 0.55 ≈ 5.5%
#
# A token only has to beat a 1-in-18 chance of matching to be worth keeping.
# So "get to 360 tokens" is the WRONG instruction — 360 is where rank 1 lands
# because their tokens are ~61% dense, not because short is good. Truncating
# your 906 tokens down to 360 cuts prose that is ~38% dense and would land near
# 0.44, well BELOW the 0.5224 you already have.
#
# What is actually free to cut is the subset whose density is ≈ 0:
#   · chunker scaffolding ([DOCUMENT] slug / repeated [ARTICLE] / [CLAUSE] n.)
#   · citation clauses the model's prose ALREADY quoted — METEOR aligns each
#     reference token once, so the repeat earns +0 numerator, +0.1 denominator.
#
# This cell MEASURES both against the val references rather than assuming.
_MISSING = [n for n in ['_raw','val_ctx','build_citation','meteor','rouge_l','vi_tokens',
                        'clean_answer','qa_by_id','split_units','strip_scaffold']
            if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS 9 and 10 first.")

from collections import Counter
import numpy as np, re

_ids = [q for q in val_ctx.index if q in qa_by_id.index and q in _raw]
_ref = {q: str(qa_by_id.at[q, "answer"]) for q in _ids}
print(f"deduplication lab over {len(_ids)} validation questions\n")


def dedup_citation(prose, cite, thr):
    """Drop a unit when >= thr of its tokens are already spoken for — by the
    prose, or by a unit kept earlier. Returns (kept_text, dropped_tokens)."""
    used, keep, dropped = Counter(vi_tokens(prose)), [], []
    first = True
    for u in split_units(cite):
        t = vi_tokens(u)
        if not t: continue
        dup = sum(1 for w in t if used[w] > 0)
        if first or dup / len(t) < thr:          # always keep the 'Căn cứ' header
            keep.append(u)
            for w in t: used[w] += 1
            first = False
        else:
            dropped.extend(t)
    return "\n".join(keep), dropped


def _cite(q):
    return build_citation(q, val_ctx, cfg.cite_mode, cfg.cite_budget, cfg.cite_n_chunks)

def _join(p, c):
    return (p + "\n\nTrích dẫn quy định:\n" + c).strip() if c.strip() else p

def _ev(fn, label, mark=""):
    outs = [fn(q) for q in _ids]
    M  = float(np.mean([meteor(o, _ref[q])  for o, q in zip(outs, _ids)]))
    RL = float(np.mean([rouge_l(o, _ref[q]) for o, q in zip(outs, _ids)]))
    L  = float(np.mean([len(vi_tokens(o)) for o in outs]))
    print(f"  {label:<40} {M:>7.4f} {RL:>8.4f} {L:>7.0f} {M-globals().get('CB_GAP',0.002):>10.4f}{mark}")
    return M, RL, L


# ══ A. DENSITY AUDIT — what is each part of the answer actually worth? ══
# Density = the share of a segment's tokens that the reference still has
# UNCONSUMED when that segment is reached. This is the ρ in the rule above,
# measured, per segment, in the order METEOR would consume them.
def density(seg_tokens, pool):
    h = 0
    for w in seg_tokens:
        if pool[w] > 0: pool[w] -= 1; h += 1
    return h / max(1, len(seg_tokens)), h

_rows = {"prose": [[], []], "scaffold": [[], []], "dup clauses": [[], []], "novel clauses": [[], []]}
for q in _ids:
    pool = Counter(vi_tokens(_ref[q]))
    p = clean_answer(_raw[q])
    d, h = density(vi_tokens(p), pool); _rows["prose"][0].append(d); _rows["prose"][1].append(len(vi_tokens(p)))
    # scaffolding = whatever strip_scaffold removes from the raw chunk text
    _sc = []
    for pos in [int(x) for x in val_ctx.at[q, "ctx_positions"][:cfg.cite_n_chunks]]:
        full = str(chunks.iloc[pos]["content"])
        kept = strip_scaffold(full)
        ft, kt = Counter(vi_tokens(full)), Counter(vi_tokens(kept))
        _sc.extend((ft - kt).elements())
    if _sc:
        d, h = density(_sc, pool); _rows["scaffold"][0].append(d); _rows["scaffold"][1].append(len(_sc))
    c = _cite(q)
    seen = Counter(vi_tokens(p)); dupt, novt = [], []
    first = True
    for u in split_units(c):
        t = vi_tokens(u)
        if not t: continue
        r = sum(1 for w in t if seen[w] > 0) / len(t)
        (novt if (first or r < 0.90) else dupt).extend(t)
        if first or r < 0.90:
            for w in t: seen[w] += 1
        first = False
    for nm, tt in (("dup clauses", dupt), ("novel clauses", novt)):
        if tt:
            d, h = density(tt, pool); _rows[nm][0].append(d); _rows[nm][1].append(len(tt))

_F = float(np.mean([meteor(_join(clean_answer(_raw[q]), _cite(q)), _ref[q]) for q in _ids]))
_BAR = 0.1 * _F
print("A. DENSITY AUDIT — share of each segment's tokens the reference still wants")
print(f"  {'segment':<24} {'tok/answer':>11} {'density ρ':>11}   verdict  (bar = 0.1·F = {_BAR:.3f})")
print("  " + "-" * 78)
for nm in ("prose", "scaffold", "dup clauses", "novel clauses"):
    if not _rows[nm][0]: continue
    d, L = float(np.mean(_rows[nm][0])), float(np.mean(_rows[nm][1]))
    print(f"  {nm:<24} {L:>11.0f} {d:>11.1%}   "
          + ("KEEP — pays for itself" if d > _BAR else "CUT — pure denominator"))
print("  " + "-" * 78)
print("  This is the whole argument. Cut what is below the bar; keep what is above,")
print("  however long it is. Do not chase a token count.\n")


# ══ B. THE SWEEP ═══════════════════════════════════════════════════════
print(f"  {'strategy':<40} {'METEOR':>7} {'ROUGE-L':>8} {'len':>7} {'proj. CB':>10}")
print("  " + "-" * 76)
res = {}
res["baseline"] = _ev(lambda q: _join(clean_answer(_raw[q]), _cite(q)),
                      "baseline (article @%d, scaffold-stripped)" % cfg.cite_budget)
_lost = {}
for t in (0.95, 0.90, 0.85, 0.80, 0.70, 0.60):
    _drop = []
    def _f(q, t=t, _drop=_drop):
        p = clean_answer(_raw[q]); c, dd = dedup_citation(p, _cite(q), t)
        _drop.append(dd); return _join(p, c)
    res[f"dedup{t}"] = _ev(_f, f"clause dedup, threshold {t:.2f}")
    _lost[t] = _drop
print("  " + "-" * 76)

# Length caps, shown ONLY so the failure mode is visible rather than argued.
for n in (150, 250, 360, 450):
    res[f"cap{n}"] = _ev(lambda q, n=n: " ".join(
        vi_tokens(_join(clean_answer(_raw[q]), _cite(q)))[:n]),
        f"blunt truncation to {n} tok", "   ← what a length target does")
print("  " + "-" * 76)

_b = max(res.items(), key=lambda kv: kv[1][0])
_cur = res["baseline"][0]
print(f"  best: {_b[0]}  →  val {_b[1][0]:.4f}  ({_b[1][0]-_cur:+.4f} vs baseline)")
print(f"        projected Codabench {_b[1][0]-globals().get('CB_GAP',0.002):.4f}"
      + ("   ← RANK 3 TERRITORY" if _b[1][0]-globals().get('CB_GAP',0.002) >= 0.5895 else "")
      + f"   |  mean length {_b[1][2]:.0f} tok")


# ══ C. WHAT DID THE WINNING THRESHOLD ACTUALLY THROW AWAY? ═════════════
# On val the references are known, so this is measured, not a risk band.
if _b[0].startswith("dedup"):
    _t = float(_b[0][5:])
    _hit = _tot = 0
    for q, dd in zip(_ids, _lost[_t]):
        pool = Counter(vi_tokens(_ref[q]))
        kept = vi_tokens(_join(clean_answer(_raw[q]), dedup_citation(clean_answer(_raw[q]), _cite(q), _t)[0]))
        for w in kept:
            if pool[w] > 0: pool[w] -= 1
        for w in dd:
            _tot += 1
            if pool[w] > 0: pool[w] -= 1; _hit += 1
    print(f"\nC. COST OF dedup@{_t:.2f}: dropped {_tot/len(_ids):.0f} tok/answer, of which "
          f"{_hit/max(1,_tot):.1%} would still have matched")
    print(f"   ({_hit/len(_ids):.1f} lost matches per answer, against the bar of {_BAR:.1%})")
    print("   " + ("✓ well below the bar — the cut is free"
                   if _hit/max(1,_tot) < _BAR else
                   "⚠ above the bar — raise the threshold"))
DEDUP_BEST = _b[0]
DEDUP_LAB  = {k: v[0] for k, v in res.items()}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 10d — SUPERVISED CLAUSE SELECTION BY MARGINAL YIELD   (CPU, ~2 min)
# ══════════════════════════════════════════════════════════════════════
# WHAT WAS WRONG WITH v10's VERSION — your own table proves it
#
#   supauto (the "marginal rule")   0.5488   844 tok
#   cell 10:  article @9000         0.5488   844 tok      ← identical
#
# The rule compared each clause's STANDALONE predicted density (mean 0.62)
# against a bar of 0.1·F ≈ 0.055. Nothing is ever below 0.055 standalone, so
# the bar never fired and the rule returned the whole 9,000-char article.
#
# The bar is only meaningful for MARGINAL density: how many NEW matches this
# clause adds given everything already emitted — the prose plus the clauses
# picked so far. METEOR aligns each reference token once, so a copy of a token
# the prose already used only matches if the reference holds MORE copies. How
# often that happens is a property of the references, so it is measured from
# them: S_t(e) = P(reference has > e copies of t | has t). Legal answers repeat
# register ('quy_định', 'theo', 'của') constantly — S stays high — and rarely
# repeat content words — S collapses. That is why cell 10c found 'duplicate'
# clauses still 31% dense, and why every blunt dedup threshold LOST.
#
#   marginal(clause) = ŷ(clause) × mean over its tokens of S_t(copies already out)
#
# ŷ comes from the ridge fit below; the selection is lazy greedy on marginal
# density (exact, because marginal density only falls as more is emitted), and
# stops when the best remaining clause is below the bar.
#
# WHAT TO EXPECT: on your data, cap-640 won by +0.0015. Selection is worth
# thousandths. The article it selects FROM is worth tenths — that is cell 6d.
_MISSING = [n for n in ['chunks','chunk_keys','SIB','labels_by_qid','qa_by_id','vi_tokens',
                        'meteor','rouge_l','val_ctx','_raw','build_citation','clean_answer',
                        'article_text','split_units','strip_scaffold','chunk_header',
                        'clean_header','RET_VAL','GEN_VAL']
            if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS 9 and 10 first.")
assert callable(W), ("W is not callable — an earlier cell rebound the Drive path "
                     "helper. Restart and re-run; do not proceed.")

import re, numpy as np
from collections import Counter

FIT_MAX_EXAMPLES = 3000
SEL_BUDGET_TOK   = 260        # a CAP, not a target — section C finds the real stop

def clauses_of(text):
    """Reuses cell 9's splitter so the lab and cell 12 cut identically. A lab
    that measures a different segmentation than production ships is a lab that
    measures nothing."""
    p = [x for x in split_units(strip_scaffold(str(text))) if x.strip()]
    return p or [str(text).strip()]

_AMT  = re.compile(r"\d{1,3}(?:\.\d{3})+")
_PEN  = re.compile(r"\bphạt|\bmức\b|\bxử phạt\b|\btước quyền\b|\btịch thu\b")
_PROC = re.compile(r"\bđiều kiện\b|\bhồ sơ\b|\bthủ tục\b|\bgiấy\b|\bđăng ký\b")
_COND = re.compile(r"\btrường hợp\b|\bkhông\b|\btrừ\b|\bnếu\b")
_XREF = re.compile(r"\btheo quy định tại\b|\bquy định tại\b|\bkhoản\b|\bđiểm\b")

def clause_features(cl, idx, n_cl, query):
    """Cheap, interpretable, and computable at inference time — no reference."""
    t   = vi_tokens(cl)
    qs  = set(vi_tokens(query))
    low = cl.lower()
    n   = max(1, len(t))
    return [
        1.0,
        idx / max(1, n_cl - 1),                            # relative position
        1.0 if idx == 0 else 0.0,                          # article/clause header
        min(len(t), 200) / 200.0,                          # length
        sum(1 for w in t if w in qs) / n,                  # query overlap (the OLD signal)
        1.0 if _AMT.search(cl) else 0.0,                   # a money amount
        1.0 if _PEN.search(low)  else 0.0,
        1.0 if _PROC.search(low) else 0.0,
        1.0 if _COND.search(low) else 0.0,
        1.0 if _XREF.search(low) else 0.0,
        sum(1 for w in t if w.isdigit()) / n,              # numeric density
        1.0 if re.match(r"^\s*[a-zđ]\)", cl) else 0.0,     # is a lettered sub-point
        1.0 if re.match(r"^\s*\d{1,2}\.", cl) else 0.0,    # is a numbered khoản
    ]
_FEAT_NAMES = ["bias","rel_pos","is_first","length","query_overlap","has_amount",
               "penalty_words","procedure_words","condition_words","cross_ref",
               "numeric_density","is_letter_point","is_numbered_clause"]

def clause_yield(cl, ref_counter):
    """Marginal reference coverage: how many of this clause's tokens the
    reference still has UNCONSUMED. Exactly what METEOR's 1-to-1 alignment
    rewards — not raw string similarity."""
    t = vi_tokens(cl)
    if not t: return 0.0
    pool, hit = ref_counter.copy(), 0
    for w in t:
        if pool[w] > 0: pool[w] -= 1; hit += 1
    return hit / len(t)


# ── marginal-yield selection core: pure functions ─────────────────────
import heapq
from collections import Counter, defaultdict
import numpy as np

def ridge_fit(X, y, lam=1.0):
    """Closed-form ridge, no sklearn: w = (XᵀX + λI)⁻¹ Xᵀy.
    The bias column is penalised with the rest — harmless at λ=1 with ~45k rows,
    and it keeps the solve a single call."""
    X, y = np.asarray(X, dtype=np.float64), np.asarray(y, dtype=np.float64)
    w = np.linalg.solve(X.T @ X + lam * np.eye(X.shape[1]), X.T @ y)
    r2 = 1 - ((y - X @ w) ** 2).sum() / max(1e-12, ((y - y.mean()) ** 2).sum())
    return w, float(r2)


class RepeatSurvival:
    """S_t(e) = P(reference contains token t MORE than e times | contains it).

    This is what turns a clause's standalone yield into its MARGINAL yield.
    METEOR aligns each reference token once, so once the prose has emitted
    'quy_định' e times, another copy only matches if the reference holds more
    than e of them. Legal answers repeat function words and legal register
    heavily, so S stays high for those and falls fast for content words — which
    is why cell 10c measured 'duplicate' citation clauses at 31% density rather
    than ~0, and why every blunt dedup threshold lost."""
    def __init__(self, refs_tokens, emax=6, alpha=20.0):
        self.emax, self.alpha = emax, alpha
        gt, n = defaultdict(lambda: np.zeros(emax + 1)), Counter()
        for toks in refs_tokens:
            for t, k in Counter(toks).items():
                n[t] += 1
                gt[t][:min(k, emax + 1)] += 1          # count > e  ⇔  e < k
        tot_n = max(1, sum(n.values()))
        tot_gt = np.zeros(emax + 1)
        for v in gt.values(): tot_gt += v
        self.glob = tot_gt / tot_n                     # S_glob(e); S_glob(0) = 1
        self.tok = {t: (gt[t] + alpha * self.glob) / (n[t] + alpha) for t in n}

    def __call__(self, t, e):
        e = min(int(e), self.emax)
        v = self.tok.get(t)
        return float(v[e]) if v is not None else float(self.glob[e])


def marginal_density(toks, emitted, yhat, S):
    """Predicted matches per token if this clause is appended NOW:
    ŷ (standalone density) × mean over its tokens of S_t(copies already out)."""
    if not toks: return 0.0
    loc, s = Counter(), 0.0
    for w in toks:
        s += S(w, emitted[w] + loc[w]); loc[w] += 1
    return yhat * s / len(toks)


def select_clauses(clause_toks, yhats, prose_toks, S, budget, bar):
    """Lazy greedy by marginal density. Exact, because marginal density can only
    FALL as more is emitted (S is non-increasing in e), so a stale heap score is
    an upper bound — recompute the top, accept it if it still leads.
    Stops when the best remaining clause is below `bar` (None = fill budget).
    Returns chosen indices in DOCUMENT order, plus how many the bar rejected."""
    emitted = Counter(prose_toks)
    heap = [(-marginal_density(t, emitted, y, S), j) for j, (t, y) in enumerate(zip(clause_toks, yhats))]
    heapq.heapify(heap)
    keep, used, stopped_by_bar = [], 0, 0
    while heap:
        neg, j = heapq.heappop(heap)
        cur = marginal_density(clause_toks[j], emitted, yhats[j], S)
        if heap and cur < -heap[0][0] - 1e-12:         # stale: re-queue with fresh score
            heapq.heappush(heap, (-cur, j)); continue
        if bar is not None and cur < bar and keep:
            stopped_by_bar = 1 + len(heap); break
        n = len(clause_toks[j])
        if used + n > budget and keep:
            continue
        keep.append(j); used += n
        for w in clause_toks[j]: emitted[w] += 1
    return sorted(keep), stopped_by_bar

# ══ A. FIT — ridge on standalone yield, survival on reference repeats ══
# Val questions are EXCLUDED: section C scores the selector on GEN_VAL, and a
# scorer fitted on those very references would be graded on its training set.
# (v10 fitted on every labelled question, val included.)
_HOLD = {str(q) for q in RET_VAL} | {str(q) for q in GEN_VAL} | {str(q) for q in val_ctx.index}
_X, _y, _refs = [], [], []
_cands = [q for q in labels_by_qid.index if q in qa_by_id.index and str(q) not in _HOLD]
print(f"fitting on up to {FIT_MAX_EXAMPLES:,} gold-labelled examples "
      f"({len(_HOLD):,} val questions held out)")
_used = 0
for q in _cands:
    if _used >= FIT_MAX_EXAMPLES: break
    row = labels_by_qid.loc[q]
    if isinstance(row, pd.DataFrame): row = row.iloc[0]
    gp = sorted(row["gold_positions"])
    if not gp: continue
    ref = str(qa_by_id.at[q, "answer"])
    if not ref.strip(): continue
    _rt = set(vi_tokens(ref))                     # the gold chunk the answer draws from
    p0 = max(gp[:400], key=lambda p: len(_rt & set(vi_tokens(chunks.iloc[p]["content"]))))
    cls = clauses_of(article_text(p0, 6000))
    if len(cls) < 2: continue
    rc    = Counter(vi_tokens(ref))
    query = str(row.get("query") or qa_by_id.at[q, "question"])
    for j, cl in enumerate(cls):
        _X.append(clause_features(cl, j, len(cls), query))
        _y.append(clause_yield(cl, rc))
    _refs.append(vi_tokens(ref))
    _used += 1
print(f"  {_used:,} articles → {len(_y):,} clauses | mean standalone yield {np.mean(_y):.3f}")
assert len(_y) > 200, "not enough labelled clauses to fit a scorer"

# NAMED RIDGE_W, NOT W: `W` is the Drive path helper; binding an array to it
# made cell 12 die on W("submission.json") in v8.
RIDGE_W, _r2 = ridge_fit(_X, _y, lam=1.0)
print(f"  ridge R² = {_r2:.3f}")
print(f"\n  {'feature':<20} {'weight':>9}")
for n_, w_ in sorted(zip(_FEAT_NAMES, RIDGE_W), key=lambda kv: -abs(kv[1])):
    print(f"  {n_:<20} {w_:>+9.4f}")
assert callable(W), "W was rebound during the fit — abort"

REPEAT_S = RepeatSurvival(_refs)
print(f"\n  reference repeat survival  S(e) = P(> e copies | ≥ 1), all tokens:")
print("   " + "  ".join(f"e={e}:{REPEAT_S.glob[e]:.2f}" for e in range(5)))
for _t in ("quy_định", "theo", "của", "phạt", "đồng"):
    if _t in REPEAT_S.tok:
        print(f"   {_t:<10} " + "  ".join(f"{REPEAT_S(_t, e):.2f}" for e in range(4)))


def clause_score(cl, idx, n_cl, query):
    """Predicted standalone density; clamped at 0 (ridge can extrapolate below)."""
    return max(0.0, float(np.dot(clause_features(cl, idx, n_cl, query), RIDGE_W)))


# ══ B. SELECT ══════════════════════════════════════════════════════════
def supervised_citation(qid, ctx, budget=None, bar=None, prose="", return_stats=False):
    """Clauses of the top-1 article, chosen by MARGINAL yield given `prose`,
    emitted in document order.
      budget : token cap (a ceiling, not a target)
      bar    : stop when the best remaining clause adds fewer than `bar`
               expected matches per token — 0.1·F_mean is where one more token
               stops paying for the +0.1 it adds to METEOR's denominator
      prose  : the model's answer; its tokens are already 'spent'"""
    budget = SEL_BUDGET_TOK if budget is None else budget
    p0    = int(ctx.at[qid, "ctx_positions"][0])
    query = str(ctx.at[qid, "question"])
    cls   = clauses_of(article_text(p0, 9000))
    toks  = [vi_tokens(c) for c in cls]
    yh    = [clause_score(c, j, len(cls), query) for j, c in enumerate(cls)]
    keep, rejected = select_clauses(toks, yh, vi_tokens(prose), REPEAT_S, budget, bar)
    body  = "\n".join(cls[j] for j in keep)
    out   = f"Căn cứ {clean_header(chunk_header(p0))}:\n{body}" if body else ""
    return (out, {"n": len(cls), "kept": len(keep), "bar_rejected": rejected}) if return_stats else out


# ══ C. MEASURE ═════════════════════════════════════════════════════════
_ids = [q for q in val_ctx.index if q in qa_by_id.index and q in _raw]
_ref = {q: str(qa_by_id.at[q, "answer"]) for q in _ids}
_pro = {q: clean_answer(_raw[q]) for q in _ids}   # clean_answer also collapses loops (CELL 9),
                                                   # exactly as cell 12 prepares the prose

def _with(q, cite):
    p = _pro[q]
    return (p + "\n\nTrích dẫn quy định:\n" + cite).strip() if cite.strip() else p

def _ev(fn, label, mark=""):
    outs = [fn(q) for q in _ids]
    M = float(np.mean([meteor(o, _ref[q])  for o, q in zip(outs, _ids)]))
    R = float(np.mean([rouge_l(o, _ref[q]) for o, q in zip(outs, _ids)]))
    L = float(np.mean([len(vi_tokens(o)) for o in outs]))
    dens = []
    for o, q in zip(outs, _ids):
        pool, h, tt = Counter(vi_tokens(_ref[q])), 0, vi_tokens(o)
        for w in tt:
            if pool[w] > 0: pool[w] -= 1; h += 1
        dens.append(h / max(1, len(tt)))
    D = float(np.mean(dens))
    print(f"  {label:<44} {M:>7.4f} {R:>8.4f} {L:>7.0f} {D:>8.1%} "
          f"{M-globals().get('CB_GAP',0.002):>10.4f}{mark}")
    return M, R, L, D

print(f"\n  {'strategy':<44} {'METEOR':>7} {'ROUGE-L':>8} {'len':>7} {'density':>8} {'proj. CB':>10}")
print("  " + "-" * 90)
out = {}
out["base"] = _ev(lambda q: _with(q, build_citation(q, val_ctx, cfg.cite_mode, cfg.cite_budget,
                                                    cfg.cite_n_chunks)),
                  f"baseline whole-article @{cfg.cite_budget}")
_BAR = 0.1 * out["base"][0]
SUPERVISED_BAR = _BAR          # cell 12 reuses THIS bar for "supauto", not a re-derived one
print("  " + "-" * 90)
for b in (180, 260, 340, 420, 520, 640, 800):
    out[f"sup{b}"] = _ev(lambda q, b=b: _with(q, supervised_citation(q, val_ctx, b, prose=_pro[q])),
                         f"marginal-greedy, cap {b} tok")
print("  " + "-" * 90)
_stats = []
def _auto(q):
    c, s = supervised_citation(q, val_ctx, 9999, bar=_BAR, prose=_pro[q], return_stats=True)
    _stats.append(s); return _with(q, c)
out["supauto"] = _ev(_auto, f"marginal rule, bar {_BAR:.3f}, no cap", "   ← the rule you asked for")
_fired = sum(1 for s in _stats if s["bar_rejected"] > 0)
_rej   = sum(s["bar_rejected"] for s in _stats); _tot = sum(s["n"] for s in _stats)
print(f"  {'':<44} the bar fired on {_fired}/{len(_stats)} questions and dropped "
      f"{_rej}/{_tot} clauses ({_rej/max(1,_tot):.1%})")
print(f"  {'':<44} (v10's rule: 0 of every one of them — it could not fire)")
print("  " + "-" * 90)
out["supo260"] = _ev(lambda q: supervised_citation(q, val_ctx, 260, prose=""),
                     "selected citation ONLY (no prose), 260 tok")

_b = max(out.items(), key=lambda kv: kv[1][0])
print("  " + "-" * 90)
print(f"  best: {_b[0]} → val {_b[1][0]:.4f} ({_b[1][0]-out['base'][0]:+.4f} vs baseline)")
print(f"  mean length {_b[1][2]:.0f} tok at density {_b[1][3]:.1%}")
if abs(_b[1][0] - out["base"][0]) < 0.003:
    print("\n  Within ±0.003 of the baseline: selection is not where the score is. The")
    print("  article being selected FROM decides the outcome — see cell 6d.")
assert callable(W), "W was rebound by this cell — cell 12 would crash on W('submission.json')"
SUPERVISED_BEST, SUPERVISED_LAB = _b[0], {k: v[0] for k, v in out.items()}


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 10e — PROMPT DENSITY LAB   (GPU, ~8-12 min on A100 — the real lever)
# ══════════════════════════════════════════════════════════════════════
# WHAT 10c AND 10d CANNOT DO
#
# Both only DELETE. Deleting raises precision but can never raise m, and
#
#       M = (1 − pen) · m / (0.9·|ref| + 0.1·|hyp|)
#
# is bounded above by m/(0.9·|ref|) no matter how short you get. With m ≈ 221
# and |ref| ≈ 364 that ceiling is 0.67, but the reachable part of it — what
# pruning alone buys — tops out near 0.55. The gap from 0.55 to rank 1's 0.61
# is not precision. It is MORE MATCHED TOKENS.
#
# You cannot get those by re-ranking or by cutting. You get them by changing
# what the model writes — and the prompt is the only knob left that does that
# without training. So this cell regenerates a val subset under competing
# system prompts and measures density directly.
#
# Set RUN_PROMPT_LAB = False to skip; everything downstream still works.
RUN_PROMPT_LAB = False      # measured in v10: no variant beat B_current. Re-enable only
                            # to test a NEW system prompt (≈10 min of A100).
PROMPT_LAB_N   = 120        # 120 × 4 prompts ≈ 8-12 min on an A100

_MISSING = [n for n in ['model','tokenizer','generate_answers','val_ctx','qa_by_id',
                        'meteor','rouge_l','vi_tokens','clean_answer','chunk_header',
                        'chunks','cfg','SYS_B','build_citation','_raw']
            if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELLS T4, 9 and 10 first.")

import numpy as np, json, time, os
from collections import Counter

# ── the candidates ────────────────────────────────────────────────────
# SYS_B (current) tells the model to quote EVERYTHING relevant. That is a
# recall instruction, and it is why your prose runs 399 tokens. The variants
# below trade breadth for register — matching how the references are actually
# written — without asking the model to be shorter for its own sake.
SYS_VARIANTS = {
    "B_current": SYS_B,

    "C_verbatim": (
        "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. Trả lời câu hỏi DỰA TRÊN "
        "các trích đoạn văn bản pháp luật được cung cấp.\n"
        "Cách trả lời bắt buộc:\n"
        "- Câu đầu tiên nêu căn cứ pháp lý đầy đủ: \"Căn cứ <khoản> <Điều> <văn bản> "
        "quy định như sau:\"\n"
        "- Sau đó TRÍCH NGUYÊN VĂN điều khoản áp dụng, giữ nguyên cách đánh số 1., 2., "
        "a), b) và giữ nguyên từng chữ của văn bản gốc. Không diễn giải lại, không "
        "thay từ đồng nghĩa, không thêm chữ của riêng bạn vào phần trích.\n"
        "- Kết thúc bằng MỘT câu kết luận trực tiếp trả lời câu hỏi.\n"
        "- Không mở bài, không nhắc đến ngữ cảnh được cung cấp, không lặp lại câu hỏi."),

    "D_scoped": (
        "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. Trả lời câu hỏi DỰA TRÊN "
        "các trích đoạn văn bản pháp luật được cung cấp.\n"
        "Cách trả lời bắt buộc:\n"
        "- Mở đầu: \"Căn cứ <khoản> <Điều> <văn bản>:\"\n"
        "- Chỉ trích nguyên văn NHỮNG khoản, điểm THỰC SỰ áp dụng cho câu hỏi. Bỏ qua "
        "các khoản không liên quan, dù chúng nằm cùng một Điều.\n"
        "- Giữ nguyên từng chữ và cách đánh số của phần được trích.\n"
        "- Kết thúc bằng một câu kết luận. Không mở bài, không nhắc ngữ cảnh."),

    "E_answer_first": (
        "Bạn là trợ lý pháp lý chuyên về pháp luật Việt Nam. Trả lời câu hỏi DỰA TRÊN "
        "các trích đoạn văn bản pháp luật được cung cấp.\n"
        "Cách trả lời bắt buộc:\n"
        "- Câu đầu: trả lời trực tiếp câu hỏi bằng một câu.\n"
        "- Tiếp theo: \"Căn cứ <khoản> <Điều> <văn bản> quy định:\" rồi TRÍCH NGUYÊN VĂN "
        "điều khoản áp dụng, giữ nguyên cách đánh số và từng chữ của bản gốc.\n"
        "- Không mở bài, không nhắc ngữ cảnh, không lặp lại câu hỏi."),
}

def _prompt(question, positions, sys_text):
    blocks = [f"[Văn bản {i}] {chunk_header(int(p))}\n"
              f"{chunks.iloc[int(p)]['content'][:cfg.max_chunk_chars]}"
              for i, p in enumerate(positions, 1)]
    user = ("Các trích đoạn văn bản pháp luật:\n\n" + "\n\n".join(blocks) +
            f"\n\nCâu hỏi: {str(question).strip()}\n\nTrả lời:")
    return tokenizer.apply_chat_template(
        [{"role": "system", "content": sys_text}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)

_ids = [q for q in val_ctx.index if q in qa_by_id.index and q in _raw][:PROMPT_LAB_N]
_ref = {q: str(qa_by_id.at[q, "answer"]) for q in _ids}
_PCACHE = W(f"prompt_lab_cache_{globals().get('RET_FP', 'v10')}.json")   # per context set
_cache  = json.load(open(_PCACHE, encoding="utf-8")) if os.path.exists(_PCACHE) else {}

def _density(text, ref):
    pool, t, h = Counter(vi_tokens(ref)), vi_tokens(text), 0
    for w in t:
        if pool[w] > 0: pool[w] -= 1; h += 1
    return h / max(1, len(t)), h

if RUN_PROMPT_LAB and _ids:
    print(f"prompt lab: {len(SYS_VARIANTS)} system prompts × {len(_ids)} val questions")
    print(f"  cached variants: {sorted(_cache)}\n")
    for _name, _sys in SYS_VARIANTS.items():
        if _name in _cache and len(_cache[_name]) >= len(_ids): continue
        _t0 = time.time()
        _o  = generate_answers([_prompt(val_ctx.at[q, "question"],
                                        val_ctx.at[q, "ctx_positions"], _sys) for q in _ids])
        _cache[_name] = dict(zip([str(q) for q in _ids], _o))
        json.dump(_cache, open(_PCACHE, "w", encoding="utf-8"), ensure_ascii=False)
        print(f"  {_name:<16} generated in {(time.time()-_t0)/60:.1f} min")

    print(f"\n  {'system prompt':<16} {'METEOR':>7} {'ROUGE-L':>8} {'prose':>7} {'ρ prose':>8} "
          f"{'m':>6} {'+cite':>8} {'proj CB':>9}")
    print("  " + "-" * 84)
    _tab = {}
    for _name in SYS_VARIANTS:
        if _name not in _cache: continue
        _p  = {q: clean_answer(_cache[_name][str(q)]) for q in _ids}
        _M  = float(np.mean([meteor(_p[q], _ref[q]) for q in _ids]))
        _RL = float(np.mean([rouge_l(_p[q], _ref[q]) for q in _ids]))
        _L  = float(np.mean([len(vi_tokens(_p[q])) for q in _ids]))
        _dh = [_density(_p[q], _ref[q]) for q in _ids]
        _D  = float(np.mean([d for d, _ in _dh]))
        _m  = float(np.mean([h for _, h in _dh]))
        _full = [(_p[q] + "\n\nTrích dẫn quy định:\n"
                  + build_citation(q, val_ctx, cfg.cite_mode, cfg.cite_budget,
                                   cfg.cite_n_chunks)).strip() for q in _ids]
        _MF = float(np.mean([meteor(o, _ref[q]) for o, q in zip(_full, _ids)]))
        _tab[_name] = (_M, _RL, _L, _D, _m, _MF)
        print(f"  {_name:<16} {_M:>7.4f} {_RL:>8.4f} {_L:>7.0f} {_D:>8.1%} {_m:>6.0f} "
              f"{_MF:>8.4f} {_MF-globals().get('CB_GAP',0.002):>9.4f}"
              + ("   ← current" if _name == "B_current" else ""))
    print("  " + "-" * 84)
    if _tab:
        _bp = max(_tab.items(), key=lambda kv: kv[1][5])
        _cu = _tab.get("B_current", _bp)[5]
        print(f"  best: {_bp[0]}  val(+citation) {_bp[1][5]:.4f}  ({_bp[1][5]-_cu:+.4f} vs current)")
        print(f"        prose {_bp[1][2]:.0f} tok at density {_bp[1][3]:.1%}, "
              f"m = {_bp[1][4]:.0f} matched tokens")
        print(f"\n  The column that matters is m. ρ and length can both improve while m")
        print(f"  falls — that is precision bought with recall, and METEOR sees through it.")
        if _bp[0] != "B_current" and _bp[1][5] > _cu + 0.004:
            print(f"\n  → ADOPT: set SYS_B = SYS_VARIANTS['{_bp[0]}'] and RE-RUN cells 10, 10c,")
            print(f"    10d and 11. Their caches are keyed by the prompt hash (v11), so every")
            print(f"    answer regenerates under the new prompt — nothing to delete.")
        else:
            print(f"\n  → KEEP B_current. No variant clears it by more than noise "
                  f"(n={len(_ids)}, se ≈ {np.std([meteor(_p[q],_ref[q]) for q in _ids])/np.sqrt(len(_ids)):.4f}).")
        PROMPT_LAB = {k: {"meteor_prose": v[0], "len": v[2], "density": v[3],
                          "m": v[4], "meteor_full": v[5]} for k, v in _tab.items()}
        PROMPT_BEST = _bp[0]
else:
    PROMPT_LAB, PROMPT_BEST = {}, "B_current"
    print("prompt lab skipped (RUN_PROMPT_LAB=False)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 11 — Full inference over the test set (resume-safe, prompt-keyed RAW)
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['test_ctx', 'to_infer_text', 'generate_answers', 'W',
                        'load_raw_cache', 'test_data'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELLS 7 and 9 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# RAW answers only, so any citation strategy is a seconds-long repack. Each
# record carries the hash of the EXACT prompt (plus adapter and decoding
# settings) that produced it, and is reused only while that hash still matches.
# v10's file was keyed by question id alone — which is how the 0.5495 run
# shipped 1,000 answers generated before any of the retrieval fixes existed.
GEN_OUT  = W("gen_raw_v11.jsonl")
_GLEGACY = (W("gen_raw_v6.jsonl"), W(f"test_ctx_k{cfg.top_k_context}.parquet"))

assert set(map(str, test_ctx.index)) == set(map(str, test_data)), (   # v13 (CELL 7 verified it)
    "test_ctx does not answer the test file CELL 3 loaded — re-run CELL 7")
_tq = [str(q) for q in test_ctx.index]
prompts = {q: to_infer_text(test_ctx.at[q, "question"], test_ctx.at[q, "ctx_positions"])
           for q in _tq}
raw_cache, _tfp, _tst = load_raw_cache(GEN_OUT, prompts, legacy=_GLEGACY)
todo = [q for q in _tq if q not in raw_cache]
print(f"test answers: {_tst['reused']} of {len(_tq):,} reusable ({_tst['legacy']} carried over from "
      f"the v10 cache, {_tst.get('by_prompt', 0)} found by their exact prompt under another id) | "
      f"{len(todo):,} to generate")
if _tst["legacy_rejected"]:
    print(f"  {_tst['legacy_rejected']} v10 answers were NOT reused: their prompt changed "
          f"(different chunks, or a different prompt format)")
if todo:
    print(f"  ≈ {len(todo) * 2.5 / 60:.0f} min on an A100 (≈2-3 s per answer on earlier runs)")
    todo.sort(key=lambda q: -len(prompts[q]))      # length-sorted → less padding waste
    B, t0 = PROFILE["gen_batch"], time.time()
    with open(GEN_OUT, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), B), desc="test set"):
            qb = todo[i:i+B]
            for qid, ans in zip(qb, generate_answers([prompts[q] for q in qb])):
                raw_cache[qid] = ans
                f.write(json.dumps({"qa_id": qid, "raw": ans, "fp": _tfp[qid]},
                                   ensure_ascii=False) + "\n")
            f.flush()                              # a disconnect costs one batch
    print(f"generated {len(todo)} in {(time.time()-t0)/60:.1f} min")
else:
    print("all answers valid for the current prompts — nothing to generate")

assert set(raw_cache) == set(_tq), f"{len(set(_tq) - set(raw_cache))} test answers missing"
GEN_STATS = {**_tst, "generated": len(todo)}
_n = np.array([len(vi_tokens(v)) for v in raw_cache.values()])
_cap = sum(1 for v in raw_cache.values()
           if len(tokenizer(v).input_ids) >= cfg.max_new_tokens - 8)
_lp = sum(1 for v in raw_cache.values() if collapse_loops(v) != v)
print(f"raw answers {len(raw_cache)} | mean {_n.mean():.0f} p90 {np.percentile(_n,90):.0f} "
      f"tok | at the {cfg.max_new_tokens} ceiling: {_cap} | decoding loops: {_lp} "
      f"(collapsed by clean_answer)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 12 — Assemble, VERIFY, package
# ══════════════════════════════════════════════════════════════════════
_MISSING = [n for n in ['raw_cache', 'build_citation', 'finalize', 'known_by_q', 'test_ctx',
                        'test_data'] if n not in globals()]
if _MISSING:
    raise RuntimeError(
        f"prerequisites missing: {_MISSING}.\n"
        "Run CELL 11 first — scroll to ITS output to see why it stopped. "
        "A NameError here is a symptom, not the cause.")

# The guards below exist because a previous submission scored identically to the
# one before it: a stale cell ran, produced the old output, printed success, and
# nothing checked. A packaging step that cannot detect its own no-op is not a
# packaging step.
import zipfile, hashlib
from collections import Counter, defaultdict

# Decoding loops are collapsed by collapse_loops(), which v11 defines in CELL 9
# and runs INSIDE clean_answer() — so the val labs measure exactly what this
# cell ships. v10 defined it here; a def dropped into the emission loop below is
# what truncated the loop body and raised "QID mismatch".
assert "collapse_loops" in globals(), "re-run CELL 9 (v11) — it defines collapse_loops"


# ── known-QA: exact → normalised → GUARDED fuzzy ──────────────────────
# A correct override is worth ~+0.45 METEOR on that question, a WRONG one costs
# ~−0.5. So the threshold is strict and four guards sit on top of it.
import math

FUZZY_KNOWN       = True
FUZZY_THRESHOLD   = 0.95   # token-set Jaccard, after norm_q_deep
FUZZY_MAX_DIFF    = 2      # at most this many tokens in the symmetric difference
FUZZY_ALLOW_SUBST = False  # a token swapped for another ("vợ"→"chồng") = new question

# Vietnamese has two tone-mark conventions: old "hoà, thuỷ" and new "hòa, thủy".
# NFC does NOT unify them — they are different code points — so the same
# question typed on two keyboards misses an EXACT match. Map old → new on both
# sides; only consistency matters.
_TONE = {"oà": "òa", "oá": "óa", "oả": "ỏa", "oã": "õa", "oạ": "ọa",
         "oè": "òe", "oé": "óe", "oẻ": "ỏe", "oẽ": "õe", "oẹ": "ọe",
         "uỳ": "ùy", "uý": "úy", "uỷ": "ủy", "uỹ": "ũy", "uỵ": "ụy"}
_TONE_RE = re.compile("|".join(map(re.escape, _TONE)))

_FILLER = re.compile(
    r"\b(?:"
    r"(?:xin\s+)?(?:cho|nhờ)\s+(?:tôi|em|mình|cháu|con)\s+(?:được\s+)?hỏi"
    r"|ban\s+biên\s+tập\s+(?:cho\s+(?:tôi|em|mình)\s+)?hỏi"
    r"|(?:luật\s+sư|anh\s+chị|admin)\s+(?:ơi|cho\s+(?:tôi|em|mình)\s+hỏi)"
    r"|(?:tôi|em|mình)\s+(?:muốn|có)\s+(?:hỏi|câu\s+hỏi)"
    r"|(?:xin\s+)?chào\s+(?:luật\s+sư|anh\s+chị|ban\s+biên\s+tập|admin)|xin\s+chào|xin\s+hỏi"
    r"|mong\s+(?:được\s+)?(?:giải\s+đáp|tư\s+vấn|hồi\s+âm|phản\s+hồi)"
    r"|xin\s+(?:chân\s+thành\s+)?cảm\s+ơn|cảm\s+ơn"
    r"|nhờ\s+(?:luật\s+sư\s+)?tư\s+vấn|xin\s+tư\s+vấn"
    r"|vậy\s+ạ|với\s+ạ|nhé|ạ"
    r"|như\s+thế\s+nào|thế\s+nào|là\s+gì|ra\s+sao"
    r")\b", re.I)


def norm_q(s):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", str(s)).lower().strip())


# "Có được … không?" / "Đã … chưa?" — here không/chưa is the QUESTION particle,
# not a negation, and people drop it or keep it at will. It is the one that
# closes a clause: followed by punctuation, the end, or a conjunction. A
# negation is followed by what it negates ("không được", "chưa nộp") and stays.
_QPART = re.compile(r"\b(?:không|chưa)\b(?=\s*(?:[?.,;:!…]|$|\b(?:và|hay|hoặc|thì|nếu|khi|nhưng|còn)\b))")


def norm_q_deep(s):
    """NFC → lowercase → one tone convention → politeness filler out →
    yes/no question particle out → punctuation out (keeping '/' for document
    numbers) → single spaces. Applied IDENTICALLY to the test question and to
    every known_qa key."""
    t = _TONE_RE.sub(lambda m: _TONE[m.group(0)], norm_q(s))
    t = _FILLER.sub(" ", t)
    t = _QPART.sub(" ", t)                  # before punctuation goes: it is the anchor
    t = re.sub(r"[^\w\s/]", " ", t)
    return re.sub(r"\s+", " ", t).strip()


def _tokset(s):
    return frozenset(norm_q_deep(s).split())


# Tokens whose difference flips the answer even at 0.95 Jaccard. On a 40-token
# question 0.95 permits ONE differing token — and if that token is "5" vs "6"
# (Điều 5 vs Điều 6) or "không", the organiser's answer is for a different
# question. A wrong override costs ~0.5 METEOR; a missed one costs ~0.45.
_POL  = re.compile(r"^(?:không|chưa|chẳng|chả|cấm|trừ|miễn|ngoại)$")
_CRIT = re.compile(r"\d|^(?:không|chưa|chẳng|chả|cấm|trừ|miễn|ngoại)$")


def _crit_seq(normed):
    """Critical tokens IN ORDER, WITH repeats — and each polarity word together
    with the word it governs. A token SET sees none of these:
      'khoản 2 điều 3'           vs 'khoản 3 điều 2'           (numbers swapped)
      'không được … chấm dứt'    vs 'được … không chấm dứt'    (negation moved)
      a second 'không' inserted where the question already has one."""
    w = normed.split()
    return [(t, w[i + 1] if i + 1 < len(w) else "") if _POL.match(t) else t
            for i, t in enumerate(w) if _CRIT.search(t)]


def build_deep_known(known_by_q):
    """{norm_q_deep(question): answer}. Two DIFFERENT answers whose questions
    normalise to the same key make that key ambiguous — it is dropped rather
    than letting whichever came last win. (The exact norm_q table still serves
    both originals.) Returns (table, n_ambiguous)."""
    deep, amb = {}, set()
    for k, v in known_by_q.items():
        d, a = norm_q_deep(k), str(v).strip()
        if not d or not a:
            continue
        if d in deep and deep[d] != a:
            amb.add(d)
        else:
            deep.setdefault(d, a)
    for d in amb:
        deep.pop(d, None)
    return deep, len(amb)


def build_fuzzy_index(known):
    """Inverted index token → keys. `known` must already be keyed by norm_q_deep."""
    inv, keys = defaultdict(list), {}
    for k in known:
        ts = frozenset(k.split())
        if not ts:
            continue
        keys[k] = ts
        for t in ts:
            inv[t].append(k)
    return inv, keys


def fuzzy_lookup(q, known, inv, keys, thr=None, exclude=None, return_reason=False, guards=True):
    """Best token-set Jaccard match at or above `thr`, then four guards:
      · the symmetric difference holds no digit and no polarity word;
      · no token was SUBSTITUTED (both sides have a token the other lacks),
        unless FUZZY_ALLOW_SUBST;
      · the symmetric difference is at most FUZZY_MAX_DIFF tokens;
      · digits and polarity words appear in the same order and count, and each
        polarity word negates the same word.
    `exclude` hides one key and guards=False skips the four checks — cell 12a
    uses both, for leave-one-out and to show what the guards prevent.

    Candidates come from PREFIX FILTERING, which is exact, not a heuristic: a key
    with Jaccard ≥ thr shares ≥ ⌈thr·|q|⌉ of the query's tokens, so it must share
    at least one of the |q| − ⌈thr·|q|⌉ + 1 RAREST ones. Only those tokens'
    posting lists are scanned — no 'top-40' cut that a common word can crowd.
    Returns (key, similarity[, reason]); key is None when nothing qualifies."""
    thr = FUZZY_THRESHOLD if thr is None else float(thr)
    qd = norm_q_deep(q)
    qs = frozenset(qd.split())
    if not qs:
        return (None, 0.0, "empty") if return_reason else (None, 0.0)
    need = min(len(qs), max(1, math.ceil(thr * len(qs) - 1e-9)))     # ε: 0.7·10 = 7.000000000000001
    rare = sorted(qs, key=lambda t: (len(inv.get(t, ())), t))[:len(qs) - need + 1]
    best, bs = None, 0.0
    for k in {k for t in rare for k in inv.get(t, ())}:
        if k == exclude:
            continue
        ks = keys[k]
        shared = len(qs & ks)
        if shared < need:
            continue
        j = shared / (len(qs) + len(ks) - shared)
        if j > bs or (j == bs and best is not None and k < best):   # deterministic ties
            best, bs = k, j
    if best is None or bs < thr:
        return (None, bs, "below threshold") if return_reason else (None, bs)
    if not guards:
        return (best, bs, "ok") if return_reason else (best, bs)
    ks = keys[best]
    diff = qs ^ ks
    crit = sorted(t for t in diff if _CRIT.search(t))
    why = ("critical token differs: " + ",".join(crit) if crit else
           "token substituted: " + ",".join(sorted(qs - ks)) + "→" + ",".join(sorted(ks - qs))
           if (qs - ks) and (ks - qs) and not FUZZY_ALLOW_SUBST else
           f"{len(diff)} tokens differ" if len(diff) > FUZZY_MAX_DIFF else
           "critical tokens differ in order, count or scope" if _crit_seq(qd) != _crit_seq(best) else None)
    if why:
        return (None, bs, why) if return_reason else (None, bs)
    return (best, bs, "ok") if return_reason else (best, bs)


_known = {norm_q(k): str(v) for k, v in known_by_q.items()} if cfg.use_known_qa_lookup else {}
_known_deep, _n_amb = build_deep_known(known_by_q) if _known else ({}, 0)
_finv, _fkeys = build_fuzzy_index(_known_deep) if (FUZZY_KNOWN and _known_deep) else (None, None)
if _n_amb:
    print(f"known_qa: {_n_amb} normalised questions carry DIFFERENT answers — dropped from the "
          f"normalised/fuzzy tables (their exact spellings still match)")

# VERIFY ON VAL — LEAVE-ONE-OUT. 118 of 120 val questions have an exact twin, so
# v10's audit only asked fuzzy about questions exact matching had already
# answered ("fuzzy 0" measured nothing). Here each val question's own entry is
# HIDDEN and fuzzy must find an answer anyway — the job it does on test.
if _finv:
    _probe, _sc = 0, []
    for q in [q for q in val_ctx.index if q in qa_by_id.index]:
        _s = val_ctx.at[q, "question"]; _own = norm_q_deep(_s)
        if _own not in _known_deep: continue
        _probe += 1
        _h, _sim = fuzzy_lookup(_s, _known_deep, _finv, _fkeys, exclude=_own)
        if _h is not None:
            _sc.append(meteor(_known_deep[_h], str(qa_by_id.at[q, "answer"])))
    print(f"known-QA fuzzy audit (leave-one-out, J ≥ {FUZZY_THRESHOLD} + guards): "
          f"{_probe} val questions probed, {len(_sc)} fuzzy hits")
    if len(_sc) >= 5:
        print(f"  METEOR of those hits: mean {np.mean(_sc):.4f} (a generated answer ≈ 0.55)")
        if np.mean(_sc) < 0.70:
            FUZZY_KNOWN, _finv = False, None
            print("  ⚠ fuzzy hits are not reliably right — DISABLED for the test set")
        else:
            print("  ✓ fuzzy hits are reliable — applied to the test set")
    else:
        print("  too few hits to measure precision on val; the strict guards stay in force "
              "(CELL 12a runs the full sweep)")

# ── v12: FUZZY THRESHOLD CALIBRATION — lowered only on evidence ───────
# 0.95 is a prior, not a measurement. Leave-one-out supplies the measurement:
# hide one known_qa entry, let fuzzy_lookup find its nearest OTHER entry, and
# score that entry's answer against the hidden one's. That is exactly the
# test-time event — a question that paraphrases a known one receives the known
# one's answer — and known_qa supplies thousands of such probes where val
# supplies about a hundred.
# Lowering the threshold only ADDS the matches in the band just below it, so
# each band must pay for itself: an override is worth it when its METEOR beats
# the ≈0.549 of a generated answer, and the band's one-sided 95% LOWER bound has
# to clear that bar on at least FUZZY_MIN_PROBES probes. The four guards stay on
# throughout — this only moves the Jaccard bar.
import random
FUZZY_AUTO       = True
FUZZY_CANDIDATES = (0.92, 0.90)      # tried in order, below the current FUZZY_THRESHOLD
FUZZY_MIN_PROBES = 20
FUZZY_MAX_PROBES = 4000
GEN_METEOR       = 0.549             # a generated answer: what an override has to beat

def fuzzy_calibration_probes(known_deep, inv, keys, floor, max_probes=FUZZY_MAX_PROBES,
                             guards=True, seed=0):
    """[(similarity, METEOR(answer found, hidden entry's answer))] over known_qa."""
    rep = {}
    for k in known_by_q: rep.setdefault(norm_q_deep(k), k)
    ks = [k for k in known_deep if k in rep]
    random.Random(seed).shuffle(ks)
    out = []
    for k in ks[:max_probes]:
        h, s = fuzzy_lookup(rep[k], known_deep, inv, keys, thr=floor, exclude=k, guards=guards)
        if h is not None:
            out.append((s, meteor(known_deep[h], known_deep[k])))
    return out

def _band(vals):
    n = len(vals)
    if n == 0: return 0, float("nan"), float("nan")
    mu = float(np.mean(vals)); se = float(np.std(vals, ddof=1) / np.sqrt(n)) if n > 1 else float("inf")
    return n, mu, mu - 1.645 * se

FUZZY_CALIB = None
if FUZZY_KNOWN and _finv and FUZZY_AUTO:
    _floor = min(FUZZY_CANDIDATES + (FUZZY_THRESHOLD,))
    _kk = fuzzy_calibration_probes(_known_deep, _finv, _fkeys, _floor)
    _thr0 = _new = FUZZY_THRESHOLD
    print(f"\nfuzzy calibration: {min(len(_known_deep), FUZZY_MAX_PROBES):,} known_qa entries "
          f"probed leave-one-out, {len(_kk)} found a guarded match ≥ {_floor}")
    _n, _mu, _lb = _band([m for s_, m in _kk if s_ >= _thr0])
    print(f"  J ≥ {_thr0:.2f}          n {_n:>4}  METEOR {_mu:.3f}  lower bound {_lb:.3f}  (current)")
    if _n >= FUZZY_MIN_PROBES and _mu < GEN_METEOR:
        FUZZY_KNOWN, _finv = False, None
        print(f"  ⚠ even at {_thr0} a fuzzy answer scores below a generated one — fuzzy matching OFF")
    else:
        for _t in sorted(set(FUZZY_CANDIDATES), reverse=True):
            if _t >= _new: continue
            _n, _mu, _lb = _band([m for s_, m in _kk if _t <= s_ < _new])
            _ok = _n >= FUZZY_MIN_PROBES and _lb > GEN_METEOR
            print(f"  {_t:.2f} ≤ J < {_new:.2f}  n {_n:>4}  METEOR {_mu:.3f}  lower bound {_lb:.3f}"
                  f"  → {'pays — lowered' if _ok else 'not proven — stop'}")
            if not _ok: break
            _new = _t
        if _new != _thr0:
            FUZZY_THRESHOLD = _new
            print(f"  ✓ FUZZY_THRESHOLD {_thr0} → {FUZZY_THRESHOLD} (every band above it paid for itself)")
        else:
            print(f"  FUZZY_THRESHOLD stays {FUZZY_THRESHOLD}")
    FUZZY_CALIB = {"probes": len(_kk), "threshold_before": _thr0, "threshold": FUZZY_THRESHOLD,
                   "enabled": bool(FUZZY_KNOWN)}

# Citation strategy — set from the cell 10c / 10d tables, never guessed.
#   "article"    whole article @ cfg.cite_budget   (current, val 0.5398)
#   "dedup"      article minus clauses the prose already supplied  (cell 10c)
#   "supervised" clauses ranked by fitted reference-yield           (cell 10d)
# AUTO-ADOPT what the labs measured. A hand-edited constant is how the earlier
# repack shipped the old configuration and scored identically twice — the file
# looked new, the settings were not. Set CITE_OVERRIDE to force a choice.
# Measured val→Codabench gap, from the two runs that have actually been scored:
#     val 0.5398 → CB 0.5224   gap -0.0174   (whole-article @4000)
#     val 0.5509 → CB 0.5486   gap -0.0023   (supervised, cap 640)
# The notebook used to hardcode -0.019 everywhere, which understated the second
# run by 0.017. Two points, sd 0.011 — treat any projection as ±0.015, and
# prefer the ranking of configurations over their absolute predicted scores.
CB_GAP = 0.002

CITE_OVERRIDE   = None          # None = use the measured winner
DEDUP_THRESHOLD = 0.90          # only used if CITE_OVERRIDE forces "dedup".
                                # 0.90 not 0.7: on the 0.5224 file, 0.70 cut 33
                                # novel tokens per answer and projected BELOW
                                # 0.90 under every assumption about |ref|.
SEL_BAR         = None

def _pick_strategy():
    """Compare what cells 10c/10d actually MEASURED. They publish explicit
    tables (DEDUP_LAB, SUPERVISED_LAB) rather than leaving `res`/`out` lying in
    the global namespace, because those names are generic enough that any later
    cell can rebind them and this picker would silently read the wrong table."""
    cands = [("article", float(globals().get("DEDUP_LAB", {}).get("baseline", 0.5398)))]
    for kk, vv in globals().get("DEDUP_LAB", {}).items():
        if kk.startswith("dedup"):
            cands.append(("dedup:" + kk[5:], float(vv)))
    for kk, vv in globals().get("SUPERVISED_LAB", {}).items():
        if kk == "supauto":
            cands.append(("supauto:0", float(vv)))
        elif kk.startswith("sup") and not kk.startswith("supo"):
            cands.append(("supervised:" + kk[3:], float(vv)))
    best = max(cands, key=lambda kv: kv[1])
    print("  measured candidates: " + " | ".join(f"{a} {b:.4f}" for a, b in cands))
    if len(cands) == 1:
        print("  ⚠ neither 10c nor 10d has run — falling back to the known-good")
        print("    configuration (val 0.5398 → CB 0.5224). Run them first; they")
        print("    cost minutes and this is where the remaining headroom is.")
    return best

if CITE_OVERRIDE:
    CITE_STRATEGY = CITE_OVERRIDE
    print(f"citation strategy: {CITE_STRATEGY} (forced via CITE_OVERRIDE)")
else:
    _name, _val = _pick_strategy()
    if _name.startswith("dedup:"):
        CITE_STRATEGY, DEDUP_THRESHOLD = "dedup", float(_name.split(":")[1])
    elif _name.startswith("supauto:"):
        CITE_STRATEGY, SEL_BUDGET_TOK = "supervised", 9999
        SEL_BAR = float(globals().get("SUPERVISED_BAR", 0.1 * _val))   # the bar 10d measured
    elif _name.startswith("supervised:"):
        CITE_STRATEGY = "supervised"
        SEL_BUDGET_TOK, SEL_BAR = int(_name.split(":")[1]), None
    else:
        CITE_STRATEGY = "article"
    print(f"citation strategy: {CITE_STRATEGY} (best measured, val {_val:.4f} "
          f"→ proj. CB {_val-globals().get('CB_GAP',0.002):.4f})")
print(f"citation strategy: {CITE_STRATEGY}"
      + (f" (threshold {DEDUP_THRESHOLD})" if CITE_STRATEGY=="dedup" else "")
      + (f" (budget {globals().get('SEL_BUDGET_TOK','?')} tok)"
         if CITE_STRATEGY=="supervised" else ""))

# v13: every question of the test FILE needs its own contexts (CELL 7 verified them;
# this re-checks, so a stale test_ctx fails here with a message, not a KeyError)
_tq_of = lambda q: test_data[q]["question"] if isinstance(test_data[q], dict) else str(test_data[q])
_nctx = [q for q in test_data if str(q) not in test_ctx.index]
assert not _nctx, f"{len(_nctx)} test questions have no contexts (e.g. {_nctx[:3]}) — re-run CELLS 7 and 11"
_qdiff = [q for q in test_data if str(test_ctx.at[str(q), "question"]) != _tq_of(q)]
assert not _qdiff, f"{len(_qdiff)} test_ctx questions differ from the test file (e.g. {_qdiff[:3]}) — re-run CELL 7"
submission, n_known, n_fallback, n_fuzzy, cite_len = {}, 0, 0, 0, []
for qid in test_data:                     # v13: the test FILE's ids, in its order
    qid = str(qid)
    # Verbatim organiser questions get the organiser's own answer. EXACT
    # normalised match only — a wrong fuzzy hit scores ~0, so a loose threshold
    # is negative expected value.
    _q = test_ctx.at[qid,"question"]
    k = _known.get(norm_q(_q)) or _known_deep.get(norm_q_deep(_q))
    if k is None and FUZZY_KNOWN and _finv:
        _hit, _sim = fuzzy_lookup(_q, _known_deep, _finv, _fkeys)
        if _hit is not None:
            k = _known_deep[_hit]; n_fuzzy += 1
    if k:
        submission[qid] = {"answer": str(k).strip()}; n_known += 1; continue
    # Use whatever cells 10c/10d proved best on val, not a hardcoded default.
    _p = clean_answer(raw_cache.get(qid, ""))           # collapses decoding loops (CELL 9)
    if CITE_STRATEGY == "supervised" and "supervised_citation" in globals():
        # prose=_p: selection is by MARGINAL yield given what the answer already says
        _c = supervised_citation(qid, test_ctx, SEL_BUDGET_TOK, bar=SEL_BAR, prose=_p)
    else:
        _c = build_citation(qid, test_ctx)
        if CITE_STRATEGY == "dedup" and "dedup_citation" in globals():
            _c, _ = dedup_citation(_p, _c, DEDUP_THRESHOLD)   # returns (kept, dropped)
    cite_len.append(len(vi_tokens(_c)))
    # Assembled HERE, not via finalize(), so the answer contains the strategy
    # this cell chose. finalize() always rebuilds a plain whole-article citation;
    # calling it after selecting a strategy silently discards the selection —
    # which is how a submission once scored identically to the one before it.
    ans = (_p + "\n\nTrích dẫn quy định:\n" + _c).strip() if _c.strip() else _p.strip()
    if not ans.strip():                            # never submit empty: METEOR = 0
        ans = _c or str(chunks.iloc[int(test_ctx.at[qid,"ctx_positions"][0])]["content"])[:1200]
        n_fallback += 1
    submission[qid] = {"answer": ans}

print(f"known-QA overrides: {n_known} (exact {n_known-n_fuzzy} + fuzzy {n_fuzzy}) "
      f"| fallbacks: {n_fallback}")
print(f"  each exact match is worth ~{(1.0-0.54)/max(1,len(test_data)):.5f} METEOR "
      f"on the {len(test_data):,}-question test set")
_cl = float(np.mean(cite_len)) if cite_len else 0.0
_L  = np.array([len(vi_tokens(v["answer"])) for v in submission.values()])
print(f"citation block: mean {_cl:.0f} tok   (sniper@3000 ≈ 185, article@4000 ≈ 450)")
print(f"answer length : mean {_L.mean():.0f} median {np.median(_L):.0f} "
      f"p90 {np.percentile(_L,90):.0f}")

# ── guards ────────────────────────────────────────────────────────────
_expected = {str(q) for q in test_data}     # v13: the test FILE — v12 compared against test_ctx,
                                            # so stale contexts would have passed this guard
_missing  = _expected - set(submission)
if _missing:
    raise RuntimeError(
        f"{len(_missing)} of {len(_expected)} questions produced no answer — the "
        f"emission loop assigned only {len(submission)}.\n"
        f"  known-QA overrides this run: {n_known}\n"
        + ("  THAT NUMBER EQUALS len(submission), so the loop body was TRUNCATED: a\n"
           "  column-0 statement (a def, a comment block) was inserted between the\n"
           "  known-QA `continue` and `submission[qid] = ...`, which ends the for\n"
           "  body in Python. Everything below it then runs once, after the loop.\n"
           if len(submission) == n_known else
           f"  sample missing qid: {sorted(_missing)[:3]}\n"
           "  raw_cache may be short — re-run CELL 11, it resumes.\n"))
assert set(submission) == _expected, "QID mismatch"
if globals().get("EXPECTED_N_TEST"):
    assert len(submission) == EXPECTED_N_TEST, f"{len(submission):,} answers, expected {EXPECTED_N_TEST:,}"
assert all(v["answer"].strip() for v in submission.values()), "empty answer present"
if CITE_STRATEGY in ("dedup", "supervised"):
    # ── LENGTH CALIBRATION ────────────────────────────────────────────
    # Rank 1 sits near 364 tokens, so that band is worth SEEING. It is not
    # worth TARGETING: they are short because their tokens are ~61% dense, and
    # length is the symptom. The gate below is therefore on the measured val
    # score, not on the token count — a config that lands in the band but
    # scores worse than the baseline is a worse submission, full stop.
    _prev = 906.0
    print(f"\n  LENGTH CALIBRATION  {_prev:.0f} → {_L.mean():.0f} tok "
          f"({100*(1-_L.mean()/_prev):+.0f}%)")
    print(f"    rank-1 band 250-400 tok: "
          f"{'✓ inside' if 250 <= _L.mean() <= 400 else '✗ outside — see below'}")
    _adopted = float(globals().get("_val", 0.0))
    _basev   = float(globals().get("DEDUP_LAB", {}).get("baseline", 0.0))
    if _basev:
        print(f"    val METEOR   baseline {_basev:.4f} → adopted {_adopted:.4f} "
              f"({_adopted-_basev:+.4f})")
        assert _adopted >= _basev - 1e-9, (
            f"the adopted strategy measured WORSE on val ({_adopted:.4f}) than the "
            f"baseline ({_basev:.4f}). Being inside the 250-400 band does not make a "
            f"submission better. Set CITE_OVERRIDE='article' or re-run 10c/10d.")
    if not (250 <= _L.mean() <= 400):
        print("    Outside the band is FINE when val improved: the band describes")
        print("    rank 1's content, not a length you can truncate your way into.")
        print("    Appending a token pays while its match rate beats 0.1·F_mean")
        print(f"    (≈{0.1*max(_adopted,0.5):.1%}) — cut below that and you lose recall.")
    assert _L.mean() > 200, (
        f"mean answer {_L.mean():.0f} tok — below 200, m ≤ min(|hyp|,|ref|) caps the "
        f"score under your current 0.5224 even at perfect precision.")
elif CITE_STRATEGY == "article" and cfg.cite_mode == "article" and cfg.cite_budget >= 2500:
    assert _cl > 300, (
        f"ARTICLE EXPANSION DID NOT APPLY — citations are {_cl:.0f} tokens, near the "
        f"185 of the sniper config. Check cell 3's 'fully named → expandable' line: "
        f"if most (doc, điều) keys are empty, article_text falls back to one chunk.")
    assert _L.mean() > 700, f"mean answer {_L.mean():.0f} tok — expected ~800"
_vb = globals().get("VAL_BEST")
if _vb and (_vb[0], _vb[1]) != (cfg.cite_mode, cfg.cite_budget):
    print(f"\n  ⚠ packaging {cfg.cite_mode}@{cfg.cite_budget}, but cell 10 measured "
          f"{_vb[0]}@{_vb[1]} higher ({_vb[2]:.4f})")

SUB_JSON = W("submission.json")
SUB_ZIP  = W(f"submission_{globals().get('TEST_TAG', 'test')}.json.zip")   # v13: named by the test set
with open(SUB_JSON, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=1)
with zipfile.ZipFile(SUB_ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(SUB_JSON, arcname="submission.json")   # ROOT of the archive: a
                                                   # nested path is a common
                                                   # silent Codabench rejection
with zipfile.ZipFile(SUB_ZIP) as z:
    assert z.namelist() == ["submission.json"], z.namelist()
    assert json.loads(z.read("submission.json").decode()) == submission

SUB_SHA = hashlib.sha256(open(SUB_ZIP,"rb").read()).hexdigest()
PREV = {"ad336eef": "0.4886 (whole-article did NOT apply)",
        "acac3f4b": "0.5224 (whole-article @4000, 906 tok)",
        "a641a7fc": "the offline dedup rebuild of the 0.5224 file"}
for pre, lbl in PREV.items():
    assert not SUB_SHA.startswith(pre), f"byte-identical to a previous submission: {lbl}"

# The zip hash above changes on every run (zip headers carry timestamps), so it
# cannot catch a no-op. The CONTENT hash can: identical answers, identical hash.
SUB_CONTENT_SHA = hashlib.sha256(json.dumps(submission, sort_keys=True, ensure_ascii=False)
                                 .encode("utf-8")).hexdigest()
PREV_CONTENT = {"8e7dcc287994833f": "the 0.5486 submission (supervised cap 640)",
                "a1cdd37e5889c332": "the 0.5224 submission (whole-article @4000)",
                "8c7659ffbe2bf8ff": "an earlier submission (584 tok/answer)",
                "30d4a8112c0d748d": "the offline-pruned copy of the 0.5486 file",
                "d3943f3d5316f97a": "the offline dedup file"}
for pre, lbl in PREV_CONTENT.items():
    assert not SUB_CONTENT_SHA.startswith(pre), (
        f"these answers are IDENTICAL to {lbl} — nothing upstream took effect")
try:
    _last = json.load(open(W("run_config_v6.json"), encoding="utf-8")).get("content_sha256")
except Exception:
    _last = None
if _last == SUB_CONTENT_SHA:
    print("  ⚠ answers identical to the last packaged run — if you changed something "
          "upstream, it did not reach the answers")

RUN_CFG = {"cite_mode": cfg.cite_mode, "cite_budget": cfg.cite_budget,
           "cite_n_chunks": cfg.cite_n_chunks, "top_k_context": cfg.top_k_context,
           "max_new_tokens": cfg.max_new_tokens, "rep_penalty": cfg.repetition_penalty,
           "use_lexref": bool(USE_LEXREF), "known_overrides": n_known,
           "cite_strategy": CITE_STRATEGY,
           "dedup_threshold": DEDUP_THRESHOLD if CITE_STRATEGY == "dedup" else None,
           "sel_budget_tok": globals().get("SEL_BUDGET_TOK") if CITE_STRATEGY == "supervised" else None,
           "sel_bar": SEL_BAR, "strip_scaffold": bool(globals().get("STRIP_SCAFFOLD", False)),
           "use_trained_adapter": bool(globals().get("USE_TRAINED_ADAPTER", False)),
           "adapter": str(globals().get("ADAPTER_IN_USE", "?")),
           "dedup_lab": globals().get("DEDUP_LAB"),
           "supervised_lab": globals().get("SUPERVISED_LAB"),
           "mean_answer_tok": round(float(_L.mean()),1),
           "mean_citation_tok": round(_cl,1), "sha256": SUB_SHA,
           "val_best": _vb, "retrieval": globals().get("RETRIEVAL_METRICS"),
           "content_sha256": SUB_CONTENT_SHA,
           "encoder": globals().get("ENCODER_SIG", "base"), "ret_fp": globals().get("RET_FP"),
           "retriever_ft": {k: v for k, v in (globals().get("RETRIEVER_FT") or {}).items()},
           "prompt_strip_scaffold": bool(globals().get("PROMPT_STRIP_SCAFFOLD", False)),
           "collapse_loops": bool(globals().get("COLLAPSE_LOOPS", False)),
           "fuzzy_threshold": FUZZY_THRESHOLD if FUZZY_KNOWN else None, "n_fuzzy": n_fuzzy,
           "fuzzy_calibration": FUZZY_CALIB, "labels": globals().get("LABEL_REPORT"),
           "hyde": {"on": bool(globals().get("USE_HYDE")), **(globals().get("HYDE") or {})},
           "rerank_variant": globals().get("RERANK_VARIANT", "v1"), "cand_pool": cfg.cand_pool,
           "gen": globals().get("GEN_STATS"), "ctx_delta": globals().get("CTX_DELTA"),
           "test": {"files": [os.path.basename(f) for f in globals().get("TEST_FILES", [])],
                    "n": len(test_data), "fp": globals().get("TEST_FP"),
                    "data_version": os.path.basename(str(globals().get("KAGGLE_DATA_ROOT", "")))},
           "corpus_fp": globals().get("CORPUS_FP")}
json.dump(RUN_CFG, open(W("run_config_v6.json"),"w"), ensure_ascii=False, indent=1,
          default=str)
print(f"\nsha256 {SUB_SHA[:24]}…")
print(f"✓ ALL GUARDS PASSED → upload {SUB_ZIP}")
print(f"  run config saved to {W('run_config_v6.json')}")


# ══════════════════════════════════════════════════════════════════════
#  Upgrade D — persist the artifact to the Hugging Face repo
# ══════════════════════════════════════════════════════════════════════
# Needs a WRITE-scoped HF_TOKEN. Skipped silently on a read token rather than
# failing the cell — the submission file is already on Drive either way.
PUSH_TO_HF = True
if PUSH_TO_HF and HF_TOKEN:
    try:
        from datetime import datetime, timezone
        _api_w = HfApi(token=HF_TOKEN)
        _role = _api_w.whoami().get("auth", {}).get("accessToken", {}).get("role", "?")
        # Do NOT gate on this string. A fine-grained token reports role
        # 'fineGrained' and may well carry write permission on this exact repo —
        # your last run printed "skipped — token role is 'fineGrained'" and
        # pushed nothing, even though the token could have written. The only
        # honest test of write access is to attempt the write.
        print(f"\nHF token role: {_role} — attempting the push (role is not a reliable gate)")
        if False:
            pass
        else:
            RUN_ID = f"{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}-{SUB_SHA[:8]}"
            _msg = (f"submission {RUN_ID} | cite={cfg.cite_mode}@{cfg.cite_budget}"
                    f" n={cfg.cite_n_chunks} | K={cfg.top_k_context}"
                    f" max_new={cfg.max_new_tokens} rep={cfg.repetition_penalty}"
                    f" | lexref={bool(USE_LEXREF)}"
                    f" | enc={globals().get('ENCODER_TAG', 'base')} ret={globals().get('RET_FP')}"
                    f" | mean_ans={RUN_CFG['mean_answer_tok']}tok"
                    f" | test={globals().get('TEST_TAG', '?')}"
                    f" | sha256={SUB_SHA[:16]}")
            for _local, _remote in ((SUB_ZIP, f"submissions/{RUN_ID}/{os.path.basename(SUB_ZIP)}"),
                                    (W("run_config_v6.json"),
                                     f"submissions/{RUN_ID}/run_config.json")):
                _api_w.upload_file(path_or_fileobj=_local, path_in_repo=_remote,
                                   repo_id=HF_SUBMIT, repo_type="model",
                                   commit_message=_msg)
                print(f"  pushed {_remote}")
            print(f"\nhttps://huggingface.co/{HF_SUBMIT}/tree/main/submissions/{RUN_ID}")
            print(f"  commit: {_msg}")
    except Exception as _ex:
        _m = str(_ex)
        print(f"\nHF push FAILED ({type(_ex).__name__}: {_m[:200]})")
        if "401" in _m or "403" in _m or "authoriz" in _m.lower() or "permission" in _m.lower():
            print(f"  → the token cannot write to {HF_SUBMIT}. On a fine-grained token,")
            print(f"    huggingface.co/settings/tokens → edit → Repositories → add")
            print(f"    {HF_SUBMIT} with 'Write access to contents/settings'.")
        elif "404" in _m or "not found" in _m.lower():
            print(f"  → {HF_SUBMIT} does not exist or is not visible to this token.")
            print(f"    Create it at huggingface.co/new, or fix HF_SUBMIT in CELL 0.")
        print("  the submission zip is still on Drive and cell 13 will download it")
else:
    print("\nHF push skipped (PUSH_TO_HF False or no HF_TOKEN)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 12a — KNOWN-QA RECALL LAB   (CPU, ~1 min)
# ══════════════════════════════════════════════════════════════════════
# An exact known-QA hit scores METEOR ≈ 1.0 against ≈ 0.55 for a generated
# answer, so each extra CORRECT override is worth ≈ +0.00045 on the test set and
# each WRONG one costs ≈ −0.0005. This cell measures which side of that line the
# CELL 12 matcher falls on, instead of assuming it.
#
# v10's audit ran on val, where 118 of 120 questions match EXACTLY — so fuzzy
# was only ever asked about questions already answered, and "fuzzy 0" measured
# nothing. Here each val question's own entry is HIDDEN and the SAME
# fuzzy_lookup CELL 12 applies (same normalisation, same guards) must find an
# answer anyway: the job it does on test. The guards are switched off once too,
# so you can see what they prevent.
_MISSING = [n for n in ['known_by_q', 'test_ctx', 'val_ctx', 'qa_by_id', 'meteor', 'norm_q',
                        'norm_q_deep', 'build_deep_known', 'build_fuzzy_index', 'fuzzy_lookup',
                        'FUZZY_THRESHOLD', 'fuzzy_calibration_probes'] if n not in globals()]
if _MISSING:
    raise RuntimeError(f"prerequisites missing: {_MISSING}. Run CELL 12 first "
                       f"(it defines the lookup); this cell then measures it.")

import numpy as np
from collections import Counter

_kd, _namb  = build_deep_known(known_by_q)
_inv, _keys = build_fuzzy_index(_kd)
_kn    = {norm_q(k) for k in known_by_q}
_FLOOR = 0.60          # lowest threshold swept — prefix filtering is exact at or above it
_TS    = (0.95, 0.92, 0.90, 0.85, 0.80, 0.75, 0.70, 0.60)
_GEN   = 0.549         # what a generated answer scores; an override must beat it
print(f"known_qa: {len(known_by_q):,} entries → {len(_kd):,} normalised questions "
      f"({_namb} ambiguous, dropped)\n")


# ══ A. HOW MUCH IS THERE TO FIND ON TEST? ══════════════════════════════
_tq    = {str(q): test_ctx.at[q, "question"] for q in test_ctx.index}
_exact = {q for q, s in _tq.items() if norm_q(s) in _kn or norm_q_deep(s) in _kd}
_rest  = [q for q in _tq if q not in _exact]
_on    = {q: fuzzy_lookup(_tq[q], _kd, _inv, _keys, thr=_FLOOR, return_reason=True) for q in _rest}
_sims  = np.array([_on[q][1] for q in _rest]) if _rest else np.zeros(0)
print(f"A. TEST COVERAGE   exact {len(_exact)}/{len(_tq)} | unmatched {len(_rest)}")
print(f"   best similarity among the unmatched:")
for lo, hi in ((0.95, 1.01), (0.90, 0.95), (0.80, 0.90), (0.70, 0.80), (0.60, 0.70)):
    n = int(((_sims >= lo) & (_sims < hi)).sum())
    print(f"     {lo:.2f}–{min(hi, 1.0):.2f}  {n:>4}  {'█' * min(60, n)}")
print(f"     < 0.60     {int((_sims < _FLOOR).sum()):>4}")
_why = Counter(_on[q][2].split(":")[0] for q in _rest if _on[q][1] >= FUZZY_THRESHOLD)
if _why:
    print(f"   at J ≥ {FUZZY_THRESHOLD}: " + " | ".join(f"{k}: {v}" for k, v in _why.most_common()))


# ══ B. IS A FUZZY HIT ACTUALLY RIGHT?  (leave-one-out on val) ══════════
print(f"\nB. LEAVE-ONE-OUT PRECISION ON VAL")
_probe = []
for q in [q for q in val_ctx.index if q in qa_by_id.index]:
    s, ref = val_ctx.at[q, "question"], str(qa_by_id.at[q, "answer"])
    own = norm_q_deep(s)
    if own not in _kd:                         # no exact twin → nothing to hide
        continue
    k0, s0 = fuzzy_lookup(s, _kd, _inv, _keys, thr=_FLOOR, exclude=own, guards=False)
    if k0 is None:
        continue
    k1, _ = fuzzy_lookup(s, _kd, _inv, _keys, thr=_FLOOR, exclude=own)
    _probe.append((s0, k1 is not None, meteor(_kd[k0], ref)))
print(f"   {len(_probe)} val questions found a candidate ≥ {_FLOOR} with their own entry hidden")
_PREC = {}
if _probe:
    _f = lambda v: f"{np.mean(v):.4f}" if v else "—"
    print(f"   {'threshold':>9} │ {'guards ON  fires':>16} {'METEOR':>7} │ "
          f"{'guards OFF  fires':>17} {'METEOR':>7}")
    print("   " + "─" * 64)
    for t in _TS:
        on  = [m for s0, ok, m in _probe if s0 >= t and ok]
        off = [m for s0, ok, m in _probe if s0 >= t]
        _PREC[t] = (float(np.mean(on)), len(on)) if on else None
        print(f"   {t:>9.2f} │ {len(on):>16} {_f(on):>7} │ {len(off):>17} {_f(off):>7}")
    print(f"   an override pays only if its METEOR beats a generated answer (≈ {_GEN})")


# ══ B2. LEAVE-ONE-OUT ON known_qa ITSELF (what CELL 12's calibration uses) ══
# ~118 val questions give a handful of probes; known_qa gives thousands. Hide
# each entry, let fuzzy find its nearest OTHER entry, and score that entry's
# answer against the hidden one's — the test-time event, with far more evidence.
print(f"\nB2. LEAVE-ONE-OUT ON known_qa  (up to {FUZZY_MAX_PROBES:,} entries)")
_kk_on  = fuzzy_calibration_probes(_kd, _inv, _keys, _FLOOR)
_kk_off = fuzzy_calibration_probes(_kd, _inv, _keys, _FLOOR, guards=False)
print(f"   {'threshold':>9} │ {'guards ON  fires':>16} {'METEOR':>7} │ "
      f"{'guards OFF  fires':>17} {'METEOR':>7}")
print("   " + "─" * 64)
for t in _TS:
    on  = [m for s0, m in _kk_on if s0 >= t]
    off = [m for s0, m in _kk_off if s0 >= t]
    _fk = lambda v: f"{np.mean(v):.4f}" if v else "—"
    print(f"   {t:>9.2f} │ {len(on):>16} {_fk(on):>7} │ {len(off):>17} {_fk(off):>7}")
# pooled precision for section C: val probes (guards on) + known_qa probes
_PREC = {}
for t in _TS:
    _v = [m for s0, ok, m in _probe if s0 >= t and ok] + [m for s0, m in _kk_on if s0 >= t]
    _PREC[t] = (float(np.mean(_v)), len(_v)) if _v else None


# ══ C. EXPECTED VALUE ON TEST ══════════════════════════════════════════
print(f"\nC. EXPECTED VALUE = new test overrides × (their METEOR − {_GEN}) / {len(_tq)}")
_rows = []
for t in _TS:
    if not _PREC.get(t):
        continue
    n = sum(1 for q in _rest if _on[q][1] >= t and _on[q][2] == "ok")
    m, k = _PREC[t]
    _rows.append((t, n, m, n * (m - _GEN) / max(1, len(_tq)), k))
    print(f"   {t:>5.2f}   new overrides {n:>4}   E[METEOR] {m:.4f} (from {k} val hits)   "
          f"Δ {_rows[-1][3]:+.4f}")
FUZZY_RECOMMEND = None
if _rows:
    _b = max(_rows, key=lambda r: r[3])
    if _b[3] > 0.002 and _b[4] >= 10:
        FUZZY_RECOMMEND = _b[0]
        if abs(_b[0] - FUZZY_THRESHOLD) < 1e-9:
            print(f"   → the current FUZZY_THRESHOLD = {FUZZY_THRESHOLD} is the best measured: "
                  f"{_b[3]:+.4f} from {_b[1]} overrides ({_b[4]} held-out hits). Nothing to change.")
        else:
            print(f"   → FUZZY_THRESHOLD = {_b[0]:.2f} in CELL 12 is worth {_b[3]:+.4f}, measured on "
                  f"{_b[4]} held-out hits. Set it there and re-run CELL 12.")
    else:
        print(f"   → keep FUZZY_THRESHOLD = {FUZZY_THRESHOLD}. Nothing clears +0.002 on ≥ 10 "
              f"measured hits, and a wrong override costs as much as a right one gains.")
KNOWN_LAB = {"exact_test": len(_exact), "precision": {t: v for t, v in _PREC.items()},
             "recommend": FUZZY_RECOMMEND}
print(f"\n  (CELL 12 already applied its own, stricter rule — each band's LOWER 95% bound must "
      f"beat {_GEN} — and ran at FUZZY_THRESHOLD = {FUZZY_THRESHOLD}.)")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CELL 13 — Locate and download the submission
# ══════════════════════════════════════════════════════════════════════
# SUB_ZIP is set by cell 12 AFTER its integrity guards. A NameError here means
# cell 12 stopped before that line — so this cell diagnoses instead of failing.
import glob
try:
    from google.colab import files
except ImportError:                    # Modal / plain Linux: no browser download
    files = None

_zip = globals().get("SUB_ZIP")
if _zip and os.path.exists(_zip):
    print(f"from cell 12: {_zip}")
else:
    if _zip is None:
        print("SUB_ZIP is not defined — cell 12 did not reach the packaging step.")
        print("Scroll up to cell 12's output: it stops at whichever assert fired.\n")
        print("  'ARTICLE EXPANSION DID NOT APPLY'  → citations are too short; check")
        print("     cell 3's 'fully named → expandable' percentage. Low coverage means")
        print("     article_text() is falling back to single chunks.")
        print("  'mean answer N tok — expected ~800' → same root cause.")
        print("  'empty answer present'              → cell 11 has gaps; re-run it.")
        print("  'QID mismatch'                      → cell 12 prints the cause above")
        print("     it. Re-run cells 7 and 11: both resume and fill only what is missing.\n")
    else:
        print(f"SUB_ZIP is set to {_zip} but the file is missing.\n")
    # Fall back to whatever IS on Drive, newest first — better than nothing.
    _cands = sorted(glob.glob(W("*.zip")), key=os.path.getmtime, reverse=True)
    if not _cands:
        raise FileNotFoundError(
            f"no .zip in {cfg.work_dir}. Fix cell 12's assertion and re-run it — "
            f"the raw answers are cached, so repackaging takes seconds, not a "
            f"regeneration.")
    print(f"found {len(_cands)} archive(s) in the work dir, newest first:")
    for _c in _cands[:5]:
        import time as _t
        print(f"  {os.path.basename(_c):<34} {os.path.getsize(_c)/1e6:6.2f} MB  "
              f"{_t.strftime('%Y-%m-%d %H:%M', _t.localtime(os.path.getmtime(_c)))}")
    _zip = _cands[0]
    print(f"\n⚠ downloading the NEWEST archive: {os.path.basename(_zip)}")
    print("  This may predate your current settings. Verify before submitting.")

# Independent verification — never submit an archive this cell has not opened.
import zipfile, hashlib
with zipfile.ZipFile(_zip) as _z:
    assert _z.namelist() == ["submission.json"], \
        f"archive layout wrong: {_z.namelist()} (Codabench needs it at the root)"
    _sub = json.loads(_z.read("submission.json").decode("utf-8"))
# v13: the archive must answer EXACTLY the questions of the test file on disk,
# re-read here — independent of test_ctx, of CELL 12 and of every cache.
if "TEST_FILES" in globals() and "load_test_set" in globals():
    _tf = load_test_set(TEST_FILES)
    _miss, _extra = set(_tf) - set(_sub), set(_sub) - set(_tf)
    assert not _miss and not _extra, (
        f"archive ≠ test file: {len(_miss)} questions unanswered, {len(_extra)} answers for ids not in "
        f"{' + '.join(os.path.basename(f) for f in TEST_FILES)} (e.g. {sorted(_miss or _extra)[:3]})")
    _n_exp = globals().get("EXPECTED_N_TEST") or len(_tf)
    assert len(_sub) == _n_exp, f"{len(_sub):,} answers, expected {_n_exp:,}"
    print(f"✓ {len(_sub):,} answers = the {len(_tf):,} questions of "
          f"{' + '.join(os.path.basename(f) for f in TEST_FILES)}")
_ln = np.array([len(str(v.get('answer','')).split()) for v in _sub.values()])
print(f"\n{len(_sub)} answers | words: mean {_ln.mean():.0f} median {np.median(_ln):.0f} "
      f"p90 {np.percentile(_ln,90):.0f} | empty: {int((_ln==0).sum())}")
print(f"sha256 {hashlib.sha256(open(_zip,'rb').read()).hexdigest()[:24]}…")
print(f"  reference: 0.4886 had mean 584 · 0.5224 (whole-article) had mean ~810")
assert (_ln == 0).sum() == 0, "empty answers present — each scores 0"

if files is not None:
    files.download(_zip)
else:
    print(f"\nnot on Colab — the submission is on the volume: {_zip}")
